#### ANÁLISE BRASIL

## Preparação das bases e parâmetros gerais

Carregamento, tratamento e definição dos recortes utilizados nas análises nacionais.


In [ ]:
from __future__ import annotations
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.lines as lines
import matplotlib.patheffects as pe
import seaborn as sns
import os
from dbfread import DBF
import geopandas as gpd
import ssl
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from pathlib import Path
import sys
import geobr
import warnings
import urllib3
import math
import openpyxl
from reportlab.lib.utils import simpleSplit
from reportlab.lib import colors
from PIL import Image
import argparse
import re
import textwrap
from matplotlib.ticker import MaxNLocator
from openpyxl.styles import Font, Alignment
from openpyxl.styles import PatternFill

# Filtro de data
data_analise = '25/11/2026'
data_gal = '25/11/2026'
se_analise = 46

In [ ]:
# ACESSO ÀS BASES DE DADOS
# Pasta atual onde o notebook está rodando
pasta_atual = Path('.').resolve()

# Sobe dois níveis para chegar na raiz (equivalente ao seu .parent.parent)
root = pasta_atual.parent

# Acessa a pasta de dados e depois a BASE_DBF
base_dados = root / 'BASES_DADOS'
base_ibge = root/ 'BASE_IBGE'

# Verificação de erro
if not base_dados.exists():
    raise FileNotFoundError('A pasta não foi encontrada. Verificar!')

print('Processamento Iniciado!')
print('-' * 50)

# VERIFICAÇÃO DE ARQUIVOS PROCESSADOS
arquivos_base = [
    arquivo for arquivo in base_dados.iterdir()
    if (arquivo.is_file()
        and arquivo.suffix.lower() == ".csv"
        and arquivo.name.upper().startswith("CHIKON_"))
]

if arquivos_base:
    print("As bases DBF já foram processadas!")
    print("-" * 50)
    print(f"Foram encontrados {len(arquivos_base)} arquivos CHIKON")
    print("Nenhuma conversão ou concatenação será executada")

else:
    print('Iniciando a conversão e concatenação dos arquivos...')
    print('-' * 50)

    # CONVERSÃO DE DBF PARA CSV E EXCLUSÃO DO DBF
    arquivos_dbf = [
        arquivo for arquivo in base_dados.iterdir()
        if (arquivo.is_file()
        and arquivo.suffix.lower() == ".dbf"
        and arquivo.name.upper().startswith("CHIKON_"))
    ]

    if arquivos_dbf:
        print(f"Foram encontrados {len(arquivos_dbf)} arquivos DBF para converter")

        for caminho_dbf in arquivos_dbf:

            # Mantém o mesmo nome e troca apenas a extensão
            caminho_xlsx = caminho_dbf.with_suffix(".csv")

            print(f"Convertendo: {caminho_dbf.name}")

            try:
                tabela_dbf = DBF(
                    str(caminho_dbf),
                    encoding="iso-8859-1"
                )

                df_temp = pd.DataFrame(iter(tabela_dbf))

                df_temp.to_csv(
                    caminho_xlsx,
                    index=False
                )

                # Excluir o DBF somente depois da criação do arquivo CSV
                caminho_dbf.unlink()

                print(f"{caminho_xlsx.name} salvo com sucesso. Arquivo DBF excluído.")

            except Exception as erro:
                print(f"Erro na conversão de {caminho_dbf.name}." f"Motivo: {erro}")

    else:
        print("Nenhum arquivo DBF foi encontrado para conversão")
        print("-" * 50)

    # CONCATENAÇÃO DOS ARQUIVOS CSV E LIMPEZA DOS DADOS
    print('-' * 50)
    print('Início da concatenação dos arquivos')

    arquivos_excel = [
        arquivo for arquivo in base_dados.iterdir()
        if (arquivo.is_file()
            and arquivo.suffix.lower() == ".csv"
            and arquivo.name.upper().startswith("CHIKON_"))
        ]

    if arquivos_excel:
        print(f"Concatenando {len(arquivos_excel)} arquivos.")
        print('-' * 50)

        dfs = []

        for caminho_excel in arquivos_excel:
            print(f"Base processada: {caminho_excel.name}")

            df_estado = pd.read_csv(caminho_excel, low_memory=False)
            dfs.append(df_estado)

        df_notificados_bruto = pd.concat(
            dfs,
            ignore_index=True
        )

        print("Arquivos concatenados com sucesso!")

        # Remover colunas desnecessárias
        colunas_para_remover = [
            "SOUNDEX", "ID_DISTRIT", "ID_GEO1", "ID_GEO2",
            "DT_PRNT", "RESUL_PRNT",
            "UF", "MUNICIPIO", "HOSPITAL", "DDD_HOSP",
            "TEL_HOSP", "TPAUTOCTO", "COUFINF", "COPAISINF",
            "COMUNINF", "CODISINF", "CO_BAINF", "NOBAIINF",
            "DOENCA_TRA", "ALRM_HIPOT", "ALRM_PLAQ", "ALRM_VOM", 
            "ALRM_SANG", "ALRM_HEMAT", "ALRM_ABDOM", "ALRM_LETAR",
            "ALRM_HEPAT", "ALRM_LIQ", "DT_ALRM",
            "GRAV_PULSO", "GRAV_CONV", "GRAV_ENCH",
            "GRAV_INSUF", "GRAV_TAQUI", "GRAV_EXTRE",
            "GRAV_HIPOT", "GRAV_HEMAT", "GRAV_MELEN",
            "GRAV_METRO", "GRAV_SANG", "GRAV_AST",
            "GRAV_MIOC", "GRAV_CONSC", "GRAV_ORGAO",
            "DT_GRAV", "MANI_HEMOR", "EPISTAXE",
            "GENGIVO", "METRO", "PETEQUIAS", "HEMATURA",
            "SANGRAM", "LACO_N", "PLASMATICO", "EVIDENCIA",
            "PLAQ_MENOR", "CON_FHD", "COMPLICA", "NU_LOTE_I",
            "NDUPLIC_N", "DT_TRANSUS", "DT_TRANSDM",
            "DT_TRANSSM", "DT_TRANSRM", "DT_TRANSRS",
            "DT_TRANSSE", "NU_LOTE_V", "NU_LOTE_H",
            "CS_FLXRET", "FLXRECEBI", "IDENT_MICR",
            "MIGRADO_W", "SG_UF_NOT", "ID_MUNICIP",
            "ID_UNIDADE", "ID_PAIS", "ID_OCUPA_N", 
            "DT_SORO", "DT_NS1", "DT_VIRAL",
            "SOROTIPO", "RESUL_VI_N", "HISTOPA_N",
            "IMUNOH_N", "TP_SISTEMA", "TP_NOT",
            "CS_ESCOL_N", "CS_GESTANT", "DS_OBS",
            "DT_ALARM", "NU_LOTE_N", "NM_REFEREN", 
            "NU_DDD_TEL", "NU_TELEFON", "IN_VINCULA", 
            "FONETICA_N"
        ]

        df_notificados_drop = df_notificados_bruto.drop(columns=colunas_para_remover, errors='ignore')
        print('Limpeza do Dataframe concluída')

        # Criar a coluna SE
        if "SEM_PRI" in df_notificados_drop.columns:
            posicao_sem_pri = df_notificados_drop.columns.get_loc("SEM_PRI")
            valores_se = (df_notificados_drop["SEM_PRI"].astype(str).str[-2:])
            df_notificados_drop.insert(loc=posicao_sem_pri + 1, column="SE", value=valores_se)

        # Salvar o resultado na mesma pasta do script
        notificados_2026 = df_notificados_drop[df_notificados_drop['SEM_PRI'] >= 202601]    # Filtro de segurança para garantir dados reais
        output_notificados = (base_dados / "chik_brasil_notificados_2026.csv")
        notificados_2026.to_csv(output_notificados, index=False)

        print('-' * 50)
        print("Arquivo com casos notificados criado com sucesso!")

        # CRIAR ARQUIVO DE CASOS PROVAVEIS
    
        # Filtragem do dataframe para retirar os casos descartados
        df_provaveis = notificados_2026[notificados_2026['CLASSI_FIN'] != 5]

        # Salvar data frame completo
        output_provaveis = (base_dados / "chik_brasil_provaveis_2026.csv")
        df_provaveis.to_csv(output_provaveis, index= False)

        print('Arquivo com casos prováveis criado com sucesso!')
        print('Bases do SINAN processadas com sucesso!')

# PROCESSAMENTO DA BASE GAL DENGUE E CHIKUNGUNYA
print('-' * 50)
print('Iniciando processamento das bases de dados GAL (Chikungunya e Dengue)')

# Dicionário com as configurações de cada base de dados
bases_gal = {
    'Chikungunya': {
        'arquivo_entrada': base_dados / 'chikungunya_analise_completa.xlsx',
        'arquivo_saida': base_dados / 'chik_gal_brasil_2026.csv',
        'sheet': 'Chikungunya_por_Municipio'
    },
    'Dengue': {
        'arquivo_entrada': base_dados / 'dengue_analise_completa.xlsx',
        'arquivo_saida': base_dados / 'dengue_gal_brasil_2026.csv',
        'sheet': 'Dengue_por_Municipio'
    }
}

# Loop para processar as duas bases com a mesma estrutura
for doenca, config in bases_gal.items():
    print(f'Processando base GAL {doenca}')

    # 1. Carregar a aba de municípios
    df_gal_bruto = pd.read_excel(
        config['arquivo_entrada'], 
        sheet_name=config['sheet'], 
        engine="openpyxl"
    )

    # 2. Desfazer (Melt) a tabela
    df_melt = df_gal_bruto.melt(
        id_vars=['paciente_municipio_ibge', 'paciente_municipio'], 
        var_name='Coluna_Original', 
        value_name='Valor'
    )

    # 3. Extrair SE e Métrica
    df_melt['SE'] = df_melt['Coluna_Original'].str.extract(r'SE(\d+)').fillna(0).astype(int)
    df_melt['Metrica'] = df_melt['Coluna_Original'].apply(lambda x: x.split('_')[-1])

    # 4. Reorganizar métricas em colunas (Pivot)
    df_gal_brasil = df_melt.pivot_table(
        index=['paciente_municipio_ibge', 'paciente_municipio', 'SE'],
        columns='Metrica',
        values='Valor',
        aggfunc='sum'
    ).reset_index()

    # 5. Limpeza e criação do código da UF
    df_gal_brasil = df_gal_brasil.fillna(0)
    df_gal_brasil.columns.name = None 

    df_gal_brasil['UF_COD'] = (df_gal_brasil['paciente_municipio_ibge'] // 10000).astype(int)
    df_gal_brasil['UF_COD'] = pd.to_numeric(df_gal_brasil['UF_COD'], errors='coerce')

    # 6. Renomear colunas
    df_gal_final = df_gal_brasil.rename(columns={
        'paciente_municipio_ibge': 'IBGE',
        'UF_COD': 'UF',
        'paciente_municipio': 'municipio',
        'exames': 'exames_realizados',
        'positivos': 'casos_positivos'
    })

    #df_gal_final = df_gal_final[df_gal_final['SE'] < df_gal_final['SE'].max()]

    # 7. Salvar o arquivo processado
    df_gal_final.to_csv(config['arquivo_saida'], index=False)
    print(f'Base GAL {doenca} processada com sucesso!')

print('Processamento de todas as bases GAL concluído com sucesso!')
print('-' * 50)

# ======================================================== #

# Definição das bases de dados
df_notificados = pd.read_csv(base_dados / 'chik_brasil_notificados_2026.csv')
df_provaveis = pd.read_csv(base_dados / 'chik_brasil_provaveis_2026.csv', low_memory=False)
df_gal_final = pd.read_csv(base_dados / 'chik_gal_brasil_2026.csv')
df_gal_dengue = pd.read_csv(base_dados / 'dengue_gal_brasil_2026.csv')

print('Bases definidas. Aguardar filtragem de oito semanas e para temporada...')

# ======================================================== #

# criar lista com as bases de dados
bases = [df_notificados, df_provaveis, df_gal_final]

# Função para garantir que a coluna SE será numerica em todas as bases
for df_bases in bases: 
    df_bases['SE'] = pd.to_numeric(df_bases['SE'], errors='coerce')

# Encontrar semana máxima das bases
se_inicio = 1
se_final = max(df_bases['SE'].max() for df_bases in bases)
se_atual = se_final
semanas_analise = se_atual - 7

df_notificados_filtrado, df_provaveis_filtrado, df_gal_final_filtrado = [
    df_bases[df_bases['SE'].between(semanas_analise, se_atual)] for df_bases in bases
]
print('Bases de oito semanas criadas com sucesso!')

print('Definindo recorte de temporada a partir da SE27/2026')
temporada_2026 = df_provaveis[df_provaveis['SE'] >= 27]
gal_temporada = df_gal_final[df_gal_final['SE'] >= 27]
print('Recorte definido para temporada 2026/2027')

se_temporada_inicial = 27
se_temporada_atual = df_provaveis['SE'].max()

# Mapeamento das unidades federativas por código IBGE
mapa_uf = {
    11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO',
    21: 'MA', 22: 'PI', 23: 'CE', 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL', 28: 'SE', 29: 'BA',
    31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP', 41: 'PR', 42: 'SC', 43: 'RS', 50: 'MS', 51: 'MT', 
    52: 'GO', 53: 'DF'
}

# Carregar base dos municípios brasileiros
df_municipios = pd.read_csv(base_ibge / 'IBGE_Municipios_Populacao_2025.csv', sep='\t', encoding='latin-1')
regioes_saude = pd.read_csv(base_ibge / 'IBGE_Municipios_Regioes_Saude_Populacao_2026.csv')
df_estados = pd.read_csv(base_ibge / 'IBGE_UF_Populacao_2025.csv', sep=';')

print('Base de municípios carregada com sucesso!')
print('-' * 50)

df_base_2025 = pd.read_csv(base_dados / 'CHIK_2025.csv', sep=';', encoding='latin-1', low_memory=False)
df_provaveis_2025 = df_base_2025[(df_base_2025['NU_ANO'] == 2025) & (df_base_2025['CLASSI_FIN'] != 5)]

print(f'Base histórica filtrada para casos do ano {df_provaveis_2025['NU_ANO'].unique()}')
print('-' * 50)

print('Todas as bases de dados foram criadas com sucesso!')
print('-' * 50)

# Criar pasta para salvar os arquivos
nome_pasta = f'analises_chikungunya_brasil_SE{se_analise}'
local_salvar = os.getcwd()
local_pasta = os.path.join(local_salvar, nome_pasta)

if not os.path.exists(local_pasta):
    os.makedirs(local_pasta)
    print(f"Pasta {nome_pasta} criada!")
else:
    print(f"A pasta {nome_pasta} já existe. Os arquivos serão salvos nela.")
    print('-' * 50)

print('Fim do processamento!')

In [ ]:
# ACESSO ÀS BASES DE DADOS
# Pasta atual onde o notebook está rodando
pasta_atual = Path('.').resolve()

# Sobe dois níveis para chegar na raiz (equivalente ao seu .parent.parent)
root = pasta_atual.parent

# Acessa a pasta de dados e depois a BASE_DBF
base_dados = root / 'BASES_DADOS'
base_ibge = root/ 'BASE_IBGE'

# Verificação de erro
if not base_dados.exists():
    raise FileNotFoundError('A pasta não foi encontrada. Verificar!')

print('Processamento Iniciado!')
print('-' * 50)

# VERIFICAÇÃO DE ARQUIVOS PROCESSADOS
arquivos_base = [
    arquivo for arquivo in base_dados.iterdir()
    if (arquivo.is_file()
        and arquivo.suffix.lower() == ".csv"
        and arquivo.name.upper().startswith("DENGON_"))
]

if arquivos_base:
    print("As bases já foram processadas!")
    print("-" * 50)
    print(f"Foram encontrados {len(arquivos_base)} arquivos DENGON")
    print("Nenhuma conversão ou concatenação será executada")
    print('-' * 50)

else:
    print('Iniciando a conversão e concatenação dos arquivos...')
    print('-' * 50)

    # CONVERSÃO DE DBF PARA CSV E EXCLUSÃO DO DBF
    arquivos_dbf = [
        arquivo for arquivo in base_dados.iterdir()
        if (arquivo.is_file()
        and arquivo.suffix.lower() == ".dbf"
        and arquivo.name.upper().startswith("DENGON_"))
    ]

    if arquivos_dbf:
        print(f"Foram encontrados {len(arquivos_dbf)} arquivos DBF para converter")

        for caminho_dbf in arquivos_dbf:

            # Mantém o mesmo nome e troca apenas a extensão
            caminho_xlsx = caminho_dbf.with_suffix(".csv")

            print(f"Convertendo: {caminho_dbf.name}")

            try:
                tabela_dbf = DBF(
                    str(caminho_dbf),
                    encoding="iso-8859-1"
                )

                df_temp = pd.DataFrame(iter(tabela_dbf))

                df_temp.to_csv(
                    caminho_xlsx,
                    index=False
                )

                # Excluir o DBF somente depois da criação do arquivo CSV
                caminho_dbf.unlink()

                print(f"{caminho_xlsx.name} salvo com sucesso. Arquivo DBF excluído.")

            except Exception as erro:
                print(f"Erro na conversão de {caminho_dbf.name}." f"Motivo: {erro}")

    else:
        print("Nenhum arquivo DBF foi encontrado para conversão")
        print("-" * 50)

    # CONCATENAÇÃO DOS ARQUIVOS CSV E LIMPEZA DOS DADOS
    print('-' * 50)
    print('Início da concatenação dos arquivos')

    arquivos_excel = [
        arquivo for arquivo in base_dados.iterdir()
        if (arquivo.is_file()
            and arquivo.suffix.lower() == ".csv"
            and arquivo.name.upper().startswith("DENGON_"))
        ]

    if arquivos_excel:
        print(f"Concatenando {len(arquivos_excel)} arquivos.")
        print('-' * 50)

        dfs = []

        for caminho_excel in arquivos_excel:
            print(f"Base processada: {caminho_excel.name}")

            df_estado = pd.read_csv(caminho_excel, low_memory=False)
            dfs.append(df_estado)

        df_notificados_bruto = pd.concat(
            dfs,
            ignore_index=True
        )

        print("Arquivos concatenados com sucesso!")

        # Remover colunas desnecessárias
        colunas_para_remover = [
            "SOUNDEX", "ID_DISTRIT", "ID_GEO1", "ID_GEO2",
            "DT_CHIK_S1", "DT_CHIK_S2", "DT_PRNT",
            "RES_CHIKS1", "RES_CHIKS2", "RESUL_PRNT",
            "UF", "MUNICIPIO", "HOSPITAL", "DDD_HOSP",
            "TEL_HOSP", "TPAUTOCTO", "COUFINF", "COPAISINF",
            "COMUNINF", "CODISINF", "CO_BAINF", "NOBAIINF",
            "DOENCA_TRA", "ALRM_HIPOT", "ALRM_PLAQ", "ALRM_VOM", 
            "ALRM_SANG", "ALRM_HEMAT", "ALRM_ABDOM", "ALRM_LETAR",
            "ALRM_HEPAT", "ALRM_LIQ", "DT_ALRM",
            "GRAV_PULSO", "GRAV_CONV", "GRAV_ENCH",
            "GRAV_INSUF", "GRAV_TAQUI", "GRAV_EXTRE",
            "GRAV_HIPOT", "GRAV_HEMAT", "GRAV_MELEN",
            "GRAV_METRO", "GRAV_SANG", "GRAV_AST",
            "GRAV_MIOC", "GRAV_CONSC", "GRAV_ORGAO",
            "DT_GRAV", "MANI_HEMOR", "EPISTAXE",
            "GENGIVO", "METRO", "PETEQUIAS", "HEMATURA",
            "SANGRAM", "LACO_N", "PLASMATICO", "EVIDENCIA",
            "PLAQ_MENOR", "CON_FHD", "COMPLICA", "NU_LOTE_I",
            "NDUPLIC_N", "DT_TRANSUS", "DT_TRANSDM",
            "DT_TRANSSM", "DT_TRANSRM", "DT_TRANSRS",
            "DT_TRANSSE", "NU_LOTE_V", "NU_LOTE_H",
            "CS_FLXRET", "FLXRECEBI", "IDENT_MICR",
            "MIGRADO_W", "SG_UF_NOT", "ID_MUNICIP",
            "ID_UNIDADE", "ID_PAIS", "ID_OCUPA_N", 
            "DT_SORO", "DT_NS1", "DT_VIRAL",
            "SOROTIPO", "RESUL_VI_N", "HISTOPA_N",
            "IMUNOH_N", "TP_SISTEMA", "TP_NOT",
            "CS_ESCOL_N", "CS_GESTANT", "DS_OBS",
            "DT_ALARM", "NU_LOTE_N", "NM_REFEREN", 
            "NU_DDD_TEL", "NU_TELEFON", "IN_VINCULA", 
            "FONETICA_N"
        ]

        df_notificados_drop = df_notificados_bruto.drop(columns=colunas_para_remover, errors='ignore')
        print('Limpeza do Dataframe concluída')

        # Criar a coluna SE
        if "SEM_PRI" in df_notificados_drop.columns:
            posicao_sem_pri = df_notificados_drop.columns.get_loc("SEM_PRI")
            valores_se = (df_notificados_drop["SEM_PRI"].astype(str).str[-2:])
            df_notificados_drop.insert(loc=posicao_sem_pri + 1, column="SE", value=valores_se)

        # Salvar o resultado na mesma pasta do script
        den_notificados_2026 = df_notificados_drop[df_notificados_drop['SEM_PRI'] >= 202601]    # Filtro de segurança para garantir dados reais
        den_output_notificados = (base_dados / "dengue_brasil_notificados_2026.csv")
        den_notificados_2026.to_csv(den_output_notificados, index=False)
    
        print('-' * 50)
        print("Arquivo com casos notificados criado com sucesso!")
    
        # CRIAR ARQUIVO DE CASOS PROVAVEIS
        
        # Filtragem do dataframe para retirar os casos descartados
        den_provaveis = den_notificados_2026[den_notificados_2026['CLASSI_FIN'] != 5]
    
        # Salvar data frame completo
        den_output_provaveis = (base_dados / "dengue_brasil_provaveis_2026.csv")
        den_provaveis.to_csv(den_output_provaveis, index= False)
    
        print('Arquivo com casos prováveis criado com sucesso!')
        print('Bases do SINAN processadas com sucesso!')

# VERIFICAÇÃO DE ARQUIVOS PROCESSADOS
arquivos_den = [
    arquivo for arquivo in base_dados.iterdir()
    if (arquivo.is_file()
        and arquivo.suffix.lower() == ".csv"
        and arquivo.name.upper().startswith("dengue_brasil_"))
]

if arquivos_den:
    print('As bases de dados não foram encontradas na pasta!')
    print('-' * 50)

else:
    print('Iniciando a leitura das bases do SINAN...')
    den_notificados = pd.read_csv(base_dados / 'dengue_brasil_notificados_2026.csv')
    den_provaveis = pd.read_csv(base_dados / 'dengue_brasil_provaveis_2026.csv')
    print("Leitura das bases executada com sucesso!")
    print('-' * 50)

# criar lista com as bases de dados
bases = [den_notificados, den_provaveis, df_gal_dengue]

# Função para garantir que a coluna SE será numerica em todas as bases
for df_bases in bases: 
    df_bases['SE'] = pd.to_numeric(df_bases['SE'], errors='coerce')

# Encontrar semana máxima das bases
se_final = max(df_bases['SE'].max() for df_bases in bases)
se_atual = se_final - 1
semanas_analise = se_atual - 7

den_notificados_filtrado, den_provaveis_filtrado, df_gal_dengue_filtrado = [
    df_bases[df_bases['SE'].between(semanas_analise, se_atual)] for df_bases in bases
]

print('Bases de oito semanas criadas com sucesso!')

print('Definindo recorte de temporada a partir da SE27/2026...')

den_temporada_2026 = den_provaveis[den_provaveis['SE'] >= 27]
print('Recorte definido para temporada 2026/2027')

print('-' * 50)
print('Fim do processamento!')

## 1. Evolução temporal e situação epidemiológica


In [ ]:
# DIAGRAMA DE CONTROLE ROBUSTO - BRASIL
# Linha de base histórica: 2016-2025
# Método: Mediana + MAD
# Indicador: coeficiente de incidência por 100 mil habitantes

# 1. CARREGAR E PREPARAR LINHA DE BASE (2016-2025)

arquivo_historico_dc = base_dados / "sinan_chik_semanas_2015_2025.csv"

if not arquivo_historico_dc.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {arquivo_historico_dc}. Coloque na pasta BASES_DADOS."
    )

# POPULAÇÃO DO BRASIL
# Fonte: IBGE - Projeção da População, Revisão 2024

populacao_brasil_dc = {
    2016: 203_871_925,
    2017: 205_211_557,
    2018: 206_529_038,
    2019: 207_900_099,
    2020: 209_164_889,
    2021: 210_103_642,
    2022: 210_862_983,
    2023: 211_695_158,
    2024: 212_583_750,
    2025: 213_421_037,
    2026: 214_211_951
}

df_historico_dc = pd.read_csv(arquivo_historico_dc)

cols_num = ["ano", "semana_epidemiologica", "casos"]
df_historico_dc[cols_num] = df_historico_dc[cols_num].apply(pd.to_numeric, errors="coerce")

# Filtra anos e semanas válidas
df_historico_dc = df_historico_dc[df_historico_dc["ano"].between(2016, 2025)
    & df_historico_dc["semana_epidemiologica"].between(1, 52)].copy()

# Adiciona o denominador populacional correspondente a cada ano
df_historico_dc["populacao"] = df_historico_dc["ano"].map(populacao_brasil_dc)

# Calcula o coeficiente de incidência semanal por 100 mil habitantes
df_historico_dc["incidencia"] = (df_historico_dc["casos"] / df_historico_dc["populacao"]) * 100_000

# Cálculo simultâneo da mediana e do MAD por semana
linha_base_dc = (df_historico_dc.groupby("semana_epidemiologica")["incidencia"].agg(mediana="median", MAD=lambda x: np.median(np.abs(x - np.median(x)))).reset_index())

# Limites robustos
linha_base_dc["sigma_robusto"] = 1.4826 * linha_base_dc["MAD"]
linha_base_dc["limite_alerta"] = linha_base_dc["mediana"] + 2 * linha_base_dc["sigma_robusto"]
linha_base_dc["limite_superior"] = linha_base_dc["mediana"] + 3 * linha_base_dc["sigma_robusto"]

# 2. SÉRIE OBSERVADA DE 2026 E CLASSIFICAÇÃO

df_provaveis["SE"] = pd.to_numeric(df_provaveis["SE"], errors="coerce")

serie_2026_dc = (
    df_provaveis.dropna(subset=["SE"])
    .groupby("SE")
    .size()
    .reset_index(name="casos_2026")
    .rename(columns={"SE": "semana_epidemiologica"})
)

serie_2026_dc["semana_epidemiologica"] = serie_2026_dc["semana_epidemiologica"].astype(int)

# Coeficiente de incidência de 2026
serie_2026_dc["incidencia_2026"] = (serie_2026_dc["casos_2026"] / populacao_brasil_dc[2026]) * 100_000
se_atual_dc = serie_2026_dc["semana_epidemiologica"].max()

resultado_dc = linha_base_dc.merge(
    serie_2026_dc,
    on="semana_epidemiologica",
    how="left")

# Classificação usando np.select
resultado_dc["classificacao"] = np.select(
    [
        resultado_dc["incidencia_2026"] > resultado_dc["limite_superior"],
        resultado_dc["incidencia_2026"] > resultado_dc["limite_alerta"]
    ],
    [
        "Acima do limite superior",
        "Alerta"
    ],
    default="Dentro do esperado"
)

resultado_dc_2026 = resultado_dc[resultado_dc["semana_epidemiologica"] <= se_atual_dc].copy()

# 3. INTERPRETAÇÃO OPERACIONAL
# Janela móvel: 3 últimas SEs consolidadas

se_fim_avaliacao_dc = max(1, se_atual_dc - 1)
se_inicio_avaliacao_dc = max(1, se_atual_dc - 3)

janela_avaliacao_dc = resultado_dc_2026[
    resultado_dc_2026["semana_epidemiologica"].between(
        se_inicio_avaliacao_dc,
        se_fim_avaliacao_dc
    )
]

n_sup = (janela_avaliacao_dc["classificacao"] == "Acima do limite superior").sum()
n_alt = (janela_avaliacao_dc["classificacao"] == "Alerta").sum()

if n_sup >= 2:
    situacao_dc = "acima do limite superior de forma persistente"
    texto_diagrama_controle = (
        f"No diagrama de controle, o coeficiente de incidência permaneceu acima do limite superior "
        f"em pelo menos duas das SE {se_inicio_avaliacao_dc} a {se_fim_avaliacao_dc}, indicando "
        f"comportamento persistentemente superior ao esperado historicamente."
    )

elif n_sup == 1:
    situacao_dc = "acima do limite superior"
    texto_diagrama_controle = (
        f"No diagrama de controle, o coeficiente de incidência de uma das SE "
        f"{se_inicio_avaliacao_dc} a {se_fim_avaliacao_dc} ultrapassou o limite superior histórico, "
        f"configurando sinal de atenção para monitoramento nas semanas subsequentes."
    )

elif n_alt >= 2:
    situacao_dc = "em alerta persistente"
    texto_diagrama_controle = (
        f"No diagrama de controle, o coeficiente de incidência permaneceu na faixa de alerta em pelo "
        f"menos duas das SE {se_inicio_avaliacao_dc} a {se_fim_avaliacao_dc}, indicando elevação "
        f"persistente em relação ao padrão histórico."
    )

elif n_alt == 1:
    situacao_dc = "em alerta"
    texto_diagrama_controle = (
        f"No diagrama de controle, o coeficiente de incidência de uma das SE "
        f"{se_inicio_avaliacao_dc} a {se_fim_avaliacao_dc} atingiu a faixa de alerta, sem persistência "
        f"do sinal nas demais semanas avaliadas."
    )

else:
    situacao_dc = "dentro do canal endêmico"
    texto_diagrama_controle = (
        f"No diagrama de controle, o coeficiente de incidência permaneceu dentro do canal endêmico "
        f"nas SE {se_inicio_avaliacao_dc} a {se_fim_avaliacao_dc}."
    )

# 4. GERAR FIGURA

fig, ax = plt.subplots(figsize=(15, 8))

se = resultado_dc["semana_epidemiologica"]

ax.fill_between(
    se, 0, resultado_dc["limite_alerta"],
    color="#e9f5db", alpha=0.55, label="Faixa esperada")

ax.fill_between(
    se, resultado_dc["limite_alerta"], resultado_dc["limite_superior"],
    color="#ffc2d1", alpha=0.55, label="Faixa de alerta")

ax.plot(
    se, resultado_dc["mediana"],
    color="#c00000", lw=2, label="Mediana histórica (2016-2025)")

ax.plot(
    se, resultado_dc["limite_alerta"],
    color="#f4a261", ls="--", lw=1.8, label="Limite de alerta")

ax.plot(
    se, resultado_dc["limite_superior"],
    color="#e76f51", ls=":", lw=2, label="Limite superior")

ax.plot(
    resultado_dc_2026["semana_epidemiologica"],
    resultado_dc_2026["incidencia_2026"],
    color="#1d3557", marker="o", ms=5, lw=2.5,
    label="Coeficiente de incidência 2026")

ax.set_title(
    "Diagrama de controle - Chikungunya - Brasil, 2026".upper(),
    fontsize=18, fontweight="bold", pad=18)

ax.set(
    xlabel="Semana Epidemiológica",
    ylabel="Coeficiente de Incidência",
    xlim=(1, 52),
    ylim=(0, None),
    xticks=range(1, 53))

ax.grid(axis="y", alpha=0.20)
ax.grid(axis="x", alpha=0.2)
ax.legend(frameon=True, ncol=1, fontsize=10)

fig.text(
    0.035, -0.03,
    "Fonte: SINAN | Série histórica utilizada: 2016-2025 | A última SE pode estar sujeita a atraso de notificação | Incidência calculada por 100 mil habitantes\n"
    "Observação: Os limites foram calculados pelo método MAD (Median Absolute Deviation) + mediana, dada a robustez frente valores outliers (anos epidêmicos).\n"
    "Isso permite a possibilidade de inclusão de anos epidêmicos sem distorção dos limites do diagrama de controle.\n"
    f"------> Situação operacional: {situacao_dc} <------",
    ha="left", fontsize=9)

plt.tight_layout(rect=[0, 0.04, 1, 1])

plt.savefig(os.path.join(local_pasta, "chik_brasil_diagrama_controle.png"), dpi=300, bbox_inches="tight")
print(f"Diagrama de controle gerado com sucesso!\nSituação operacional: {situacao_dc}.")

plt.show()

In [ ]:
# Tratamento da Classificação
# Transformar em número. O que for vazio ('') vira NaN (Nulo)
notificados_chik = df_notificados[df_notificados['CLASSI_FIN'] != 8.0]

notificados_chik['CLASSI_FIN'] = pd.to_numeric(notificados_chik['CLASSI_FIN'], errors='coerce')

# Preencher os valores nulos (NaN) com um código numérico fictício (ex: 99) 
# Isso impede que o groupby ignore os casos "Em Investigação"
notificados_chik['CLASSI_FIN'] = notificados_chik['CLASSI_FIN'].fillna(99)

# Novo dicionário com o código 99 representando o que estava vazio
mapa_class = {5: 'Descartado', 13: 'Confirmado', 99: 'Em Investigação'}

# Agrupar os dados
# Dataframe para o gráfico de barras empilhadas e tabela
df_class = notificados_chik.groupby(['SE', 'CLASSI_FIN']).size().unstack(fill_value=0) 
df_class = df_class.rename(columns=mapa_class)

# Configurar o estilo visual
grafico_class = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_class.add_subplot(gs[0])

# Criar o Gráfico de Barras Empilhadas
# Adicionei cores distintas para cada classificação
cores = {'Confirmado': '#9db5b2', 'Descartado': '#cc4e00', 'Em Investigação': '#a3b4a2'}
df_class.plot(kind='bar', stacked=True, ax=ax, width=0.8, color=[cores.get(col, '#333333') for col in df_class.columns])

# Adicionar os rótulos de dados (bar_label)
# Como são barras empilhadas, precisamos passar por CADA container (cada pedaço da barra)
for c in ax.containers:
    # Condição para só escrever o número se ele for maior que 0 (evita poluir o gráfico com zeros)
    labels = [f'{v.get_height():.0f}' if v.get_height() > 0 else '' for v in c]
    # label_type='center' coloca o número bem no meio da cor respectiva
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='white', fontweight='bold')

ax.set_title('Classificação final dos casos notificados de chikungunya por SE - Brasil - 2026'.upper(), 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Notificados', fontsize=16, labelpad=16)

# O eixo X no pandas fica rotacionado em 90 graus por padrão. Isso conserta e deixa reto:
plt.xticks(rotation=0) 

# Ajustar o limite do eixo Y (adapte se necessário)
# ax.set_ylim(0, 1400) # Removi temporariamente para o pandas ajustar automaticamente às barras somadas

# Melhorar a legenda
ax.legend(title='Classificação Final', fontsize=11, title_fontsize=12, loc='upper right')

ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7) # Tirar as linhas verticais para ficar mais limpo

# Informações rodapé 
ax_rodape = grafico_class.add_subplot(gs[1])
ax_rodape.axis('off') # Minúsculo
rodape_texto = f'Fonte: SINAN | Dados até: {data_analise}'
# Ajustado para ficar centralizado verticalmente no rodapé
ax_rodape.text(0.0, 0.5, rodape_texto, ha='left', va='center', fontsize=10, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_classificacao.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# CASOS PROVÁVEIS POR SEMANA EPIDEMIOLÓGICA (2025 vs 2026)
# 1 e 2. PREPARAÇÃO E AGREGAÇÃO DOS DADOS

# Limpeza e conversão da Semana Epidemiológica (SE) em uma linha
se_2025 = pd.to_numeric(df_provaveis_2025['SE'], errors='coerce').dropna().astype(int)
se_2026 = pd.to_numeric(df_provaveis['SE'], errors='coerce').dropna().astype(int)

# Descobre a última semana disponível em 2026
ultima_se_2026 = se_2026.max()

# Função para filtrar, contar e preencher semanas sem casos com 0
def agrupar_casos(serie_se, ano, max_se):
    df = serie_se[serie_se <= max_se].value_counts().reindex(range(1, max_se + 1), fill_value=0).reset_index()
    df.columns = ['SE', 'Total']
    df['Ano'] = ano
    return df

# Agrupa as duas bases e junta (concatena) de uma vez só
df_comparacao_casos = pd.concat([
    agrupar_casos(se_2025, 'Prováveis 2025', ultima_se_2026),
    agrupar_casos(se_2026, 'Prováveis 2026', ultima_se_2026)
], ignore_index=True)

total_2025 = df_comparacao_casos.loc[df_comparacao_casos['Ano'] == 'Prováveis 2025', 'Total'].sum()
total_2026 = df_comparacao_casos.loc[df_comparacao_casos['Ano'] == 'Prováveis 2026', 'Total'].sum()

# 3. CONFIGURAÇÃO DO GRÁFICO E COLUNAS JUSTAPOSTAS

sns.set_theme(style='whitegrid')
grafico_casos = plt.figure(figsize=(22, 11))
gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_casos.add_subplot(gs[0])

sns.barplot(
    data=df_comparacao_casos, x='SE', y='Total', hue='Ano', hue_order=['Prováveis 2025', 'Prováveis 2026'],
    palette={'Prováveis 2025': '#9db5b2', 'Prováveis 2026': '#cc4e00'}, width=0.9, errorbar=None, ax=ax
)

# 4. RÓTULOS DAS COLUNAS E ESTÉTICA

# Rótulos de dados (escondendo os que são iguais a 0)
for container in ax.containers:
    rotulos = [f'{int(b.get_height())}' if b.get_height() > 0 else '' for b in container]
    ax.bar_label(container, labels=rotulos, padding=3, fontsize=9)

# Títulos e Eixos consolidados
ax.set_title('Casos prováveis de chikungunya por semana epidemiológica - Brasil - 2025 e 2026'.upper(), 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
ax.set_xlim(-1, df_comparacao_casos['SE'].max() + 0)

# Grades e Legenda
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.legend(title='', fontsize=11, loc='upper right', frameon=True)

# 5. RODAPÉ E SALVAMENTO

ax_rodape = grafico_casos.add_subplot(gs[1])
ax_rodape.axis('off')

# Formata a string substituindo a vírgula americana pelo ponto brasileiro
rodape_texto = (f'Fonte: SINAN | Data da análise: {data_analise}')
ax_rodape.text(0.0, 0.3, rodape_texto, ha='left', va='center', fontsize=10, color='black')

# Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_casos_provaveis_2025_2026.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# COMPARAÇÃO DA INCIDÊNCIA SEMANAL DE CHIKUNGUNYA (2025 vs 2026)

# 1. DENOMINADORES POPULACIONAIS

populacao_brasil_2025 = 213_421_037
populacao_brasil_2026 = 214_211_951
constante_incidencia = 100_000

if populacao_brasil_2025 <= 0 or populacao_brasil_2026 <= 0:
    raise ValueError('Informe as populações do Brasil para 2025 e 2026.')

# 2 e 3. PREPARAÇÃO, CONTAGEM E CÁLCULO (VETORIZADO)

# Descobre a última semana disponível em 2026 (limitada até a SE 53)
se_2026_raw = pd.to_numeric(df_provaveis['SE'], errors='coerce').dropna().astype(int)
ultima_se_2026 = se_2026_raw[se_2026_raw.between(1, 53)].max()

# Função unificada para limpar, contar, preencher zeros e calcular incidência
def calcular_incidencia(df_raw, ano, populacao, max_se):
    se = pd.to_numeric(df_raw['SE'], errors='coerce').dropna().astype(int)
    se_valida = se[se.between(1, max_se)] # Filtra entre a SE 1 e a última SE da análise
    
    # Agrupa, preenche semanas vazias com 0 e nomeia colunas
    df = se_valida.value_counts().reindex(range(1, max_se + 1), fill_value=0).reset_index()
    df.columns = ['SE', 'Casos']
    
    # Cálculos
    df['Incidencia'] = (df['Casos'] / populacao) * constante_incidencia
    df['Ano'] = ano
    return df

# Aplica a função para os dois anos
df_inc_2025 = calcular_incidencia(df_provaveis_2025, 'Incidência 2025', populacao_brasil_2025, ultima_se_2026)
df_inc_2026 = calcular_incidencia(df_provaveis, 'Incidência 2026', populacao_brasil_2026, ultima_se_2026)

# Concatena em um único DataFrame
df_comparacao_incidencia = pd.concat([df_inc_2025, df_inc_2026], ignore_index=True)

# 4. CONFIGURAÇÃO DO GRÁFICO E COLUNAS

sns.set_theme(style='whitegrid')
grafico_incidencia = plt.figure(figsize=(22, 11))
gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_incidencia.add_subplot(gs[0])

sns.barplot(
    data=df_comparacao_incidencia, x='SE', y='Incidencia', hue='Ano', hue_order=['Incidência 2025', 'Incidência 2026'],
    palette={'Incidência 2025': '#9db5b2', 'Incidência 2026': '#cc4e00'}, width=0.9, errorbar=None, ax=ax
)

# 5. RÓTULOS DAS COLUNAS, TÍTULOS E EIXOS

# Rótulos (substituindo . por , e ignorando zeros)
for container in ax.containers:
    rotulos = [f'{b.get_height():.1f}'.replace('.', ',') if b.get_height() > 0 else '' for b in container]
    ax.bar_label(container, labels=rotulos, padding=3, fontsize=9)

# Textos e grades
ax.set_title('Coeficiente de incidência de chikungunya por semana epidemiológica - Brasil - 2025 e 2026'.upper(), fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Taxa de Incidência', fontsize=16, labelpad=16)
ax.set_xlim(-1, df_comparacao_incidencia['SE'].max() + 0)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.legend(title='', fontsize=11, loc='upper right', frameon=True)

# 6. RODAPÉ E SALVAMENTO

# Cálculos de incidência acumulada
inc_acumulada_2025 = (df_inc_2025['Casos'].sum() / populacao_brasil_2025) * constante_incidencia
inc_acumulada_2026 = (df_inc_2026['Casos'].sum() / populacao_brasil_2026) * constante_incidencia

ax_rodape = grafico_incidencia.add_subplot(gs[1])
ax_rodape.axis('off')

# String formatada substituindo decimais para o padrão brasileiro
rodape_texto = (f'Fonte: SINAN | Data da análise: {data_analise} | Incidência calculada por 100 mil habitantes')

ax_rodape.text(0.0, 0.5, rodape_texto, ha='left', va='center', fontsize=10, color='black')

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_incidencia_semanal_2025_2026.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

## 2. Distribuição espacial dos casos


In [ ]:
# MAPA COROPLÉTICO DOS CASOS PROVÁVEIS POR MUNICÍPIO

from geobr import read_municipality
from geobr import read_state

with warnings.catch_warnings():

    warnings.simplefilter(
        "ignore",
        urllib3.exceptions.InsecureRequestWarning
    )

# 1. Carregar a malha de municípios do Brasil direto do GitHub do geobr (sem precisar instalar a biblioteca)
mapa_municipios = read_municipality(code_muni='all', year=2025)
mapa_limites_estados = read_state(code_state="all", year=2020)

# 2. Ajustar os códigos da malha municipal para seis dígitos
mapa_municipios['code_muni_6'] = mapa_municipios['code_muni'].astype(str).str[:6]

# 3. Preparar a sua base do SINAN
df_mun_mapa = (temporada_2026["ID_MN_RESI"].dropna().astype(str).str.replace(".0", "", regex=False)
               .value_counts().rename_axis("code_muni_6").reset_index(name="Total"))

# Mecanismo de garantia para que a base do SINAN tenha seis dígitos
df_mun_mapa["code_muni_6"] = (
    df_mun_mapa["code_muni_6"]
    .str.zfill(6))

# 4. Cruzar os dados com o mapa
mapa_casos = mapa_municipios.merge(df_mun_mapa, on='code_muni_6', how='left')
mapa_casos['Total'] = mapa_casos['Total'].fillna(0).astype(int)

# CRIAR AS FAIXAS DE CORES

cortes = [-1, 0, 50, 100, 300, 500, 1000, float('inf')]

rotulos = [
    'Sem casos',
    '1 - 50',
    '51 - 100',
    '101 - 300',
    '301 - 500',
    '501 - 1000',
    '>1000'
]

mapa_casos['Categoria'] = pd.cut(mapa_casos['Total'], bins=cortes, labels=rotulos)

dicionario_cores = {
    'Sem casos': '#f4f4f4',
    '1 - 50': '#fdf0d5',
    '51 - 100': '#fdd49e',
    '101 - 300': '#fc8d59',
    '301 - 500': '#ef6548',
    '501 - 1000': '#b30000',
    '>1000': '#7f0000'
}

mapa_casos['Cor_Estado'] = mapa_casos['Categoria'].map(dicionario_cores)

# PLOTAR O MAPA
fig, ax = plt.subplots(figsize=(12, 12))

# Reduzir o linewidth para 0.1 ou 0.2 (pois o Brasil tem +5.500 municípios e bordas grossas deixam o mapa preto)
mapa_casos.plot(
    color=mapa_casos['Cor_Estado'], 
    linewidth=0.1, 
    edgecolor='0.6', 
    ax=ax
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.5,
    edgecolor="black",
    ax=ax,
)

# CRIAR A LEGENDA CUSTOMIZADA
lista_legendas = [
    mpatches.Patch(
        color=dicionario_cores[rotulo],
        label=rotulo,
    )
    for rotulo in rotulos
]

ax.legend(
    handles=lista_legendas,
    #title= 'Nº de casos prováveis',
    title_fontsize=10,
    fontsize=12,
    ncols=1,
    loc="lower right",
    bbox_to_anchor=(0.25, 0.18),
    frameon=False,
    shadow=False
)

# Estilização Final
ax.set_title(f'Casos prováveis de Chikungunya - SE27 a SE{se_temporada_atual} - 2026'.upper(), 
             fontsize=16, fontweight='bold', pad=8)
ax.axis('off')
ax.text(
    0.4, 0.03,
    f'Fonte: SINAN | Dados até: {data_analise}',
    fontsize=8,
    color='black',
    transform=ax.transAxes,
    ha='left',
    va='top'
)

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_mapa_municipio.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# MAPA COROPLÉTICO DOS CASOS PROVÁVEIS POR MUNICÍPIO (OITO SEMANAS)

from geobr import read_municipality
from geobr import read_state

with warnings.catch_warnings():

    warnings.simplefilter(
        "ignore",
        urllib3.exceptions.InsecureRequestWarning
    )

# 1. Carregar a malha de municípios do Brasil direto do GitHub do geobr (sem precisar instalar a biblioteca)
mapa_municipios = read_municipality(code_muni= 'all', year=2025)
mapa_limites_estados = read_state(code_state="all", year=2020)

# 2. Ajustar os códigos da malha municipal para seis dígitos
mapa_municipios['code_muni_6'] = mapa_municipios['code_muni'].astype(str).str[:6]

# 3. Preparar a sua base do SINAN
df_mun_mapa = (df_provaveis_filtrado["ID_MN_RESI"].dropna().astype(str).str.replace(".0", "", regex=False)
               .value_counts().rename_axis("code_muni_6").reset_index(name="Total"))

# Mecanismo de garantia para que a base do SINAN tenha seis dígitos
df_mun_mapa["code_muni_6"] = (
    df_mun_mapa["code_muni_6"]
    .str.zfill(6))

# 4. Cruzar os dados com o mapa
mapa_casos = mapa_municipios.merge(df_mun_mapa, on='code_muni_6', how='left')
mapa_casos['Total'] = mapa_casos['Total'].fillna(0).astype(int)

# CRIAR AS FAIXAS DE CORES

cortes = [-1, 0, 50, 100, 300, 500, float('inf')]

rotulos = [
    'Sem casos',
    '1 - 50',
    '51 - 100',
    '101 - 300',
    '301 - 500',
    '>500'
]

mapa_casos['Categoria'] = pd.cut(mapa_casos['Total'], bins=cortes, labels=rotulos)

dicionario_cores = {
    'Sem casos': '#edede9',
    '1 - 50': '#ffb627',
    '51 - 100': '#ff9505',
    '101 - 300': '#e2711d',
    '301 - 500': "#cc4e00",
    '>500': '#691e06'
}

mapa_casos['Cor_Estado'] = mapa_casos['Categoria'].map(dicionario_cores)

# PLOTAR O MAPA
fig, ax = plt.subplots(figsize=(12, 12))

# Reduzir o linewidth para 0.1 ou 0.2 (pois o Brasil tem +5.500 municípios e bordas grossas deixam o mapa preto)
mapa_casos.plot(
    color=mapa_casos['Cor_Estado'], 
    linewidth=0.1, 
    edgecolor='0.6', 
    ax=ax
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.5,
    edgecolor="black",
    ax=ax,
)

# CRIAR A LEGENDA CUSTOMIZADA
lista_legendas = [
    mpatches.Patch(
        color=dicionario_cores[rotulo],
        label=rotulo,
    )
    for rotulo in rotulos
]

ax.legend(
    handles=lista_legendas,
    title= f'Nº de casos prováveis - SE{semanas_analise} a SE{se_atual}\n------------------------------------------------',
    title_fontsize=10,
    fontsize=10,
    ncols= 2,
    loc="lower right",
    bbox_to_anchor=(0.35, 0.18),    # Horizontal, vertical
    frameon=False,
    shadow=False
)

# Estilização Final
ax.set_title(f'Casos prováveis de Chikungunya - SE{semanas_analise} a SE{se_atual} - 2026'.upper(), 
             fontsize=16, fontweight='bold', pad=8)
ax.axis('off')
ax.text(
    0.079, 0.18,     # Horizontal, vertical
    f'Fonte: SINAN | Dados até: {data_analise}',
    fontsize=8,
    color='black',
    transform=ax.transAxes,
    ha='left',
    va='top'
)

#salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_mapa_municipio_oito_semanas.png')
#plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from geobr import read_state

with warnings.catch_warnings():
    warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)


# --- FUNÇÃO AUXILIAR DE PROCESSAMENTO ---
def processar_incidencia(df_sinan, df_pop_estado, mapa_estados):
    """Filtra casos do SINAN por código IBGE, consolida população e calcula a incidência."""
    # 1. Contagem de casos por código IBGE (code_state)
    df_casos = (
        df_sinan["SG_UF"]
        .dropna()
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(2)
        .value_counts()
        .rename_axis("code_state")
        .reset_index(name="Total")
    )
    df_casos["code_state"] = df_casos["code_state"].astype(int)

    # 2. Merge do mapa do geobr com os casos e a população
    mapa = mapa_estados.merge(df_casos, on="code_state", how="left")
    mapa = mapa.merge(df_pop_estado, on="abbrev_state", how="left")

    mapa["Total"] = mapa["Total"].fillna(0).astype(int)
    mapa["POPULACAO"] = mapa["POPULACAO"].fillna(1)

    # 3. Cálculo da incidência por 100 mil hab.
    mapa["Incidencia"] = (mapa["Total"] / mapa["POPULACAO"]) * 100000
    return mapa


# --- PREPARAÇÃO DOS DADOS DE POPULAÇÃO ---
df_pop_estado = df_estados[["UF", "POPULACAO"]].copy()
df_pop_estado = df_pop_estado.rename(columns={"UF": "abbrev_state"})
df_pop_estado["POPULACAO"] = (
    df_pop_estado["POPULACAO"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_pop_estado["POPULACAO"] = pd.to_numeric(
    df_pop_estado["POPULACAO"], errors="coerce"
)

# Carrega a malha dos estados
mapa_estados = read_state(code_state="all", year=2020)

# Processa os dois datasets (Anual e Últimas 8 Semanas)

prim_semestre = df_provaveis[df_provaveis['SE'].between(1, 26)]
mapa_acumulado = processar_incidencia(
    prim_semestre, df_pop_estado, mapa_estados
)
mapa_8semanas = processar_incidencia(
    temporada_2026, df_pop_estado, mapa_estados
)


# --- CONFIGURAÇÃO DE FAIXAS E CORES PARA CADA PERÍODO ---

# 1. Faixas para o Acumulado do Ano (Valores maiores)
cortes_ano = [-1, 10, 30, 50, 100, 300, float("inf")]
rotulos_ano = [
    "0,1 - 10",
    "10,1 - 30",
    "30,1 - 50",
    "50,1 - 100",
    "100,1 - 300",
    "Acima de 300",
]

# 2. Faixas para as Últimas 8 Semanas (Valores menores)
cortes_8sem = [-1, 5, 10, 15, 20, 50, float("inf")]
rotulos_8sem = [
    "0,1 - 5",
    "5,1 - 10",
    "10,1 - 15",
    "15,1 - 20",
    "20,1 - 50",
    "Acima de 50",
]

dicionario_cores = {
    "0": "#fff7ec",
    "1": "#fdd49e",
    "2": "#fc8d59",
    "3": "#ef6548",
    "4": "#b30000",
    "5": "#7f0000",
}

# Categorização do Acumulado
mapa_acumulado["Categoria"] = pd.cut(
    mapa_acumulado["Incidencia"], bins=cortes_ano, labels=rotulos_ano
)
mapa_acumulado["Cor"] = mapa_acumulado["Categoria"].map(
    dict(zip(rotulos_ano, dicionario_cores.values()))
)

# Categorização das 8 Semanas
mapa_8semanas["Categoria"] = pd.cut(
    mapa_8semanas["Incidencia"], bins=cortes_8sem, labels=rotulos_8sem
)
mapa_8semanas["Cor"] = mapa_8semanas["Categoria"].map(
    dict(zip(rotulos_8sem, dicionario_cores.values()))
)


# --- CONSTRUÇÃO DA FIGURA (2 SUBPLOTS LADO A LADO) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# ---------------- MAPA 1: ACUMULADO ----------------
mapa_acumulado.plot(
    color=mapa_acumulado["Cor"], linewidth=0.5, edgecolor="black", ax=ax1
)
ax1.set_title(
    f"INCIDÊNCIA SE 1 A SE 26", fontsize=16, fontweight="bold", pad=10
)
ax1.axis("off")

legendas_ano = [
    mpatches.Patch(color=dicionario_cores[str(i)], label=rotulos_ano[i])
    for i in range(len(rotulos_ano))
]
ax1.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=9,
    handles=legendas_ano,
    fontsize=14,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.09, 0.2),
    frameon=False,
)

# ---------------- MAPA 2: ÚLTIMAS 8 SEMANAS ----------------
mapa_8semanas.plot(
    color=mapa_8semanas["Cor"], linewidth=0.5, edgecolor="black", ax=ax2
)
ax2.set_title(
    f"INCIDÊNCIA SE 27 A SE {se_temporada_atual}",
    fontsize=16,
    fontweight="bold",
    pad=10,
)
ax2.axis("off")

legendas_8sem = [
    mpatches.Patch(color=dicionario_cores[str(i)], label=rotulos_8sem[i])
    for i in range(len(rotulos_8sem))
]
ax2.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=9,
    handles=legendas_8sem,
    fontsize=14,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.09, 0.2),
    frameon=False,
)

# --- TÍTULO GERAL E RODAPÉ ---
plt.suptitle(
    "COMPARATIVO DE INCIDÊNCIA DE CHIKUNGUNYA POR UF - BRASIL - 2026",
    fontsize=18,
    fontweight="bold",
    y=0.98,
)

fig.text(
    0.4,
    0.1,
    f"Fonte: SINAN | Dados até: {data_analise}",
    fontsize=9,
    color="black",
    ha="left",
)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

#salvar_pasta = os.path.join(local_pasta, "chik_brasil_incidencia_uf_comparativo_2026.png")
#plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from geobr import read_state

with warnings.catch_warnings():
    warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)

# --- FUNÇÃO AUXILIAR DE PROCESSAMENTO ---
def processar_incidencia(df_sinan, df_pop_estado, mapa_estados):
    """Filtra casos do SINAN por código IBGE, consolida população e calcula a incidência."""
    # 1. Contagem de casos por UF (SG_UF)
    df_casos = (
        df_sinan["SG_UF"]
        .dropna()
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(2)
        .value_counts()
        .rename_axis("code_state")
        .reset_index(name="Total")
    )
    df_casos["code_state"] = df_casos["code_state"].astype(int)

    # 2. Merge do mapa do geobr com os casos e a população
    mapa = mapa_estados.merge(df_casos, on="code_state", how="left")
    mapa = mapa.merge(df_pop_estado, on="abbrev_state", how="left")

    mapa["Total"] = mapa["Total"].fillna(0).astype(int)
    mapa["POPULACAO"] = mapa["POPULACAO"].fillna(1)

    # 3. Cálculo da incidência por 100 mil hab.
    mapa["Incidencia"] = (mapa["Total"] / mapa["POPULACAO"]) * 100000
    return mapa


# --- PREPARAÇÃO DOS DADOS DE POPULAÇÃO ---
df_pop_estado = df_estados[["UF", "POPULACAO"]].copy()
df_pop_estado = df_pop_estado.rename(columns={"UF": "abbrev_state"})
df_pop_estado["POPULACAO"] = (
    df_pop_estado["POPULACAO"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_pop_estado["POPULACAO"] = pd.to_numeric(
    df_pop_estado["POPULACAO"], errors="coerce"
)

# Carrega a malha dos estados
mapa_estados = read_state(code_state="all", year=2020)

# Processa o dataset da temporada
mapa_temporada = processar_incidencia(
    temporada_2026, df_pop_estado, mapa_estados
)

# --- CONFIGURAÇÃO DE FAIXAS E CORES ---
cortes_8sem = [-1, 5, 10, 15, 20, 50, float("inf")]
rotulos_8sem = [
    "0,1 - 5",
    "5,1 - 10",
    "10,1 - 15",
    "15,1 - 20",
    "20,1 - 50",
    "Acima de 50",
]

dicionario_cores = {
    "0": "#fff7ec",
    "1": "#fdd49e",
    "2": "#fc8d59",
    "3": "#ef6548",
    "4": "#b30000",
    "5": "#7f0000",
}

# Categorização
mapa_temporada["Categoria"] = pd.cut(
    mapa_temporada["Incidencia"], bins=cortes_8sem, labels=rotulos_8sem
)
mapa_temporada["Cor"] = mapa_temporada["Categoria"].map(
    dict(zip(rotulos_8sem, dicionario_cores.values()))
)

# --- CONSTRUÇÃO DA FIGURA (MAPA ÚNICO - ALTA RESOLUÇÃO) ---
fig, ax = plt.subplots(1, 1, figsize=(12, 12))

mapa_temporada.plot(
    color=mapa_temporada["Cor"], linewidth=0.5, edgecolor="black", ax=ax
)

ax.axis("off")

legendas_8sem = [
    mpatches.Patch(color=dicionario_cores[str(i)], label=rotulos_8sem[i])
    for i in range(len(rotulos_8sem))
]
ax.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=11,
    handles=legendas_8sem,
    fontsize=14,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.09, 0.2),
    frameon=False,
)

# --- TÍTULO GERAL E RODAPÉ ---
plt.suptitle(
    f"INCIDÊNCIA DE CHIKUNGUNYA POR UF\nBRASIL - SE27 a SE {se_temporada_atual} - 2026".upper(),
    fontsize=18,
    fontweight="bold",
    y=0.9,
)

fig.text(
    0.41,
    0.07,
    f"Fonte: SINAN | Dados até: {data_analise}",
    fontsize=9,
    color="black",
    ha="left",
)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])

salvar_pasta = os.path.join(local_pasta, "chik_brasil_incidencia_uf_2026.png")
plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from geobr import read_municipality, read_state

with warnings.catch_warnings():
    warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)

# --- 1. PREPARAÇÃO DA BASE DE POPULAÇÃO MUNICIPAL ---
df_populacao = df_municipios.copy()
df_populacao["code_muni_6"] = (
    df_populacao["COD_IBGE"].astype(str).str.replace(".0", "", regex=False).str.strip().str.zfill(6).str[:6]
)

# Garantir que a população é numérica
df_populacao["POPULACAO"] = (
    df_populacao["POPULACAO"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_populacao["POPULACAO"] = pd.to_numeric(df_populacao["POPULACAO"], errors="coerce")


# --- 2. CARREGAR MALHA MUNICIPAL E ESTADUAL (geobr) ---
mapa_limites_estados = read_state(code_state="all", year=2020)
mapa_municipios = read_municipality(code_muni="all", year=2020)
mapa_municipios["code_muni_6"] = mapa_municipios["code_muni"].astype(str).str[:6]


# --- 3. FUNÇÃO AUXILIAR DE PROCESSAMENTO MUNICIPAL ---
def processar_incidencia_muni(df_sinan, df_pop, mapa_muni):
    df_casos = (
        df_sinan["ID_MN_RESI"]
        .dropna()
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(6)
        .str[:6]
        .value_counts()
        .rename_axis("code_muni_6")
        .reset_index(name="Total")
    )

    mapa = mapa_muni.merge(df_casos, on="code_muni_6", how="left")
    mapa = mapa.merge(df_pop[["code_muni_6", "POPULACAO"]], on="code_muni_6", how="left")

    mapa["Total"] = mapa["Total"].fillna(0).astype(int)
    mapa["POPULACAO"] = mapa["POPULACAO"].fillna(1)  # Evita divisão por zero

    mapa["Incidencia"] = (mapa["Total"] / mapa["POPULACAO"]) * 100000
    return mapa


# Processar os dois datasets
mapa_acumulado = processar_incidencia_muni(prim_semestre, df_populacao, mapa_municipios)
mapa_8semanas = processar_incidencia_muni(temporada_2026, df_populacao, mapa_municipios)


# --- 4. DEFINIÇÃO DE FAIXAS E CORES PARA CADA PERÍODO ---

# Faixas para o Acumulado do Ano (Iniciando em -0.001 para isolar o zero)
cortes_ano = [-0.001, 0, 10, 30, 50, 100, 500, float("inf")]
rotulos_ano = [
    'Sem Casos',
    '0,1 - 10',
    '10,1 - 30',
    '30,1 - 50',
    '50,1 - 100',
    '100,1 - 500',
    'Acima de 500'
]

# Faixas para as Últimas 8 Semanas (Iniciando em -0.001 para isolar o zero)
cortes_8sem = [-0.001, 0, 10, 30, 50, 100, 500, float("inf")]
rotulos_8sem = [
    "Sem Casos",
    "0,1 - 10",
    "10,1 - 30",
    "30,1 - 50",
    "50,1 - 100",
    "100,1 - 500",
    "Acima de 500",
]

# Dicionário de cores unificado mapeado diretamente pelos rótulos de texto
dicionario_cores = {
    "Sem Casos": "#e9ecef",
    "0,1 - 10": "#fff7ec",
    "10,1 - 30": "#fdd49e",
    "30,1 - 50": "#fc8d59",
    "50,1 - 100": "#ef6548",
    "100,1 - 500": "#b30000",
    "Acima de 500": "#7f0000",
}

# Categorização do Acumulado
mapa_acumulado["Categoria"] = pd.cut(mapa_acumulado["Incidencia"], bins=cortes_ano, labels=rotulos_ano)
mapa_acumulado["Cor"] = mapa_acumulado["Categoria"].map(dicionario_cores)

# Categorização das 8 Semanas
mapa_8semanas["Categoria"] = pd.cut(mapa_8semanas["Incidencia"], bins=cortes_8sem, labels=rotulos_8sem)
mapa_8semanas["Cor"] = mapa_8semanas["Categoria"].map(dicionario_cores)


# --- 5. CONSTRUÇÃO DA FIGURA (2 MAPAS LADO A LADO) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 11))

# ---------------- MAPA 1: ACUMULADO DO ANO ----------------
mapa_acumulado.plot(
    color=mapa_acumulado["Cor"],
    linewidth=0.05,
    edgecolor="0.7",
    ax=ax1,
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.8,
    edgecolor="black",
    ax=ax1,
)

ax1.set_title(f"INCIDÊNCIA SE 1 A SE 26", fontsize=16, fontweight="bold", pad=10)
ax1.axis("off")

legendas_ano = [
    mpatches.Patch(color=dicionario_cores[rotulo], label=rotulo)
    for rotulo in rotulos_ano
]
ax1.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=9,
    handles=legendas_ano,
    fontsize=14,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.09, 0.2),
    frameon=False,
)

# ---------------- MAPA 2: ÚLTIMAS 8 SEMANAS ----------------
mapa_8semanas.plot(
    color=mapa_8semanas["Cor"],
    linewidth=0.05,
    edgecolor="0.7",
    ax=ax2,
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.8,
    edgecolor="black",
    ax=ax2,
)

ax2.set_title(
    f"INCIDÊNCIA SE 27 A SE {se_temporada_atual}",
    fontsize=16,
    fontweight="bold",
    pad=10,
)
ax2.axis("off")

legendas_8sem = [
    mpatches.Patch(color=dicionario_cores[rotulo], label=rotulo)
    for rotulo in rotulos_8sem
]
ax2.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=9,
    handles=legendas_8sem,
    fontsize=14,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.09, 0.2),
    frameon=False,
)

# --- TÍTULO GERAL E RODAPÉ ---
plt.suptitle(
    "COMPARATIVO DE INCIDÊNCIA DE CHIKUNGUNYA POR MUNICÍPIO - BRASIL - 2026",
    fontsize=18,
    fontweight="bold",
    y=0.98,
)

fig.text(
    0.4,
    0.1,
    f"Fonte: SINAN | Dados até: {data_analise}",
    fontsize=9,
    color="black",
    ha="left",
)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

#salvar_pasta = os.path.join(local_pasta, "chik_brasil_incidencia_municipio_comparativo_2026.png")
#plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from geobr import read_municipality, read_state

with warnings.catch_warnings():
    warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)


# --- 1. PREPARAÇÃO DA BASE DE POPULAÇÃO MUNICIPAL ---
df_populacao = df_municipios.copy()
df_populacao["code_muni_6"] = (
    df_populacao["COD_IBGE"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
    .str.zfill(6)
    .str[:6]
)

# Garantir que a população é numérica
df_populacao["POPULACAO"] = (
    df_populacao["POPULACAO"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_populacao["POPULACAO"] = pd.to_numeric(df_populacao["POPULACAO"], errors="coerce")


# --- 2. CARREGAR MALHA MUNICIPAL E ESTADUAL (geobr) ---
mapa_limites_estados = read_state(code_state="all", year=2020)
mapa_municipios = read_municipality(code_muni="all", year=2020)
mapa_municipios["code_muni_6"] = mapa_municipios["code_muni"].astype(str).str[:6]


# --- 3. FUNÇÃO AUXILIAR DE PROCESSAMENTO MUNICIPAL ---
def processar_incidencia_muni(df_sinan, df_pop, mapa_muni):
    df_casos = (
        df_sinan["ID_MN_RESI"]
        .dropna()
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(6)
        .str[:6]
        .value_counts()
        .rename_axis("code_muni_6")
        .reset_index(name="Total")
    )

    mapa = mapa_muni.merge(df_casos, on="code_muni_6", how="left")
    mapa = mapa.merge(df_pop[["code_muni_6", "POPULACAO"]], on="code_muni_6", how="left")

    mapa["Total"] = mapa["Total"].fillna(0).astype(int)
    mapa["POPULACAO"] = mapa["POPULACAO"].fillna(1)  # Evita divisão por zero

    mapa["Incidencia"] = (mapa["Total"] / mapa["POPULACAO"]) * 100000
    return mapa


# Processar apenas o dataset recente (SE 27 em diante)
mapa_8semanas = processar_incidencia_muni(temporada_2026, df_populacao, mapa_municipios)


# --- 4. DEFINIÇÃO DE FAIXAS E CORES ---
cortes_8sem = [-0.001, 0, 10, 30, 50, 100, 500, float("inf")]

rotulos_8sem = [
    "Sem Casos",
    "0,1 - 10",
    "10,1 - 30",
    "30,1 - 50",
    "50,1 - 100",
    "100,1 - 500",
    "Acima de 500",
]

# Dicionário mapeando diretamente o texto do rótulo para a cor correspondente
dicionario_cores = {
    "Sem Casos": "#e9ecef",    # Ou use um tom neutro como "#f4f4f4"
    "0,1 - 10": "#fff7ec",
    "10,1 - 30": "#fdd49e",
    "30,1 - 50": "#fc8d59",
    "50,1 - 100": "#ef6548",
    "100,1 - 500": "#b30000",
    "Acima de 500": "#7f0000",
}

# Categorização
mapa_8semanas["Categoria"] = pd.cut(
    mapa_8semanas["Incidencia"], bins=cortes_8sem, labels=rotulos_8sem
)
mapa_8semanas["Cor"] = mapa_8semanas["Categoria"].map(dicionario_cores)

# --- 5. CONSTRUÇÃO DA FIGURA (MAPA ÚNICO - ALTA RESOLUÇÃO) ---
fig, ax = plt.subplots(1, 1, figsize=(14, 12))

mapa_8semanas.plot(
    color=mapa_8semanas["Cor"],
    linewidth=0.05,
    edgecolor="0.7",
    ax=ax,
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.8,
    edgecolor="black",
    ax=ax,
)

ax.axis("off")

legendas_8sem = [
    mpatches.Patch(color=dicionario_cores[rotulo], label=rotulo)
    for rotulo in rotulos_8sem
]

ax.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=11,
    handles=legendas_8sem,
    fontsize=14,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.09, 0.2),
    frameon=False,
)

# --- TÍTULO GERAL E RODAPÉ ---
plt.suptitle(
    f"INCIDÊNCIA DE CHIKUNGUNYA POR MUNICÍPIO\nBRASIL - SE 27 a SE {se_temporada_atual} - 2026".upper(),
    fontsize=18,
    fontweight="bold",
    y=0.90,
)

fig.text(
    0.41,
    0.07,
    f"Fonte: SINAN | Dados até: {data_analise}",
    fontsize=9,
    color="black",
    ha="left",
)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])

salvar_pasta = os.path.join(local_pasta, "chik_brasil_incidencia_municipio_2026.png")
plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# MAPA COROPLÉTICO DE INCIDÊNCIA POR REGIÃO DE SAÚDE

# 1. CONFIGURAÇÃO DE CAMINHOS
pasta_atual = Path('.').resolve()
root = pasta_atual.parent

base_dados = root / 'BASES_DADOS'
base_ibge = root / 'BASE_IBGE'

arquivo_malha = base_ibge / "Regioes_Saude_Brasil.gpkg"
arquivo_vinculo = base_ibge / "vinculo_municipio_regiao.csv"

if not base_dados.exists():
    raise FileNotFoundError('A pasta não foi encontrada. Verificar!')


# 2. FUNÇÕES AUXILIARES (VETORIZADAS)
def padronizar_codigo_municipal(serie):
    """Preserva códigos de 6 dígitos ou remove o dígito verificador de 7 dígitos."""
    codigo = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )
    return codigo.where(codigo.str.fullmatch(r"\d{6,7}", na=False)).str[:6]


def converter_populacao(serie):
    """Converte valores de população de forma vetorizada e robusta."""
    s = serie.astype("string").str.strip()
    
    # Substituições vetorizadas para formatos com vírgula/ponto
    has_comma = s.str.contains(",", na=False)
    
    clean_s = s.copy()
    clean_s.loc[has_comma] = (
        clean_s.loc[has_comma]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    
    # Remove pontos de milhar caso não haja vírgula
    has_dot_thousands = ~has_comma & clean_s.str.contains(r"\d{1,3}(?:\.\d{3})+", regex=True)
    clean_s.loc[has_dot_thousands] = clean_s.loc[has_dot_thousands].str.replace(".", "", regex=False)
    
    return pd.to_numeric(clean_s, errors="coerce")


# 3. CARREGAR MALHAS GEOGRÁFICAS
mapa_regioes = (
    gpd.read_file(arquivo_malha, layer="regioes_saude")[
        ["COD_REGIAO_SAUDE", "REGIAO_SAUDE", "geometry"]
    ]
    .rename(columns={
        "COD_REGIAO_SAUDE": "code_health_region",
        "REGIAO_SAUDE": "name_health_region"
    })
    .assign(code_health_region=lambda x: x["code_health_region"].astype("string").str.strip())
)

mapa_limites_estados = gpd.read_file(arquivo_malha, layer="limites_uf").to_crs(mapa_regioes.crs)


# 4. CARREGAR E VALIDAR VÍNCULO MUNICÍPIO–REGIÃO
vinculo = (
    pd.read_csv(arquivo_vinculo, dtype="string", encoding="utf-8-sig")
    .assign(code_muni_6=lambda df: padronizar_codigo_municipal(df["COD_IBGE"]))
    [["code_muni_6", "Municipio", "UF", "COD_REGIAO_SAUDE", "REGIAO_SAUDE"]]
    .rename(columns={
        "Municipio": "name_muni",
        "UF": "abbrev_state",
        "COD_REGIAO_SAUDE": "code_health_region",
        "REGIAO_SAUDE": "name_health_region"
    })
)

if vinculo.isna().any().any():
    raise ValueError("Há campos ausentes ou códigos inválidos no vínculo.")
if vinculo["code_muni_6"].duplicated().any():
    raise ValueError("Há municípios duplicados na tabela de vínculo.")
if mapa_regioes["code_health_region"].duplicated().any():
    raise ValueError("Há regiões duplicadas na malha.")
if set(vinculo["code_health_region"]) != set(mapa_regioes["code_health_region"]):
    raise ValueError("Os códigos regionais do vínculo e da malha são diferentes.")


# 5. CONTABILIZAR CASOS POR MUNICÍPIO
df_casos_municipio = (
    padronizar_codigo_municipal(temporada_2026["ID_MN_RESI"])
    .value_counts(dropna=False)
    .rename_axis("code_muni_6")
    .reset_index(name="Total")
)

casos_sem_vinculo = df_casos_municipio.loc[
    ~df_casos_municipio["code_muni_6"].isin(vinculo["code_muni_6"])
].copy()

if not casos_sem_vinculo.empty:
    print("\nCasos sem vínculo com a regionalização:")
    print(casos_sem_vinculo.to_string(index=False))
    print(f"\nTotal de casos não incluídos no mapa: {int(casos_sem_vinculo['Total'].sum())}")


# 6. PREPARAR POPULAÇÃO MUNICIPAL
df_populacao = (
    df_municipios[["COD_IBGE", "POPULACAO"]]
    .assign(
        code_muni_6=lambda df: padronizar_codigo_municipal(df["COD_IBGE"]),
        POPULACAO=lambda df: converter_populacao(df["POPULACAO"])
    )
    .assign(POPULACAO=lambda df: np.where(df["POPULACAO"].gt(0), df["POPULACAO"], np.nan))
    [["code_muni_6", "POPULACAO"]]
    .dropna(subset=["code_muni_6"])
    .drop_duplicates()
)

if df_populacao["code_muni_6"].duplicated().any():
    raise ValueError("Há mais de uma população para o mesmo município.")


# 7. INTEGRAR DADOS MUNICIPAIS
dados_municipais = (
    vinculo
    .merge(df_casos_municipio, on="code_muni_6", how="left", validate="one_to_one")
    .merge(df_populacao, on="code_muni_6", how="left", validate="one_to_one")
    .assign(Total=lambda df: df["Total"].fillna(0).astype(int))
)

casos_mapeados = int(dados_municipais["Total"].sum())
casos_nao_mapeados = int(casos_sem_vinculo["Total"].sum())

if casos_mapeados + casos_nao_mapeados != len(temporada_2026):
    raise ValueError("Houve divergência na contabilização dos casos.")


# 8. AGREGAR POR REGIÃO DE SAÚDE E CALCULAR INCIDÊNCIA
df_regioes = (
    dados_municipais
    .groupby(["code_health_region", "name_health_region"], as_index=False)
    .agg(
        Total=("Total", "sum"),
        POPULACAO=("POPULACAO", "sum"),
        Municipios=("code_muni_6", "nunique"),
        Municipios_sem_populacao=("POPULACAO", lambda x: x.isna().sum())
    )
    .assign(
        POPULACAO=lambda df: np.where(
            df["Municipios_sem_populacao"].eq(0) & df["POPULACAO"].gt(0),
            df["POPULACAO"],
            np.nan
        ),
        Incidencia=lambda df: df["Total"].div(df["POPULACAO"]).mul(100_000)
    )
)


# 9. ASSOCIAR À MALHA E CLASSIFICAR
# Ajustado para usar -0.001 para capturar o zero de forma limpa
cortes = [-0.001, 0, 10, 50, 100, 200, 300, np.inf]

rotulos = [
    "Sem casos",
    "0,1 a 10",
    "10,1 a 50",
    "50,1 a 100",
    "100,1 a 200",
    "200,1 a 300",
    "Acima de 300"
]
sem_populacao = "População incompleta ou indisponível"

# Chaves padronizadas idênticas aos elementos da lista 'rotulos'
dicionario_cores = {
    "Sem casos": "#e9ecef",         # Corrigido para "Sem casos" (minúsculo)
    "0,1 a 10": "#fff7ec",
    "10,1 a 50": "#fdd49e",
    "50,1 a 100": "#fc8d59",
    "100,1 a 200": "#ef6548",
    "200,1 a 300": "#b30000",
    "Acima de 300": "#7f0000",
    sem_populacao: "#d9d9d9"
}

mapa_casos = (
    mapa_regioes
    .merge(
        df_regioes.drop(columns="name_health_region"),
        on="code_health_region",
        how="left",
        validate="one_to_one"
    )
    .assign(
        Total=lambda df: df["Total"].astype(int),
        Categoria=lambda df: pd.cut(df["Incidencia"], bins=cortes, labels=rotulos, right=True),
        Categoria_mapa=lambda df: df["Categoria"].astype("string").fillna(sem_populacao),
        Cor_Regiao=lambda df: df["Categoria_mapa"].map(dicionario_cores)
    )
)

if mapa_casos["Total"].isna().any():
    raise ValueError("Há regiões da malha sem correspondência nos indicadores.")


# 10. RELATÓRIO DE PROBLEMAS POPULACIONAIS
problemas_populacao = dados_municipais.loc[
    dados_municipais["POPULACAO"].isna(),
    ["code_muni_6", "name_muni", "abbrev_state", "code_health_region", "name_health_region", "Total"]
]

if not problemas_populacao.empty:
    print("\nMunicípios sem população válida:")
    print(problemas_populacao.to_string(index=False))

# 11. PLOTAGEM DO MAPA
fig, ax = plt.subplots(figsize=(12, 12))

mapa_casos.plot(
    color=mapa_casos["Cor_Regiao"],
    linewidth=0.35,
    edgecolor="0.55",
    ax=ax
)

mapa_limites_estados.boundary.plot(
    linewidth=0.5,
    color="black",
    ax=ax
)

# Legenda
ordem_legenda = rotulos.copy()
if mapa_casos["Incidencia"].isna().any():
    ordem_legenda.append(sem_populacao)

lista_legendas = [
    mpatches.Patch(facecolor=dicionario_cores[rotulo], edgecolor="none", label=rotulo)
    for rotulo in ordem_legenda
]

ax.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=11,
    handles=lista_legendas,
    fontsize=12,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.08, 0.17),
    frameon=False,
)

# Títulos e Rodapé (Correção do erro de sintaxe do hífen no f-string anterior)
ax.set_title(
    "INCIDÊNCIA DE CASOS PROVÁVEIS DE CHIKUNGUNYA\n"
    f"REGIÃO DE SAÚDE - SE {se_temporada_inicial} A SE {se_temporada_atual} - 2026",
    fontsize=18,
    fontweight="bold",
    pad=10
)

ax.axis("off")

ax.text(
    0.37,
    -0.01,
    f"Fonte: SINAN | Dados até: {data_analise}\n",
    fontsize=9,
    color="black",
    transform=ax.transAxes,
    ha="left",
    va="bottom"
)

# SALVAR
salvar_pasta = Path(local_pasta) / ("chik_brasil_incidencia_regiao_saude.png")
fig.savefig(salvar_pasta, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# MAPA DE CALOR DE CHIKUNGUNYA | SE01–SE DE ANÁLISE / 2026
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from matplotlib.cm import ScalarMappable
from scipy.ndimage import gaussian_filter
from shapely import contains_xy

semana_calor = int(se_analise)
if not 1 <= semana_calor <= 53:
    raise ValueError(f'Semana epidemiológica inválida: {semana_calor}')

# Fontes geográficas em BASE_IBGE. Se a malha compactada não estiver nessa
# pasta, reutilizar a malha geobr 2025 carregada acima no próprio notebook.
malha_calor_zip = Path(base_ibge) / 'BR_Municipios_2025.zip'
malha_calor_shp = Path(base_ibge) / 'BR_Municipios_2025.shp'
if malha_calor_zip.is_file():
    municipios_calor = gpd.read_file('zip://' + str(malha_calor_zip.resolve()))
elif malha_calor_shp.is_file():
    municipios_calor = gpd.read_file(malha_calor_shp)
elif 'mapa_municipios' in globals() and isinstance(mapa_municipios, gpd.GeoDataFrame):
    municipios_calor = mapa_municipios.copy()
    if 'code_muni' not in municipios_calor.columns:
        raise ValueError('A malha geobr existente não contém code_muni.')
    municipios_calor['CD_MUN'] = (
        municipios_calor['code_muni'].astype('string').str.replace(r'\.0$', '', regex=True)
    )
    codigos_uf_calor = {
        '11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
        '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL',
        '28':'SE','29':'BA','31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR',
        '42':'SC','43':'RS','50':'MS','51':'MT','52':'GO','53':'DF'
    }
    municipios_calor['SIGLA_UF'] = municipios_calor['CD_MUN'].str[:2].map(codigos_uf_calor)
else:
    raise FileNotFoundError(
        'Coloque BR_Municipios_2025.zip (ou os arquivos do shapefile) em BASE_IBGE, '
        'ou execute antes a célula que define mapa_municipios com a malha geobr 2025.'
    )

municipios_calor = municipios_calor[['CD_MUN', 'SIGLA_UF', 'geometry']].copy()
municipios_calor['CD_MUN'] = municipios_calor['CD_MUN'].astype('string').str.replace(r'\.0$', '', regex=True)
# O shapefile IBGE 2025 também inclui duas áreas operacionais em lagoas do RS.
municipios_calor = municipios_calor.loc[
    ~municipios_calor['CD_MUN'].isin(['4300001', '4300002'])
].copy()
municipios_calor['MUN6'] = municipios_calor['CD_MUN'].str[:6]
if municipios_calor['MUN6'].isna().any() or municipios_calor['MUN6'].duplicated().any():
    raise ValueError('A malha apresenta códigos municipais ausentes ou repetidos.')
if municipios_calor['SIGLA_UF'].isna().any():
    raise ValueError('A malha apresenta UF não identificada.')
if municipios_calor.crs is None:
    raise ValueError('A malha municipal não informa o sistema de coordenadas.')
municipios_calor = municipios_calor.to_crs('EPSG:5880')  # SIRGAS 2000 / Brasil Policônica (metros)
estados_calor = municipios_calor.dissolve(by='SIGLA_UF', as_index=False)

# SEM_PRI codifica ano e semana de início dos sintomas; ID_MN_RESI é a residência.
se_inicio_calor = pd.to_numeric(temporada_2026['SEM_PRI'], errors='coerce')
recorte_calor = temporada_2026.loc[
    se_inicio_calor.between(202601, 202600 + semana_calor), ['ID_MN_RESI']
].copy()
recorte_calor['MUN6'] = (
    recorte_calor['ID_MN_RESI'].astype('string').str.strip()
    .str.replace(r'\.0$', '', regex=True).str.zfill(6)
)
recorte_calor['MUN6'] = recorte_calor['MUN6'].where(
    recorte_calor['MUN6'].str.fullmatch(r'\d{6}', na=False)
)
contagens_calor = recorte_calor.groupby('MUN6', dropna=False).size().rename('casos')
sem_malha_calor = contagens_calor.loc[
    ~contagens_calor.index.isin(municipios_calor['MUN6'])
]
mapa_calor_mun = municipios_calor.merge(
    contagens_calor.reset_index(), on='MUN6', how='left', validate='one_to_one'
)
mapa_calor_mun['casos'] = mapa_calor_mun['casos'].fillna(0).astype(int)
print(f'Chikungunya SE01–SE{semana_calor:02d}/2026: {len(recorte_calor):,} casos; '
      f'{mapa_calor_mun["casos"].sum():,} localizados; '
      f'{int(sem_malha_calor.sum()):,} sem município na malha.')
if len(sem_malha_calor):
    print('Códigos sem correspondência:', sem_malha_calor.to_dict())
assert mapa_calor_mun['casos'].sum() + sem_malha_calor.sum() == len(recorte_calor)

limites_calor = tuple(municipios_calor.total_bounds)  # xmin, ymin, xmax, ymax
passo_calor = 3000.0  # metros
largura_calor = int(np.ceil((limites_calor[2]-limites_calor[0]) / passo_calor))
altura_calor = int(np.ceil((limites_calor[3]-limites_calor[1]) / passo_calor))
# Máscara territorial por teste vetorizado de pertinência. A função Shapely
# contém os centros dos pixels nas geometrias estaduais; não requer Rasterio.
passo_x_calor = (limites_calor[2]-limites_calor[0]) / largura_calor
passo_y_calor = (limites_calor[3]-limites_calor[1]) / altura_calor
x_centros_calor = limites_calor[0] + (np.arange(largura_calor)+.5) * passo_x_calor
y_centros_calor = limites_calor[3] - (np.arange(altura_calor)+.5) * passo_y_calor
mascara_calor = np.zeros((altura_calor, largura_calor), dtype=bool)
for geometria_uf_calor in estados_calor.geometry:
    x0_uf, y0_uf, x1_uf, y1_uf = geometria_uf_calor.bounds
    colunas_uf = np.flatnonzero((x_centros_calor >= x0_uf) & (x_centros_calor <= x1_uf))
    linhas_uf = np.flatnonzero((y_centros_calor >= y0_uf) & (y_centros_calor <= y1_uf))
    if len(colunas_uf) and len(linhas_uf):
        mascara_calor[np.ix_(linhas_uf, colunas_uf)] |= contains_xy(
            geometria_uf_calor,
            x_centros_calor[colunas_uf][None, :],
            y_centros_calor[linhas_uf][:, None]
        )
ativos_calor = mapa_calor_mun.loc[mapa_calor_mun['casos'] > 0]
pontos_calor = ativos_calor.geometry.representative_point()
coluna_calor = np.floor((pontos_calor.x.to_numpy()-limites_calor[0]) /
                         (limites_calor[2]-limites_calor[0]) * largura_calor).astype(int)
linha_calor = np.floor((limites_calor[3]-pontos_calor.y.to_numpy()) /
                        (limites_calor[3]-limites_calor[1]) * altura_calor).astype(int)
impulsos_calor = np.zeros((altura_calor, largura_calor), dtype=np.float64)
np.add.at(impulsos_calor,
          (np.clip(linha_calor,0,altura_calor-1), np.clip(coluna_calor,0,largura_calor-1)),
          ativos_calor['casos'].to_numpy())
densidade_calor = gaussian_filter(
    impulsos_calor, sigma=30_000/passo_calor, mode='constant', truncate=4.0
) / (passo_calor/1000)**2
positivos_calor = densidade_calor[mascara_calor & (densidade_calor > 0)]
if not len(positivos_calor):
    raise ValueError('Não há casos com localização válida para produzir o mapa de calor.')
teto_calor = float(np.percentile(positivos_calor, 99.5))
densidade_calor = np.ma.masked_where(
    (~mascara_calor) | (densidade_calor < 0.00006), densidade_calor
)

cores_calor = LinearSegmentedColormap.from_list(
    'cgarb_chik',
    ['#3188bb','#72c7b7','#d6e88e','#fff0a3','#fdb46b','#ec653f','#bc283a'],
    N=256
)
escala_calor = PowerNorm(gamma=0.72, vmin=0, vmax=teto_calor, clip=True)
fig_calor = plt.figure(figsize=(12,12), facecolor='white')
eixo_calor = fig_calor.add_axes([.015,.15,.97,.78])
eixo_calor.set_facecolor('white')
eixo_calor.imshow(
    densidade_calor,
    extent=(limites_calor[0],limites_calor[2],limites_calor[1],limites_calor[3]),
    origin='upper', cmap=cores_calor, norm=escala_calor,
    interpolation='bilinear', zorder=1
)
mapa_calor_mun.geometry.simplify(1800, preserve_topology=True).boundary.plot(
    ax=eixo_calor, color='#415463', linewidth=.16, alpha=.65, zorder=2
)
estados_calor.boundary.plot(
    ax=eixo_calor, color='#263b4b', linewidth=.8, alpha=.95, zorder=3
)
for _, estado_calor in estados_calor.iterrows():
    centro_calor = estado_calor.geometry.representative_point()
    eixo_calor.text(
        centro_calor.x, centro_calor.y, estado_calor['SIGLA_UF'],
        ha='center', va='center', fontsize=8.5, color='#000000', weight='bold',
        path_effects=[pe.withStroke(linewidth=2.4, foreground='white')], zorder=4
    )
eixo_calor.set_xlim(limites_calor[0], limites_calor[2])
eixo_calor.set_ylim(limites_calor[1], limites_calor[3])
eixo_calor.set_aspect('equal')
eixo_calor.axis('off')
fig_calor.text(
    .5, .960,
    f'Mapa de calor de casos prováveis de chikungunya\nBrasil - SE27 e SE{semana_calor:02d} - 2026'.upper(),
    ha='center', va='center', fontsize=18, weight='bold', color='#000000'
)
eixo_escala_calor = fig_calor.add_axes([.27,.105,.46,.025])
barra_calor = fig_calor.colorbar(
    ScalarMappable(norm=escala_calor, cmap=cores_calor),
    cax=eixo_escala_calor, orientation='horizontal'
)
barra_calor.set_ticks([])
barra_calor.outline.set_visible(False)
fig_calor.text(.27,.083,'Menor intensidade',ha='left',fontsize=10,color='#000000')
fig_calor.text(.73,.083,'Maior intensidade',ha='right',fontsize=10,color='#000000')

arquivo_calor = Path(local_pasta) / f'chik_brasil_mapa_calor.png'
fig_calor.savefig(arquivo_calor, dpi=300, facecolor='white')
plt.show()

## 3. Concentração territorial dos casos


In [ ]:
from PIL import Image

# Remove o limite de pixels para evitar o erro de decompression bomb
Image.MAX_IMAGE_PIXELS = None

# Casos prováveis por semana epidemiológica: 27 UFs na mesma figura
se_final_grafico = se_analise
semanas = range(1, se_final_grafico + 1)

dados_grafico = df_provaveis[['SG_UF', 'SE']].copy()
dados_grafico['SG_UF'] = pd.to_numeric(dados_grafico['SG_UF'], errors='coerce')
dados_grafico['SE'] = pd.to_numeric(dados_grafico['SE'], errors='coerce')
dados_grafico = dados_grafico[dados_grafico['SE'].between(1, se_final_grafico)]

casos_uf_se = dados_grafico.groupby(['SG_UF', 'SE']).size()
ufs = sorted(mapa_uf.items(), key=lambda item: item[1])

# Aumentamos um pouco a proporção horizontal mantendo a altura controlada (ex: 32x16)
fig, eixos = plt.subplots(4, 7, figsize=(35, 16), sharex=True)

fig.suptitle(
    f'Casos prováveis de chikungunya por semana epidemiológica e UF, SE 1 a SE {se_analise}, 2026'.upper(),
    fontsize=24, fontweight='bold', y=0.98
)

for ax, (codigo_uf, sigla_uf) in zip(eixos.flat, ufs):
    contagens = (
        casos_uf_se.loc[codigo_uf].reindex(semanas, fill_value=0)
        if codigo_uf in casos_uf_se.index.get_level_values('SG_UF')
        else pd.Series(0, index=semanas)
    )

    ax.bar(
        semanas, contagens,
        color=['#fdd49e' if se <= 26 else '#c32f27' for se in semanas],
        width=0.88
    )
    ax.axvline(26.5, color='#74879a', linestyle='--', linewidth=1.5)
    
    # Título do subplot proporcional e dinâmico
    ax.set_title(
        f'{sigla_uf} | {int(contagens.sum()):,} casos (SE1 - SE{se_analise})'.replace(',', '.'),
        fontsize=14, fontweight='bold'
    )
    
    ax.set_xlim(0, se_final_grafico + 1)
    ax.set_xticks([1, 5, 10, 15, 20, 25, 30, 35, 40, se_analise])
    
    # Fontes dos eixos proporcionais para não embolar
    ax.tick_params(axis='x', labelbottom=True, labelsize=10)    
    ax.tick_params(axis='y', labelsize=10)
    
    ax.grid(axis='x', linewidth=0.8, linestyle='--', alpha=0.2)
    ax.grid(axis='y', linewidth=0.8, linestyle='--', alpha=0.2)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

# Ocultar os eixos vazios extras (já que 4 linhas x 7 colunas = 28 espaços, e temos 27 UFs)
for ax in eixos.flat[len(ufs):]:
    ax.set_visible(False)

fig.supxlabel('Semana Epidemiológica', fontsize=16, y=0.04)
fig.supylabel('Casos Prováveis', fontsize=16, x=0.04)

fig.tight_layout(rect=(0.04, 0.04, 0.98, 0.96), h_pad=2.5, w_pad=1.5)

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_casos_provaveis_painel_uf.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from pathlib import Path
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from openpyxl.styles import Font, Alignment


# ==========================================================
# FUNÇÕES VISUAIS COMPARTILHADAS
# Utilizadas também pelo bloco da tabela top 15.
# ==========================================================

def formatar_valor_tabela_padronizada(valor, casas=0):
    if pd.isna(valor):
        return "—"

    return (
        f"{valor:,.{casas}f}"
        .replace(",", "@")
        .replace(".", ",")
        .replace("@", ".")
    )


def desenhar_tabela_padronizada(
    linhas,
    cabecalhos,
    larguras,
    titulo,
    rodape,
    caminho_saida,
    coluna_variacao=None,
    valores_variacao=None,
    destacar_total=True
):
    linhas_visuais = [
        [
            textwrap.fill(str(valor), width=32)
            if coluna == 0 else str(valor)
            for coluna, valor in enumerate(linha)
        ]
        for linha in linhas
    ]

    soma_larguras = sum(larguras)
    larguras = [
        largura / soma_larguras
        for largura in larguras
    ]

    altura_cabecalho = 0.65

    alturas_linhas = [
        0.37 + 0.20 * max(
            str(valor).count("\n")
            for valor in linha
        )
        for linha in linhas_visuais
    ]

    altura_tabela = altura_cabecalho + sum(alturas_linhas)

    linhas_titulo = str(titulo).count("\n") + 1
    linhas_rodape = str(rodape).count("\n") + 1

    altura_topo = 0.45 + 0.25 * (linhas_titulo - 1)
    altura_rodape = max(
        0.60,
        0.22 * linhas_rodape + 0.12
    )

    altura_figura = (
        altura_tabela + altura_topo + altura_rodape
    )

    with plt.rc_context({
        "font.family": "DejaVu Sans",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "text.color": "black"
    }):
        figura = plt.figure(
            figsize=(19, altura_figura),
            facecolor="white"
        )

        eixo = figura.add_axes([
            0.012,
            altura_rodape / altura_figura,
            0.976,
            altura_tabela / altura_figura
        ])
        eixo.axis("off")

        tabela_visual = eixo.table(
            cellText=linhas_visuais,
            colLabels=cabecalhos,
            colWidths=larguras,
            cellLoc="right",
            bbox=[0, 0, 1, 1]
        )

        tabela_visual.auto_set_font_size(False)
        tabela_visual.set_fontsize(11)

        for (linha, coluna), celula in (
            tabela_visual.get_celld().items()
        ):
            celula.set_edgecolor("#DCE3EB")
            celula.set_linewidth(0.55)
            celula.PAD = 0.035

            if linha == 0:
                celula.set_height(
                    altura_cabecalho / altura_tabela
                )
                celula.set_facecolor("#DCE3EB")
                celula.set_text_props(
                    color="black",
                    weight="bold",
                    ha="left" if coluna == 0 else "center",
                    va="center"
                )
                continue

            celula.set_height(
                alturas_linhas[linha - 1] / altura_tabela
            )
            celula.set_facecolor("#FFFFFF")
            celula.set_text_props(
                color="black",
                weight="normal",
                ha="left" if coluna == 0 else "right",
                va="center"
            )

            eh_total = (
                destacar_total
                and str(linhas[linha - 1][0]).strip().upper()
                == "TOTAL"
            )

            if eh_total:
                celula.set_text_props(
                    color="black",
                    weight="bold"
                )

            elif (
                coluna_variacao is not None
                and coluna == coluna_variacao
                and valores_variacao is not None
            ):
                variacao = pd.to_numeric(
                    valores_variacao[linha - 1],
                    errors="coerce"
                )

                texto_variacao = str(
                    linhas[linha - 1][coluna]
                )

                if pd.notna(variacao):
                    if variacao > 0:
                        celula.set_facecolor("#FCE4D6")
                        celula.set_text_props(
                            color="#FB0800",
                            weight="bold"
                        )

                    elif variacao < 0:
                        celula.set_facecolor("#E2EFDA")
                        celula.set_text_props(
                            color="#008504",
                            weight="bold"
                        )

                elif "NOVO" in texto_variacao.upper():
                    celula.set_facecolor("#FCE4D6")
                    celula.set_text_props(
                        color="#FF0800",
                        weight="bold"
                    )

        figura.text(
            0.016,
            1 - 0.08 / altura_figura,
            titulo,
            ha="left",
            va="top",
            fontsize=16,
            fontweight="bold",
            color="black"
        )

        figura.text(
            0.016,
            0.12 / altura_figura,
            rodape,
            ha="left",
            va="bottom",
            fontsize=9,
            color="#64748B"
        )

        figura.savefig(
            caminho_saida,
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.06,
            facecolor="white"
        )

        plt.show()


def gerar_figura_regioes_padronizada(
    resultado,
    territorio,
    semana_final,
    data_referencia,
    ano=2026
):
    dados_visuais = resultado["tabela"].copy()

    linhas_visuais = []

    for linha in dados_visuais.itertuples(
        index=False,
        name=None
    ):
        linhas_visuais.append([
            str(linha[0]),
            formatar_valor_tabela_padronizada(linha[1]),
            formatar_valor_tabela_padronizada(linha[2]),
            formatar_valor_tabela_padronizada(linha[3], 1),
            formatar_valor_tabela_padronizada(linha[4], 2),
            formatar_valor_tabela_padronizada(linha[5]),
            formatar_valor_tabela_padronizada(linha[6], 1),
            formatar_valor_tabela_padronizada(linha[7], 2)
        ])

    desenhar_tabela_padronizada(
        linhas=linhas_visuais,
        cabecalhos=[
            "Região de Saúde",
            "População",
            "Casos\nSE1–26",
            "Incidência\nSE1–26",
            "Positividade (%)\nSE1–26",
            f"Casos\nSE27–{int(semana_final)}",
            f"Incidência\nSE27–{int(semana_final)}",
            f"Positividade (%)\nSE27–{int(semana_final)}"
        ],
        larguras=[
            0.255, 0.115, 0.09, 0.105,
            0.115, 0.09, 0.105, 0.125
        ],
        titulo=(
            "CASOS PROVÁVEIS, INCIDÊNCIA E POSITIVIDADE "
            f"— REGIÕES DE SAÚDE — {territorio} — {ano}"
        ).upper(),
        rodape=(
            "Incidência por 100 mil habitantes. "
            "Positividade = positivos / exames realizados × 100. "
            "Total das regiões apresentadas.\n"
            f"Fonte: SINAN e GAL | Dados até: {data_referencia}."
        ),
        caminho_saida=resultado["arquivo_tabela"]
    )


# ==========================================================
# REGIÕES DE SAÚDE: TABELA, PLANILHA E PAINEL SEMANAL
# ==========================================================

def gerar_analise_regioes_saude(
    base_casos,
    base_populacao,
    arquivo_vinculo,
    semana_final,
    territorio,
    pasta_saida,
    data_referencia,
    uf=None,
    ano=2026,
    base_gal=None
):
    semana_final = int(semana_final)

    if not 27 <= semana_final <= 53:
        raise ValueError(
            "A semana final deve estar entre 27 e 53."
        )

    if base_gal is None:
        raise ValueError(
            "Informe a base GAL no parâmetro base_gal."
        )

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    def codigo_municipal(serie):
        s = (
            serie.astype("string")
            .str.strip()
            .str.replace(r"\.0+$", "", regex=True)
        )

        return s.where(
            s.str.fullmatch(r"\d{6,7}", na=False)
        ).str[:6]

    def numero_populacao(serie):
        if pd.api.types.is_numeric_dtype(serie):
            return pd.to_numeric(
                serie, errors="coerce"
            )

        s = serie.astype("string").str.strip()
        tem_virgula = s.str.contains(",", na=False)

        s.loc[tem_virgula] = (
            s.loc[tem_virgula]
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )

        milhar = (
            ~tem_virgula
            & s.str.fullmatch(
                r"\d{1,3}(?:\.\d{3})+",
                na=False
            )
        )

        s.loc[milhar] = s.loc[milhar].str.replace(
            ".", "", regex=False
        )

        return pd.to_numeric(
            s, errors="coerce"
        )

    def fmt(valor, decimais=0):
        if pd.isna(valor):
            return "—"

        return (
            f"{valor:,.{decimais}f}"
            .replace(",", "@")
            .replace(".", ",")
            .replace("@", ".")
        )

    # ======================================================
    # 1. VÍNCULO MUNICÍPIO–REGIÃO
    # ======================================================

    vinculo_rs = pd.read_csv(
        arquivo_vinculo,
        sep=None,
        engine="python",
        dtype="string",
        encoding="utf-8-sig"
    )

    vinculo_rs = vinculo_rs[[
        "COD_IBGE",
        "UF",
        "COD_REGIAO_SAUDE",
        "REGIAO_SAUDE"
    ]].copy()

    vinculo_rs["COD_IBGE"] = codigo_municipal(
        vinculo_rs["COD_IBGE"]
    )

    for coluna in [
        "UF", "COD_REGIAO_SAUDE", "REGIAO_SAUDE"
    ]:
        vinculo_rs[coluna] = (
            vinculo_rs[coluna].str.strip()
        )

    vinculo_rs["UF"] = (
        vinculo_rs["UF"].str.upper()
    )

    vinculo_rs["COD_REGIAO_SAUDE"] = (
        vinculo_rs["COD_REGIAO_SAUDE"]
        .str.replace(r"\.0+$", "", regex=True)
    )

    if uf is not None:
        vinculo_rs = vinculo_rs.loc[
            vinculo_rs["UF"].eq(
                str(uf).strip().upper()
            )
        ].copy()

    if vinculo_rs.empty:
        raise ValueError(
            f"Não há municípios no vínculo para {territorio}."
        )

    if (
        vinculo_rs.isna().any().any()
        or vinculo_rs.eq("").any().any()
    ):
        raise ValueError(
            "Há campos ausentes ou inválidos "
            "no vínculo regional."
        )

    if vinculo_rs["COD_IBGE"].duplicated().any():
        raise ValueError(
            "Há municípios duplicados no vínculo regional."
        )

    # ======================================================
    # 2. POPULAÇÃO MUNICIPAL
    # ======================================================

    populacao_rs = base_populacao[
        ["COD_IBGE", "POPULACAO"]
    ].copy()

    populacao_rs["COD_IBGE"] = codigo_municipal(
        populacao_rs["COD_IBGE"]
    )

    populacao_rs["POPULACAO"] = numero_populacao(
        populacao_rs["POPULACAO"]
    )

    populacao_rs = (
        populacao_rs
        .dropna(subset=["COD_IBGE"])
        .drop_duplicates()
    )

    if populacao_rs["COD_IBGE"].duplicated().any():
        raise ValueError(
            "Há populações diferentes "
            "para o mesmo município."
        )

    municipios_rs = vinculo_rs.merge(
        populacao_rs,
        on="COD_IBGE",
        how="left",
        validate="one_to_one"
    )

    municipios_rs["POPULACAO"] = (
        municipios_rs["POPULACAO"].where(
            municipios_rs["POPULACAO"].gt(0)
        )
    )

    chaves = [
        "UF",
        "COD_REGIAO_SAUDE",
        "REGIAO_SAUDE"
    ]

    resumo_rs = (
        municipios_rs
        .groupby(chaves, as_index=False)
        .agg(
            POPULACAO=("POPULACAO", "sum"),
            SEM_POPULACAO=(
                "POPULACAO",
                lambda s: int(s.isna().sum())
            )
        )
    )

    resumo_rs.loc[
        resumo_rs["SEM_POPULACAO"].gt(0),
        "POPULACAO"
    ] = np.nan

    # ======================================================
    # 3. CASOS POR RESIDÊNCIA E SEMANA DE SINTOMAS
    # ======================================================

    casos_rs = base_casos[
        ["ID_MN_RESI", "DT_SIN_PRI", "SE"]
    ].copy()

    casos_rs["COD_IBGE"] = codigo_municipal(
        casos_rs["ID_MN_RESI"]
    )

    casos_rs["DT_SIN_PRI"] = pd.to_datetime(
        casos_rs["DT_SIN_PRI"],
        errors="coerce"
    )

    casos_rs["SE"] = pd.to_numeric(
        casos_rs["SE"],
        errors="coerce"
    )

    casos_rs = casos_rs.loc[
        casos_rs["DT_SIN_PRI"].dt.year.eq(ano)
        & casos_rs["SE"].between(1, semana_final)
        & casos_rs["SE"].mod(1).eq(0)
    ].copy()

    casos_rs["SE"] = (
        casos_rs["SE"].astype(int)
    )

    casos_rs = casos_rs.merge(
        vinculo_rs,
        on="COD_IBGE",
        how="left",
        validate="many_to_one",
        indicator=True
    )

    sem_vinculo_rs = (
        casos_rs.loc[
            casos_rs["_merge"].eq("left_only")
        ]
        .groupby("COD_IBGE", dropna=False)
        .size()
        .reset_index(name="Casos sem vínculo")
    )

    casos_rs = casos_rs.loc[
        casos_rs["_merge"].eq("both")
    ].copy()

    if not sem_vinculo_rs.empty:
        print(
            f"{territorio}: "
            f"{int(sem_vinculo_rs['Casos sem vínculo'].sum())} "
            "registros sem vínculo regional "
            "não foram incluídos."
        )

    indice_rs = pd.MultiIndex.from_frame(
        resumo_rs[chaves]
    )

    serie_rs = (
        casos_rs
        .groupby(chaves + ["SE"])
        .size()
        .unstack("SE", fill_value=0)
        .reindex(
            index=indice_rs,
            columns=range(1, semana_final + 1),
            fill_value=0
        )
        .fillna(0)
        .astype(int)
    )

    resumo_rs = resumo_rs.set_index(chaves)

    resumo_rs["CASOS_INICIAL"] = (
        serie_rs.loc[:, 1:26].sum(axis=1)
    )

    resumo_rs["CASOS_RECENTE"] = (
        serie_rs.loc[:, 27:semana_final].sum(axis=1)
    )

    resumo_rs["INC_INICIAL"] = (
        resumo_rs["CASOS_INICIAL"]
        / resumo_rs["POPULACAO"]
        * 100_000
    )

    resumo_rs["INC_RECENTE"] = (
        resumo_rs["CASOS_RECENTE"]
        / resumo_rs["POPULACAO"]
        * 100_000
    )

    # ======================================================
    # 3.1. POSITIVIDADE REGIONAL — GAL
    # ======================================================

    gal_rs = base_gal[[
        "IBGE",
        "SE",
        "exames_realizados",
        "casos_positivos"
    ]].copy()

    gal_rs["COD_IBGE"] = codigo_municipal(
        gal_rs["IBGE"]
    )

    gal_rs["SE"] = pd.to_numeric(
        gal_rs["SE"],
        errors="coerce"
    )

    gal_rs = gal_rs.loc[
        gal_rs["SE"].between(1, semana_final)
        & gal_rs["SE"].mod(1).eq(0)
    ].copy()

    gal_rs["SE"] = gal_rs["SE"].astype(int)

    for coluna in [
        "exames_realizados",
        "casos_positivos"
    ]:
        gal_rs[coluna] = pd.to_numeric(
            gal_rs[coluna],
            errors="coerce"
        ).fillna(0)

    gal_rs = gal_rs.merge(
        vinculo_rs,
        on="COD_IBGE",
        how="inner",
        validate="many_to_one"
    )

    gal_regional_semanal = (
        gal_rs.groupby(chaves + ["SE"])[
            ["exames_realizados", "casos_positivos"]
        ]
        .sum()
    )

    if (
        gal_regional_semanal["casos_positivos"]
        > gal_regional_semanal["exames_realizados"]
    ).any():
        raise ValueError(
            "Há regiões/semanas com mais positivos "
            "do que exames."
        )

    if gal_regional_semanal.lt(0).any().any():
        raise ValueError(
            "Há quantidades negativas de exames no GAL."
        )

    def positividade_periodo_rs(inicio, fim):
        totais = (
            gal_rs.loc[
                gal_rs["SE"].between(inicio, fim)
            ]
            .groupby(chaves)[
                ["exames_realizados", "casos_positivos"]
            ]
            .sum()
            .reindex(
                resumo_rs.index,
                fill_value=0
            )
        )

        return (
            totais["casos_positivos"]
            / totais["exames_realizados"].where(
                totais["exames_realizados"].gt(0)
            )
            * 100
        )

    resumo_rs["POSITIVIDADE_INICIAL"] = (
        positividade_periodo_rs(1, 26)
    )

    resumo_rs["POSITIVIDADE_RECENTE"] = (
        positividade_periodo_rs(27, semana_final)
    )

    indice_gal_rs = pd.MultiIndex.from_tuples(
        [
            tuple(regiao) + (semana,)
            for regiao in resumo_rs.index
            for semana in range(1, semana_final + 1)
        ],
        names=chaves + ["SE"]
    )

    gal_grade_rs = (
        gal_regional_semanal.reindex(indice_gal_rs)
    )

    positividade_semanal_rs = (
        gal_grade_rs["casos_positivos"]
        / gal_grade_rs["exames_realizados"].where(
            gal_grade_rs["exames_realizados"].gt(0)
        )
        * 100
    ).unstack("SE")

    # ======================================================
    # 4. DEZ MAIORES TOTAIS NO SEGUNDO PERÍODO
    # ======================================================

    selecionadas_rs = (
        resumo_rs.reset_index()
        .loc[lambda d: d["CASOS_RECENTE"].gt(0)]
        .sort_values(
            [
                "CASOS_RECENTE",
                "INC_RECENTE",
                "UF",
                "REGIAO_SAUDE"
            ],
            ascending=[False, False, True, True],
            na_position="last"
        )
        .head(10)
        .copy()
    )

    if selecionadas_rs.empty:
        raise ValueError(
            f"{territorio}: não há casos "
            f"nas SE27–{semana_final}."
        )

    selecionadas_rs["Região de saúde"] = (
        selecionadas_rs["REGIAO_SAUDE"]
        if uf is not None
        else (
            selecionadas_rs["REGIAO_SAUDE"]
            + " ("
            + selecionadas_rs["UF"]
            + ")"
        )
    )

    colunas_dados = [
        "Região de saúde",
        "POPULACAO",
        "CASOS_INICIAL",
        "INC_INICIAL",
        "POSITIVIDADE_INICIAL",
        "CASOS_RECENTE",
        "INC_RECENTE",
        "POSITIVIDADE_RECENTE"
    ]

    tabela_rs = selecionadas_rs[
        colunas_dados
    ].copy()

    populacao_total = (
        selecionadas_rs["POPULACAO"].sum(
            min_count=len(selecionadas_rs)
        )
    )

    inicial_total = (
        selecionadas_rs["CASOS_INICIAL"].sum()
    )

    recente_total = (
        selecionadas_rs["CASOS_RECENTE"].sum()
    )

    indice_selecionadas = (
        pd.MultiIndex.from_frame(
            selecionadas_rs[chaves]
        )
    )

    gal_selecionadas_rs = gal_rs.loc[
        pd.MultiIndex.from_frame(
            gal_rs[chaves]
        ).isin(indice_selecionadas)
    ].copy()

    def positividade_total_rs(inicio, fim):
        dados = gal_selecionadas_rs.loc[
            gal_selecionadas_rs["SE"].between(
                inicio, fim
            )
        ]

        exames = dados["exames_realizados"].sum()
        positivos = dados["casos_positivos"].sum()

        return (
            positivos / exames * 100
            if exames > 0
            else np.nan
        )

    tabela_rs.loc[len(tabela_rs)] = [
        "Total",
        populacao_total,
        inicial_total,
        inicial_total / populacao_total * 100_000,
        positividade_total_rs(1, 26),
        recente_total,
        recente_total / populacao_total * 100_000,
        positividade_total_rs(27, semana_final)
    ]

    cabecalhos_excel = [
        "Região de saúde",
        "População",
        "CASOS_SE1_SE26",
        "INC_SE1_SE26",
        "Positividade (%) SE1–26",
        f"CASOS_SE27_SE{semana_final}",
        f"INC_SE27_SE{semana_final}",
        f"Positividade (%) SE27–{semana_final}"
    ]

    tabela_excel_rs = tabela_rs.copy()
    tabela_excel_rs.columns = cabecalhos_excel

    prefixo = "chik_brasil_regioes_saude"

    caminho_excel = (
        pasta_saida / f"{prefixo}.xlsx"
    )
    caminho_tabela = (
        pasta_saida / f"{prefixo}_tabela.png"
    )
    caminho_painel = (
        pasta_saida / f"{prefixo}_painel.png"
    )

    serie_selecionadas = serie_rs.reindex(
        indice_selecionadas
    )

    # ======================================================
    # 5. PLANILHA XLSX
    # ======================================================

    todas_regioes_excel = (
        resumo_rs.reset_index()
        .drop(columns=["SEM_POPULACAO"])
        .rename(columns={
            "CASOS_INICIAL": "CASOS_SE1_SE26",
            "INC_INICIAL": "INC_SE1_SE26",
            "POSITIVIDADE_INICIAL": (
                "Positividade (%) SE1–26"
            ),
            "CASOS_RECENTE": (
                f"CASOS_SE27_SE{semana_final}"
            ),
            "INC_RECENTE": (
                f"INC_SE27_SE{semana_final}"
            ),
            "POSITIVIDADE_RECENTE": (
                f"Positividade (%) SE27–{semana_final}"
            )
        })
        [[
            "UF",
            "COD_REGIAO_SAUDE",
            "REGIAO_SAUDE",
            "POPULACAO",
            "CASOS_SE1_SE26",
            "INC_SE1_SE26",
            "Positividade (%) SE1–26",
            f"CASOS_SE27_SE{semana_final}",
            f"INC_SE27_SE{semana_final}",
            f"Positividade (%) SE27–{semana_final}"
        ]]
    )

    with pd.ExcelWriter(
        caminho_excel,
        engine="openpyxl"
    ) as writer:

        tabela_excel_rs.to_excel(
            writer,
            sheet_name="Tabela",
            index=False
        )

        todas_regioes_excel.to_excel(
            writer,
            sheet_name="Todas_as_regioes",
            index=False
        )

        serie_selecionadas.reset_index().to_excel(
            writer,
            sheet_name="Casos_semanais",
            index=False
        )

        for nome_aba in [
            "Tabela",
            "Todas_as_regioes"
        ]:
            ws = writer.sheets[nome_aba]
            ws.freeze_panes = "B2"

            for celula in ws[1]:
                celula.font = Font(bold=True)
                celula.alignment = Alignment(
                    horizontal="center",
                    wrap_text=True
                )

                ws.column_dimensions[
                    celula.column_letter
                ].width = 24

            if nome_aba == "Tabela":
                ws.column_dimensions["A"].width = 45
            else:
                ws.column_dimensions["A"].width = 10
                ws.column_dimensions["C"].width = 45

            ws.row_dimensions[1].height = 42

            for cabecalho in ws[1]:
                nome_coluna = str(
                    cabecalho.value
                )

                if (
                    nome_coluna.startswith("INC_")
                    or nome_coluna.startswith(
                        "Positividade"
                    )
                ):
                    formato = "0.00"

                elif (
                    nome_coluna.startswith("CASOS_")
                    or nome_coluna in [
                        "POPULACAO",
                        "População"
                    ]
                ):
                    formato = "#,##0"

                else:
                    continue

                for linha in ws.iter_rows(
                    min_row=2,
                    min_col=cabecalho.column,
                    max_col=cabecalho.column
                ):
                    linha[0].number_format = formato

    # ======================================================
    # 6. TABELA PNG — LAYOUT APROVADO
    # ======================================================

    gerar_figura_regioes_padronizada(
        resultado={
            "tabela": tabela_excel_rs,
            "arquivo_tabela": caminho_tabela
        },
        territorio=territorio,
        semana_final=semana_final,
        data_referencia=data_referencia,
        ano=ano
    )

    # ======================================================
    # 7. PAINEL: CASOS E POSITIVIDADE SEMANAL
    # ======================================================

    regioes_painel_rs = selecionadas_rs.head(4)

    with plt.rc_context({
        "font.family": "Arial"
    }):
        fig_painel, eixos_rs = plt.subplots(
            2, 2,
            figsize=(18, 10),
            facecolor="white"
        )

        semanas_rs = np.arange(
            1, semana_final + 1
        )

        cores_rs = np.where(
            semanas_rs <= 26,
            "#fdd49e",
            "#c32f27"
        )

        for ax_rs, (_, regiao_rs) in zip(
            eixos_rs.flat,
            regioes_painel_rs.iterrows()
        ):
            chave = tuple(
                regiao_rs[c]
                for c in chaves
            )

            valores_rs = (
                serie_rs.loc[chave].to_numpy()
            )

            ax_rs.bar(
                semanas_rs,
                valores_rs,
                color=cores_rs,
                width=0.82,
                zorder=3
            )

            ax_rs.axvline(
                26.5,
                color="#8C939C",
                linestyle="--",
                linewidth=1.2
            )

            ax_positividade_rs = (
                ax_rs.twinx()
            )

            valores_positividade_rs = (
                positividade_semanal_rs
                .loc[chave]
                .reindex(semanas_rs)
            )

            ax_positividade_rs.plot(
                semanas_rs,
                valores_positividade_rs.to_numpy(),
                color="#1F4E78",
                linewidth=1.8,
                marker="o",
                markersize=3,
                zorder=5
            )

            ax_positividade_rs.set_ylim(
                0, 105
            )

            ax_positividade_rs.set_yticks(
                [0, 25, 50, 75, 100]
            )

            ax_positividade_rs.set_ylabel(
                "Positividade (%)",
                fontsize=10,
                color="#000000"
            )

            ax_positividade_rs.tick_params(
                axis="y",
                labelsize=9,
                colors="#000000"
            )

            ax_positividade_rs.grid(False)

            ax_rs.set_title(
                textwrap.fill(
                    regiao_rs["Região de saúde"],
                    width=40
                ),
                loc="left",
                fontsize=14,
                fontweight="bold",
                pad=32
            )

            ax_rs.text(
                0, 1.06,
                f"SE1–26: {fmt(regiao_rs['CASOS_INICIAL'])}"
                f"   |   SE27–{semana_final}: "
                f"{fmt(regiao_rs['CASOS_RECENTE'])}",
                transform=ax_rs.transAxes,
                fontsize=11,
                color="#4B5563"
            )

            ax_rs.set_xlabel(
                "Semana Epidemiológica",
                fontsize=11
            )

            ax_rs.set_ylabel(
                "Casos prováveis",
                fontsize=11
            )

            ax_rs.set_xlim(
                0.3, semana_final + 0.7
            )

            ax_rs.set_ylim(
                0,
                max(
                    1,
                    valores_rs.max() * 1.15
                )
            )

            ax_rs.set_xticks(sorted(set(
                [1, 25, semana_final]
                + list(
                    range(
                        5,
                        semana_final + 1,
                        5
                    )
                )
            )))

            ax_rs.yaxis.set_major_locator(
                MaxNLocator(
                    integer=True,
                    nbins=5
                )
            )

            ax_rs.grid(
                axis="y",
                linewidth=0.5,
                linestyle="--",
                color="#DCE1E7",
                zorder=0
            )

            ax_rs.grid(
                axis="x",
                linewidth=0.5,
                linestyle="--",
                color="#DCE1E7",
                zorder=0
            )

            ax_rs.spines[
                ["top", "right"]
            ].set_visible(True)

        for ax_rs in list(eixos_rs.flat)[
            len(regioes_painel_rs):
        ]:
            ax_rs.set_visible(False)

        fig_painel.tight_layout(
            h_pad=4,
            w_pad=3
        )

        fig_painel.savefig(
            caminho_painel,
            dpi=300,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.show()

    return {
        "tabela": tabela_excel_rs,
        "selecionadas": selecionadas_rs,
        "todas_regioes": todas_regioes_excel,
        "serie_semanal": serie_selecionadas,
        "positividade_semanal": (
            positividade_semanal_rs
        ),
        "sem_vinculo": sem_vinculo_rs,
        "arquivo_tabela": caminho_tabela,
        "arquivo_excel": caminho_excel,
        "arquivo_painel": caminho_painel
    }


# ==========================================================
# BRASIL — EXECUTAR A ANÁLISE
# ==========================================================

resultado_regioes_brasil = gerar_analise_regioes_saude(
    base_casos=df_provaveis,
    base_populacao=df_municipios,
    arquivo_vinculo=(
        base_ibge / "vinculo_municipio_regiao.csv"
    ),
    semana_final=se_analise,
    territorio="Brasil",
    pasta_saida=(
        Path.cwd()
        / f"analises_chikungunya_brasil_SE{se_analise}"
    ),
    data_referencia=data_analise,
    base_gal=df_gal_final
)

In [ ]:
from PIL import Image

# Remove o limite de pixels para evitar o erro de decompression bomb
Image.MAX_IMAGE_PIXELS = None

# VIGILÂNCIA EPIDEMIOLÓGICA E LABORATORIAL POR UF
# Últimas oito semanas epidemiológicas

ufs = list(mapa_uf.values())

def contar_sinan_por_uf(base):
    return (
        pd.to_numeric(base["SG_UF"], errors="coerce")
        .map(mapa_uf)
        .value_counts()
        .reindex(ufs, fill_value=0)
    )

notificados_temporada = df_notificados[df_notificados['SE'] >= 27]

notificados = contar_sinan_por_uf(notificados_temporada)
provaveis = contar_sinan_por_uf(temporada_2026)

gal = gal_temporada.copy()
gal["UF_SIGLA"] = pd.to_numeric(gal["UF"], errors="coerce").map(mapa_uf)

for coluna in ["exames_realizados", "casos_positivos"]:
    gal[coluna] = pd.to_numeric(gal[coluna], errors="coerce").fillna(0)

totais_gal = (
    gal.groupby("UF_SIGLA")[["exames_realizados", "casos_positivos"]]
    .sum()
    .reindex(ufs, fill_value=0)
)

dados_uf = pd.DataFrame({
    "UF": ufs,
    "Notificados": notificados.to_numpy(),
    "Prováveis": provaveis.to_numpy(),
    "Exames": totais_gal["exames_realizados"].to_numpy(),
    "Positivos": totais_gal["casos_positivos"].to_numpy(),
})

# Ordena as 27 UFs pelo número de casos prováveis
dados_uf = dados_uf.sort_values(
    ["Prováveis", "Notificados", "UF"], ascending=[False, False, True]
).reset_index(drop=True)

# 1. Inverte a ordem das linhas para que o maior valor apareça no TOPO do gráfico horizontal
dados_uf = dados_uf.iloc[::-1].reset_index(drop=True)

if (dados_uf["Prováveis"] > dados_uf["Notificados"]).any():
    raise ValueError("Há UFs com mais casos prováveis do que notificações.")

if (dados_uf["Positivos"] > dados_uf["Exames"]).any():
    raise ValueError(
        "Há UFs com mais exames positivos do que exames realizados."
    )

# 2. Aproximando os painéis reduzindo o wspace de 0.24 para 0.08
fig, (ax_sinan, ax_gal) = plt.subplots(
    1, 2, figsize=(35, 20), sharey=True, gridspec_kw={"wspace": 0.02}
)

posicoes = np.arange(len(dados_uf))
altura = 0.72

# Painel SINAN: prováveis + demais notificações = total notificado.
ax_sinan.barh(
    posicoes,
    dados_uf["Prováveis"],
    height=altura,
    color="#1b263b",
    label="Prováveis",
)
ax_sinan.barh(
    posicoes,
    dados_uf["Notificados"] - dados_uf["Prováveis"],
    left=dados_uf["Prováveis"],
    height=altura,
    color="#a9b7c8",
    label="Notificados",
)

# Painel GAL: positivos + demais exames = total de exames.
ax_gal.barh(
    posicoes,
    dados_uf["Positivos"],
    height=altura,
    color="#c32f27",
    label="Confirmados",
)
ax_gal.barh(
    posicoes,
    dados_uf["Exames"] - dados_uf["Positivos"],
    left=dados_uf["Positivos"],
    height=altura,
    color="#f4a261",
    label="Exames Realizados",
)

# Configurações de rótulos e eixos para cada painel
paineis_config = [
    {
        "ax": ax_sinan,
        "col_total": "Notificados",
        "col_destaque": "Prováveis",
        "lbl_total": "Notificados",
        "lbl_destaque": "Prováveis",
    },
    {
        "ax": ax_gal,
        "col_total": "Exames",
        "col_destaque": "Positivos",
        "lbl_total": "Testes",
        "lbl_destaque": "Confirmados",
    },
]

for cfg in paineis_config:
    ax = cfg['ax']
    coluna_total = cfg['col_total']
    coluna_destaque = cfg['col_destaque']

    maximo = max(dados_uf[coluna_total].max(), 1)
    ax.set_xlim(0, maximo * 1.55)
    ax.set_xticks([])

    # 3. Linhas horizontais/verticais ultra finas e transparentes
    # 'set_axisbelow(True)' garante que fiquem ATRÁS das barras
    ax.set_axisbelow(True)
    ax.grid(
        axis='x', 
        color='#e0e0e0', 
        linestyle='--', 
        linewidth=0.3,  # Espessura bem fina
        alpha=0.6
    )

    # 4. Formatação dos rótulos com proteção de fundo (evita que qualquer linha atravesse o texto)
    for indice, linha in dados_uf.iterrows():
        total = int(linha[coluna_total])
        destaque = int(linha[coluna_destaque])

        percentual = f'{100 * destaque / total:.0f}%' if total > 0 else '—'

        str_total = f'{total:,}'.replace(',', '.')
        str_destaque = f'{destaque:,}'.replace(',', '.')

        rotulo = f"{cfg['lbl_total']}: {str_total} | {cfg['lbl_destaque']}: {str_destaque} ({percentual})"

        # O parâmetro bbox adiciona uma caixa branca ao fundo do texto, ocultando qualquer linha de grade por trás
        ax.text(
            total + maximo * 0.012,
            indice,
            rotulo,
            va='center',
            fontsize=20,
            color='#000000',
            bbox=dict(boxstyle='square,pad=0.1', facecolor='white', edgecolor='none', alpha=1.0)
        )

    for borda in ax.spines.values():
        borda.set_visible(False)

    ax.legend(
        loc='lower left',
        fontsize=20,
        ncols= 2,
        frameon=False
    )

ax_sinan.set_yticks(posicoes, dados_uf["UF"])
ax_sinan.tick_params(axis="y", length=0, labelsize=25, grid_linestyle='--')
ax_gal.tick_params(axis="y", left=False, labelleft=False, grid_linestyle='--')

ax_sinan.set_title(
    "Notificações e casos prováveis".upper(), fontsize=25, fontweight="bold", pad= -30,
)
ax_gal.set_title(
    "Exames realizados e confirmados".upper(), fontsize=25, fontweight="bold", pad= -30,
)

fig.suptitle(
    f"Chikungunya por UF - SE 27 a SE{se_temporada_atual} - Brasil - 2026".upper(),
    fontsize=30,
    fontweight="bold",
)

fig.subplots_adjust(left=0.06, right=0.98, top=0.91, bottom=0.08)

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_painel_sinan_gal_uf.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# BASE DE POPULAÇÃO E MAPEAMENTO DE REGIÕES (IBGE)
# População aproximada por UF (Essencial para calcular o Coeficiente de Incidência)
dados_uf_pop = {
    'Estado': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO', 
              'MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA',
              'MG', 'ES', 'RJ', 'SP', 
              'PR', 'SC', 'RS',
              'MS', 'MT', 'GO', 'DF'],
    'Nome_UF': ['Rondônia', 'Acre', 'Amazonas', 'Roraima', 'Pará', 'Amapá', 'Tocantins',
                'Maranhão', 'Piauí', 'Ceará', 'Rio Grande do Norte', 'Paraíba', 'Pernambuco', 
                'Alagoas', 'Sergipe', 'Bahia', 'Minas Gerais', 'Espírito Santo', 'Rio de Janeiro', 
                'São Paulo', 'Paraná', 'Santa Catarina', 'Rio Grande do Sul', 'Mato Grosso do Sul', 
                'Mato Grosso', 'Goiás', 'Distrito Federal'],
    'Regiao': ['Norte']*7 + ['Nordeste']*9 + ['Sudeste']*4 + ['Sul']*3 + ['Centro-Oeste']*4,
    'Populacao': [1581196, 830018, 3941613, 636707, 8120131, 733759, 1511460, # Norte
                 6776805, 3270174, 8794957, 3303953, 3974687, 9058931, 3125254, 2210003, 14141626, # Nordeste
                 20539989, 3833712, 16055174, 44411238, # Sudeste
                 11444380, 7610361, 10882965, # Sul
                 2757013, 3658649, 7056495, 2817381]} # Centro-Oeste

df_pop = pd.DataFrame(dados_uf_pop)

# PROCESSAMENTO E CRUZAMENTO DOS DADOS (COM LIMPEZA FORÇADA)

# Dicionário interno (com os códigos já em texto) para traduzir o IBGE do SINAN.
mapa_ibge_sigla = {
    '11': 'RO', '12': 'AC', '13': 'AM', '14': 'RR', '15': 'PA', '16': 'AP', '17': 'TO',
    '21': 'MA', '22': 'PI', '23': 'CE', '24': 'RN', '25': 'PB', '26': 'PE', '27': 'AL', '28': 'SE', '29': 'BA',
    '31': 'MG', '32': 'ES', '33': 'RJ', '35': 'SP',
    '41': 'PR', '42': 'SC', '43': 'RS',
    '50': 'MS', '51': 'MT', '52': 'GO', '53': 'DF'
}

# 1. Processar 2026 usando SG_UF
df_chik_26_base = df_provaveis.copy()
df_filtrado = df_chik_26_base[df_chik_26_base['SE'] >= 27]
df_chik_26 = df_filtrado['SG_UF'].value_counts().reset_index()
df_chik_26.columns = ['Estado', 'Provaveis_2026']


# LIMPEZA: Transforma em texto, tira o ".0" (se for número convertido) e tira espaços invisíveis
df_chik_26['Estado'] = (df_chik_26['Estado']
                        .astype(str)
                        .str.replace('.0', '', regex=False)
                        .str.strip())
# Traduz para a sigla (se a base já estiver com as siglas, essa linha não altera nada, é seguro)
df_chik_26['Estado'] = df_chik_26['Estado'].replace(mapa_ibge_sigla)

# 2. Processar 2025 lendo a base bruta
df_base_2025 = df_provaveis_2025.copy()
df_base_2025_filtrada = df_base_2025[df_base_2025['SE'].between(27, se_atual, inclusive='both')]

df_chik_25 = df_base_2025_filtrada['SG_UF'].value_counts().reset_index()
df_chik_25.columns = ['Estado', 'Provaveis_2025']

# LIMPEZA idêntica para 2025
df_chik_25['Estado'] = (df_chik_25['Estado']
                        .astype(str)
                        .str.replace('.0', '', regex=False)
                        .str.strip())
df_chik_25['Estado'] = df_chik_25['Estado'].replace(mapa_ibge_sigla)

# 3. Juntar os dados de 2025 e 2026 em uma tabela temporária
df_casos_anos = pd.merge(df_chik_25, df_chik_26, on='Estado', how='outer').fillna(0)

# (Opcional - Debug) - Tire o '#' da linha abaixo se quiser ver no console os estados que ele encontrou
# print("Estados identificados no SINAN:", df_casos_anos['Estado'].unique())

# 4. Cruzar as contagens com a SUA tabela de população (que continua intacta)
df_consolidado = pd.merge(df_pop, df_casos_anos, on='Estado', how='left').fillna(0)

# Calcular Incidência por 100 mil habitantes
df_consolidado['Inc_2026'] = (df_consolidado['Provaveis_2026'] / df_consolidado['Populacao']) * 100000
df_consolidado['Inc_2025'] = (df_consolidado['Provaveis_2025'] / df_consolidado['Populacao']) * 100000

# MONTAGEM DA HIERARQUIA (SUBTOTAIS POR REGIÃO)

linhas_tabela = []

# Agrupar por Região respeitando a ordem geográfica
ordem_regioes = ['Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste']

for reg in ordem_regioes:
    df_reg = df_consolidado[df_consolidado['Regiao'] == reg]
    if df_reg.empty:
        continue
        
    # Subtotal da Região
    pop_reg = df_reg['Populacao'].sum()
    c26_reg = df_reg['Provaveis_2026'].sum()
    c25_reg = df_reg['Provaveis_2025'].sum()
    inc26_reg = (c26_reg / pop_reg) * 100000 if pop_reg > 0 else 0
    inc25_reg = (c25_reg / pop_reg) * 100000 if pop_reg > 0 else 0
    var_reg = ((c26_reg - c25_reg) / c25_reg * 100) if c25_reg > 0 else (100.0 if c26_reg > 0 else 0.0)
    
    # Adiciona a Linha da Região (Subtotal)
    linhas_tabela.append({
        'Local': reg, 'is_regiao': True,
        'Provaveis_2026': c26_reg, 'Inc_2026': inc26_reg,
        'Provaveis_2025': c25_reg, 'Inc_2025': inc25_reg,
        'Var': var_reg
    })
    
    # Adiciona as Linhas de cada Estado da Região
    for _, row in df_reg.iterrows():
        c26 = row['Provaveis_2026']
        c25 = row['Provaveis_2025']
        var_uf = ((c26 - c25) / c25 * 100) if c25 > 0 else (100.0 if c26 > 0 else 0.0)
        
        linhas_tabela.append({
            'Local': f"  {row['Nome_UF']}", 'is_regiao': False,
            'Provaveis_2026': c26, 'Inc_2026': row['Inc_2026'],
            'Provaveis_2025': c25, 'Inc_2025': row['Inc_2025'],
            'Var': var_uf
        })

df_report = pd.DataFrame(linhas_tabela)

# FORMATAÇÃO TEXTUAL E MONTAGEM DA MATRIZ DA TABELA

# Formatadores
fmt_int = lambda x: f"{int(x):,}".replace(',', '.')
fmt_dec = lambda x: f"{x:,.1f}".replace(',', 'X').replace('.', ',').replace('X', '.')

# Função para formatar decimais com as setas ▲ e ▼
def fmt_dec_setas(val):
    if pd.isna(val) or val == 0:
        return "0,0"
    
    # Formata com uma casa decimal no padrão BR
    valor_abs_str = f"{abs(val):,.0f}".replace(',', 'X').replace('.', ',').replace('X', '.')
    
    if val > 0:
        return f"▲ {valor_abs_str}"
    else:
        return f"▼ {valor_abs_str}"

# Construir células de texto
celulas_texto = []
# Linhas de Cabeçalho Superior e Inferior
celulas_texto.append(['REGIÃO/UF', 'Casos 2026', 'INCIDÊNCIA', 'Casos 2025', 'INCIDÊNCIA', '% de variação'])

for _, row in df_report.iterrows():
    celulas_texto.append([
        row['Local'],
        fmt_int(row['Provaveis_2026']),
        fmt_dec(row['Inc_2026']),  
        fmt_int(row['Provaveis_2025']),
        fmt_dec(row['Inc_2025']),  
        fmt_dec_setas(row['Var'])  
    ])

# RENDERIZAÇÃO VISUAL COM MATPLOTLIB

fig, ax = plt.subplots(figsize=(11, len(celulas_texto) * 0.38 + 1))
ax.axis('tight')
ax.axis('off')

tabla = ax.table(
    cellText=celulas_texto,
    cellLoc='center',
    loc='center'
)

tabla.auto_set_font_size(False)
tabla.set_fontsize(11)
tabla.scale(1.2, 1.5)

# --- MESCLAGEM DE CÉLULAS DO CABEÇALHO ---
for col in range(1, 6):
    tabla[(0, col)].set_facecolor('#d9d9d9')

tabla[(0, 1)].get_text().set_text('PROVAVEIS 2026')
tabla[(0, 3)].get_text().set_text('PROVAVEIS 2025')
tabla[(0, 5)].get_text().set_text('VARIAÇÃO (%)')

# Estilização Célula por Célula
n_cabecalho_offset = 1  

for (row, col), cell in tabla.get_celld().items():
    cell.set_linewidth(0.6)
    cell.set_edgecolor('#a0a0a0')
    
    # Estilo dos Cabeçalhos
    if row < n_cabecalho_offset:
        cell.set_facecolor('#cccccc')
        cell.set_text_props(weight='bold', color='black')
        if col == 0:
            cell.set_facecolor('#d9d9d9')
    else:
        idx_data = row - n_cabecalho_offset
        info_linha = df_report.iloc[idx_data]
        
        # Alinhar nomes das UFs/Regiões à esquerda
        if col == 0:
            cell.set_text_props(ha='left')
        
        # Destaque de LINHA DE REGIÃO (SUBTOTAL)
        if info_linha['is_regiao']:
            cell.set_facecolor('#d9d9d9')
            cell.set_text_props(weight='bold')
        else:
            cell.set_facecolor('#ffffff')
            
        # Destaque Condicional: Variação positiva (Aumento de casos) em VERMELHO
        if col == 5:
            val_var = info_linha['Var']
            if val_var > 0:
                cell.set_facecolor('#fce4d6')  
                cell.set_text_props(color='#c00000', weight='bold') 

# RODAPÉ
texto_rodape = (
    "Fonte: SINAN - Sistema de Informação de Agravos de Notificação\n"
    "Nota1: Incidência calculada por 100.000 habitantes com base na população estimada (IBGE)\n"
    f"Nota2: Os dados da tabela são comparativos entre a SE 27 e SE {se_temporada_atual} de 2025 e 2026"
)

plt.figtext(
    0.05,        
    0.18,       
    texto_rodape,
    ha='left',   
    va='top',    
    fontsize=9,
    style='italic',
    color="#000000")

plt.title(
    'Casos Prováveis de Chikungunya e Incidência por Região e UF - Brasil - 2025 e 2026'.upper(), 
    fontsize=15, weight='bold', pad= -60
)

# Salvar tabela
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_tabela_comparativa_uf.png')
plt.savefig(salvar_pasta , bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# MUNICÍPIOS COM MAIS CASOS PROVÁVEIS NAS ÚLTIMAS OITO SEMANAS: SINAN E GAL

# Os 15 municípios são escolhidos somente pelo número de casos prováveis.
# O GAL é consultado depois para esses mesmos municípios e semanas.
def codigo_municipio_6(serie):
    return (serie.astype('string').str.strip()
            .str.replace(r'\.0$', '', regex=True).str[:6])

municipios_referencia = df_municipios[['COD_IBGE', 'MUNICÍPIO - IBGE - UF']].copy()
municipios_referencia['codigo_municipio'] = codigo_municipio_6(municipios_referencia['COD_IBGE'])
if municipios_referencia['codigo_municipio'].duplicated().any():
    raise ValueError('Há códigos municipais duplicados na tabela de referência.')

sinan_oito = temporada_2026[['ID_MN_RESI']].copy()
sinan_oito['codigo_municipio'] = codigo_municipio_6(sinan_oito['ID_MN_RESI'])
sinan_oito = sinan_oito[sinan_oito['codigo_municipio'].str.fullmatch(r'\d{6}', na=False)]

ranking = (sinan_oito.groupby('codigo_municipio').size()
           .rename('Casos prováveis').reset_index()
           .sort_values(['Casos prováveis', 'codigo_municipio'], ascending=[False, True])
           .head(15))
if ranking.empty:
    raise ValueError('Não há casos prováveis no recorte das últimas oito semanas.')

gal_oito = gal_temporada[['IBGE', 'casos_positivos']].copy()
gal_oito['codigo_municipio'] = codigo_municipio_6(gal_oito['IBGE'])
gal_oito['casos_positivos'] = pd.to_numeric(gal_oito['casos_positivos'], errors='coerce')
gal_oito = gal_oito[gal_oito['codigo_municipio'].str.fullmatch(r'\d{6}', na=False)]
confirmados = (gal_oito.groupby('codigo_municipio')['casos_positivos']
               .sum().rename('Positivos GAL').reset_index())

top_municipios = (ranking.merge(confirmados, on='codigo_municipio', how='left', validate='one_to_one')
                  .merge(municipios_referencia[['codigo_municipio', 'MUNICÍPIO - IBGE - UF']],
                         on='codigo_municipio', how='left', validate='one_to_one'))
top_municipios['Positivos GAL'] = top_municipios['Positivos GAL'].fillna(0)
top_municipios['Município'] = top_municipios['MUNICÍPIO - IBGE - UF'].fillna(top_municipios['codigo_municipio'])

# Barras justapostas; a posição dos municípios segue o ranking de casos prováveis.
grafico_municipio = plt.figure(figsize=(20, 10))
gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1], figure=grafico_municipio)
ax = grafico_municipio.add_subplot(gs[0])
posicoes = np.arange(len(top_municipios))
largura = 0.4
barras_sinan = ax.bar(posicoes - largura / 2, top_municipios['Casos prováveis'],
                      width=largura, color='#1b263b', label='Casos prováveis (SINAN)')
barras_gal = ax.bar(posicoes + largura / 2, top_municipios['Positivos GAL'],
                    width=largura, color='#778da9', label='Casos confirmados (GAL)')
for barras in (barras_sinan, barras_gal):
    ax.bar_label(barras, labels=[f'{valor:.0f}' for valor in barras.datavalues],
                 padding=3, fontsize=9)

ax.set_title(f'Casos prováveis e confirmados para chikungunya nos municípios com mais casos prováveis '
             f'- SE 27 a SE {se_temporada_atual} - Brasil - 2026'.upper(),
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(posicoes, top_municipios['Município'], rotation=20, ha='right')
ax.set_xlabel('Município-UF', fontsize=14, labelpad=12)
ax.set_ylabel('Total de casos', fontsize=14)
ax.set_ylim(0, max(top_municipios[['Casos prováveis', 'Positivos GAL']].max()) * 1.2)
ax.set_xlim(-0.6, len(top_municipios) - 0.4)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.grid(axis='x', linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
ax.legend(loc='upper right', frameon=True)

ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fontes: SINAN e GAL | Dados até: {data_analise}')
ax_rodape.text(0, 0.3, rodape_texto, ha='left', va='center', fontsize=9)
grafico_municipio.subplots_adjust(hspace=0.3, bottom=0.12)

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_municipios_sinan_gal_oito_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS PROVÁVEIS EM OITO SEMANAS (COM SEGMENTAÇÃO DE QUATRO SEMANAS)

# 1. TRATAMENTO E UNIÃO DAS BASES (Com a limpeza aplicada!)
df_municipios['COD_IBGE_6'] = (df_municipios['COD_IBGE']
                               .astype(str)
                               .str.replace('.0', '', regex=False)
                               .str.strip()
                               .str[:6])

temporada_2026['ID_MN_RESI'] = (temporada_2026['ID_MN_RESI']
                                       .astype(str)
                                       .str.replace('.0', '', regex=False)
                                       .str.strip()
                                       .str[:6])

df_completo = pd.merge(
    temporada_2026, 
    df_municipios, 
    left_on='ID_MN_RESI', 
    right_on='COD_IBGE_6', 
    how='left'
)

# 2. DEFINIÇÃO DOS DOIS PERÍODOS (PRIMEIRAS 4 X ÚLTIMAS 4 SEMANAS)
se_corte = semanas_analise + 3

# Rótulos para a legenda do gráfico
rotulo_p1 = f'SE{semanas_analise} a SE{se_corte}'
rotulo_p2 = f'SE{se_corte + 1} a SE{se_temporada_atual}'

df_completo['Período'] = np.where(
    df_completo['SE'] <= se_corte, 
    rotulo_p1, 
    rotulo_p2
)

top_10_municipios = df_completo['MUNICÍPIO - IBGE - UF'].value_counts().head(15).index

# Filtrar o DataFrame apenas com esses 10 municípios
df_top = df_completo[df_completo['MUNICÍPIO - IBGE - UF'].isin(top_10_municipios)]

# Agrupar contagem de casos por Município e Período
df_agrupado = df_top.groupby(['MUNICÍPIO - IBGE - UF', 'Período']).size().reset_index(name='Casos')

# Forçar a ordem correta do Período (Primeiro Período -> Segundo Período)
df_agrupado['Período'] = pd.Categorical(
    df_agrupado['Período'], 
    categories=[rotulo_p1, rotulo_p2], 
    ordered=True)

# Manter a ordem decrescente dos municípios com base no total de casos
df_agrupado['MUNICÍPIO - IBGE - UF'] = pd.Categorical(
    df_agrupado['MUNICÍPIO - IBGE - UF'], 
    categories=top_10_municipios, 
    ordered=True
)
df_agrupado = df_agrupado.sort_values(['MUNICÍPIO - IBGE - UF', 'Período'])

# 3. CONSTRUÇÃO DO GRÁFICO
grafico_municipio = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Gráfico de Colunas Agrupadas (usando 'hue' para comparar os 2 períodos)
sns.barplot(
    data=df_agrupado, 
    x='MUNICÍPIO - IBGE - UF', 
    y='Casos', 
    hue='Período', 
    hue_order=[rotulo_p1, rotulo_p2],
    palette=['#778da9', '#1b263b'],
    ax=ax)

# Adicionar rótulos com a quantidade em todas as colunas
# O 'for' aqui já funciona como uma trava natural: se não houver colunas, ele não tenta colocar rótulos
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3, fontsize=9)

# Títulos e Eixos
ax.set_title(
    f'Casos prováveis de Chikungunya - SE {semanas_analise} a SE {se_temporada_atual} - Brasil - 2026'.upper(), 
    fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município - UF', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)  
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.3)

# 4. Legenda Segura (Evitando o UserWarning)
handles, labels = ax.get_legend_handles_labels()
if handles: # Se existirem itens para a legenda, ele desenha
    ax.legend(handles, labels, title='Recorte Semanal', fontsize=11, loc='upper right', frameon=True)
else:
    print("Aviso: Legenda não gerada porque o gráfico está vazio. Verifique os dados.")

# 5. RODAPÉ E SALVAMENTO
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (
    f'Fonte: SINAN | Dados até: {data_analise}\n')
ax_rodape.text(0.0, -0.3, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_municipios_comparativo_oito_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
TOP_TABELA_MUNICIPIOS = 15
SE_LIMITE_TABELA = int(se_analise)

pasta_tabela_top15 = Path.cwd() / f"analises_chikungunya_brasil_SE{SE_LIMITE_TABELA}"
pasta_tabela_top15.mkdir(parents=True, exist_ok=True)

# 1. FUNÇÕES AUXILIARES

def codigo_top15(serie):
    codigo = serie.astype("string").str.strip().str.replace(
        r"\.0+$", "", regex=True
    )
    return codigo.where(
        codigo.str.fullmatch(r"\d{6,7}", na=False)
    ).str[:6]

def contar_casos_top15(base, ano):
    dados = base.copy()

    if "SEM_PRI" in dados.columns:
        semana_completa = pd.to_numeric(
            dados["SEM_PRI"], errors="coerce"
        )
        ano_sintomas, semana = semana_completa // 100, semana_completa % 100
    else:
        ano_sintomas = pd.to_datetime(
            dados["DT_SIN_PRI"], errors="coerce"
        ).dt.year
        semana = pd.to_numeric(dados["SE"], errors="coerce")

    dados = dados.loc[
        ano_sintomas.eq(ano)
        & semana.between(1, SE_LIMITE_TABELA)
    ].copy()

    dados["codigo_municipio"] = codigo_top15(dados["ID_MN_RESI"])

    ufs_validas = (
        [str(k) for k in mapa_uf.keys()]
        if isinstance(mapa_uf, dict)
        else [str(x) for x in mapa_uf]
    )

    return (
        dados.loc[
            dados["codigo_municipio"].str[:2].isin(ufs_validas)
        ]
        .groupby("codigo_municipio")
        .size()
    )


def formatar_top15(valor, percentual=False, incluir_sinal_variacao=False):
    if pd.isna(valor) or not isinstance(valor, (int, float)):
        return "—"

    if percentual:
        if incluir_sinal_variacao:
            sinal = "▲ " if valor > 0 else ("▼ " if valor < 0 else "")
            return f"{sinal}{abs(valor) * 100:.2f}%".replace(".", ",")
        else:
            return f"{valor * 100:.2f}%".replace(".", ",")

    return f"{int(valor):,}".replace(",", ".")

# 2. PROCESSAMENTO E RANKING DOS DADOS

casos_2025 = contar_casos_top15(
    df_provaveis_2025, 2025
).rename("Casos 2025")

casos_2026 = contar_casos_top15(
    df_provaveis, 2026
).rename("Casos 2026")

ranking = (
    casos_2026.reset_index()
    .sort_values(
        ["Casos 2026", "codigo_municipio"],
        ascending=[False, True]
    )
    .head(TOP_TABELA_MUNICIPIOS)
)

if ranking.empty:
    raise ValueError("Não há casos prováveis em 2026 no período selecionado.")

tabela = ranking.merge(
    casos_2025.reset_index(),
    on="codigo_municipio",
    how="left",
    validate="one_to_one"
)

tabela["Casos 2025"] = tabela["Casos 2025"].fillna(0).astype(int)

# Nomes dos Municípios
ref_mun = df_municipios[
    ["COD_IBGE", "MUNICÍPIO - IBGE - UF"]
].copy()

ref_mun["codigo_municipio"] = codigo_top15(ref_mun["COD_IBGE"])

ref_mun = (
    ref_mun.dropna(subset=["codigo_municipio"])
    .drop_duplicates(subset=["codigo_municipio"])
)

tabela = tabela.merge(
    ref_mun,
    on="codigo_municipio",
    how="left",
    validate="one_to_one"
)

tabela["Município"] = tabela[
    "MUNICÍPIO - IBGE - UF"
].fillna(tabela["codigo_municipio"])

# GAL 2026
gal = df_gal_final[
    ["IBGE", "SE", "exames_realizados", "casos_positivos"]
].copy()

gal["codigo_municipio"] = codigo_top15(gal["IBGE"])
gal["SE"] = pd.to_numeric(gal["SE"], errors="coerce")

gal = gal.loc[
    gal["SE"].between(1, SE_LIMITE_TABELA)
    & gal["codigo_municipio"].isin(tabela["codigo_municipio"])
].copy()

for col in ["exames_realizados", "casos_positivos"]:
    gal[col] = pd.to_numeric(gal[col], errors="coerce").fillna(0)

resumo_gal = (
    gal.groupby("codigo_municipio")[
        ["exames_realizados", "casos_positivos"]
    ]
    .sum()
    .rename(columns={
        "exames_realizados": "Exames realizados",
        "casos_positivos": "Exames positivos"
    })
    .reset_index()
)

tabela = tabela.merge(
    resumo_gal,
    on="codigo_municipio",
    how="left",
    validate="one_to_one"
)

tabela[["Exames realizados", "Exames positivos"]] = (
    tabela[["Exames realizados", "Exames positivos"]].fillna(0)
)

if (tabela["Exames positivos"] > tabela["Exames realizados"]).any():
    raise ValueError("Há mais exames positivos do que realizados.")

# Totais e Indicadores Finais
contagens = [
    "Casos 2025",
    "Casos 2026",
    "Exames realizados",
    "Exames positivos"
]

tabela = tabela[["Município"] + contagens]

total_row = {
    "Município": "TOTAL",
    **{col: tabela[col].sum() for col in contagens}
}

tabela = pd.concat(
    [tabela, pd.DataFrame([total_row])],
    ignore_index=True
)

tabela["Variação (%)"] = (
    (tabela["Casos 2026"] - tabela["Casos 2025"])
    / tabela["Casos 2025"].where(tabela["Casos 2025"].gt(0))
)

tabela["Positividade"] = (
    tabela["Exames positivos"]
    / tabela["Exames realizados"].where(
        tabela["Exames realizados"].gt(0)
    )
)

tabela = tabela[[
    "Município",
    "Casos 2025",
    "Casos 2026",
    "Variação (%)",
    "Exames realizados",
    "Exames positivos",
    "Positividade"
]]

# 3. EXPORTAÇÃO PLANILHA XLSX

arquivo_xlsx = pasta_tabela_top15 / (
    f"chik_brasil_top15_municipios_2025_2026_SE{SE_LIMITE_TABELA}.xlsx"
)

with pd.ExcelWriter(arquivo_xlsx, engine="openpyxl") as writer:
    tabela.to_excel(
        writer, sheet_name="Top15_municipios", index=False
    )

    ws = writer.sheets["Top15_municipios"]
    ws.freeze_panes = "B2"
    ws.column_dimensions["A"].width = 42

    for col in ["B", "C", "D", "E", "F", "G"]:
        ws.column_dimensions[col].width = 22

    # Cabeçalho com cor #a0a0a0
    for cell in ws[1]:
        cell.fill = PatternFill("solid", fgColor="A0A0A0")
        cell.font = Font(color="000000", bold=True)
        cell.alignment = Alignment(
            horizontal="center", wrap_text=True
        )

    for row in ws.iter_rows(min_row=2):
        for cell in row[1:]:
            cell.number_format = (
                "0.00%" if cell.column in [4, 7] else "#,##0"
            )

        var_val = row[3].value

        if var_val is not None and pd.notna(var_val):
            try:
                v = float(var_val)
                row[3].fill = PatternFill(
                    "solid",
                    fgColor=(
                        "FCE4D6" if v > 0
                        else ("E2EFDA" if v < 0 else "000000")
                    )
                )

                if v > 0:
                    row[3].font = Font(color="C00000", bold=True)
                elif v < 0:
                    row[3].font = Font(color="375623", bold=True)

            except (ValueError, TypeError):
                pass

    # Linha de totais com cor #a0a0a0
    for cell in ws[ws.max_row]:
        cell.fill, cell.font = (
            PatternFill("solid", fgColor="A0A0A0"),
            Font(color="000000", bold=True)
        )

    ws.cell(
        row=ws.max_row + 2,
        column=1,
        value=(
            f"SE1–{SE_LIMITE_TABELA} de cada ano. "
            "GAL referente a 2026. Total dos municípios."
        )
    )

# 4. EXPORTAÇÃO FIGURA PNG — TOP 15 MUNICÍPIOS

def gerar_figura_top15_padronizada(
    tabela_top15,
    pasta_saida,
    semana_final,
    data_referencia
):
    dados_visuais = tabela_top15.copy()

    def percentual_visual(valor):
        if pd.isna(valor):
            return "—"

        return f"{float(valor) * 100:.2f}%".replace(".", ",")

    linhas_visuais = []
    variacoes_visuais = []

    for _, linha in dados_visuais.iterrows():
        variacao = pd.to_numeric(
            linha["Variação (%)"],
            errors="coerce"
        )

        if (
            linha["Casos 2025"] == 0
            and linha["Casos 2026"] > 0
        ):
            texto_variacao = "Novo*"

        elif pd.isna(variacao):
            texto_variacao = "—"

        else:
            sinal = (
                "▲ " if variacao > 0
                else ("▼ " if variacao < 0 else "")
            )

            texto_variacao = (
                sinal + percentual_visual(abs(variacao))
            )

        linhas_visuais.append([
            str(linha["Município"]),
            formatar_valor_tabela_padronizada(
                linha["Casos 2025"]
            ),
            formatar_valor_tabela_padronizada(
                linha["Casos 2026"]
            ),
            texto_variacao,
            formatar_valor_tabela_padronizada(
                linha["Exames realizados"]
            ),
            formatar_valor_tabela_padronizada(
                linha["Exames positivos"]
            ),
            percentual_visual(linha["Positividade"])
        ])

        variacoes_visuais.append(variacao)

    caminho_figura = pasta_saida / (
        "chik_brasil_top15_municipios_2025_2026_"
        f"SE{int(semana_final)}.png"
    )

    quantidade_municipios = (
        ~dados_visuais["Município"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("TOTAL")
    ).sum()

    desenhar_tabela_padronizada(
        linhas=linhas_visuais,
        cabecalhos=[
            "Município",
            "Casos 2025",
            "Casos 2026",
            "Variação (%)",
            "Exames realizados",
            "Exames positivos",
            "Positividade"
        ],
        larguras=[
            0.27, 0.10, 0.10, 0.14,
            0.14, 0.13, 0.12
        ],
        titulo=(
            f"MUNICÍPIOS COM MAIS CASOS "
            "PROVÁVEIS DE CHIKUNGUNYA "
            f"— BRASIL — SE1 A SE{int(semana_final)} — 2026"
        ),
        rodape=(
            f"Fontes: SINAN e GAL | Dados até: {data_referencia}. "
            f"Comparação SE1–{int(semana_final)} de cada ano. "
            "Exames e positividade referentes a 2026.\n"
            "*Novo: sem registros em 2025 no período. "
            "Total dos municípios apresentados."
        ),
        caminho_saida=caminho_figura,
        coluna_variacao=3,
        valores_variacao=variacoes_visuais
    )

    return caminho_figura


arquivo_png = gerar_figura_top15_padronizada(
    tabela_top15=tabela,
    pasta_saida=pasta_tabela_top15,
    semana_final=SE_LIMITE_TABELA,
    data_referencia=data_analise
)

# 5. NOVA ABA XLSX — TODOS OS MUNICÍPIOS DO BRASIL

# Processamento independente para preservar a tabela e a figura top 15.
gal_todos_municipios = df_gal_final[
    ["IBGE", "SE", "exames_realizados", "casos_positivos"]
].copy()

gal_todos_municipios["codigo_municipio"] = codigo_top15(
    gal_todos_municipios["IBGE"]
)

gal_todos_municipios["SE"] = pd.to_numeric(
    gal_todos_municipios["SE"], errors="coerce"
)

gal_todos_municipios = gal_todos_municipios.loc[
    gal_todos_municipios["SE"].between(1, SE_LIMITE_TABELA)
    & gal_todos_municipios["SE"].mod(1).eq(0)
    & gal_todos_municipios["codigo_municipio"].notna()
].copy()

gal_todos_municipios["SE"] = (
    gal_todos_municipios["SE"].astype(int)
)

for coluna in ["exames_realizados", "casos_positivos"]:
    gal_todos_municipios[coluna] = pd.to_numeric(
        gal_todos_municipios[coluna], errors="coerce"
    ).fillna(0)

gal_semanal_municipios = (
    gal_todos_municipios
    .groupby(["codigo_municipio", "SE"], as_index=False)[
        ["exames_realizados", "casos_positivos"]
    ]
    .sum()
)

if (
    gal_semanal_municipios["casos_positivos"]
    > gal_semanal_municipios["exames_realizados"]
).any():
    raise ValueError(
        "Há municípios/semanas com mais exames positivos "
        "do que realizados."
    )

if (
    gal_semanal_municipios[
        ["exames_realizados", "casos_positivos"]
    ].lt(0).any().any()
):
    raise ValueError("Há quantidades negativas de exames no GAL.")

gal_acumulado_municipios = (
    gal_semanal_municipios
    .groupby("codigo_municipio")[
        ["exames_realizados", "casos_positivos"]
    ]
    .sum()
    .rename(columns={
        "exames_realizados": "Exames realizados",
        "casos_positivos": "Exames positivos"
    })
    .reset_index()
)

# A referência IBGE inclui também municípios sem casos no período.
tabela_chik_municipios = (
    ref_mun[
        ["codigo_municipio", "MUNICÍPIO - IBGE - UF"]
    ]
    .merge(
        casos_2025.reset_index(),
        on="codigo_municipio",
        how="left",
        validate="one_to_one"
    )
    .merge(
        casos_2026.reset_index(),
        on="codigo_municipio",
        how="left",
        validate="one_to_one"
    )
    .merge(
        gal_acumulado_municipios,
        on="codigo_municipio",
        how="left",
        validate="one_to_one"
    )
)

tabela_chik_municipios["Município"] = (
    tabela_chik_municipios["MUNICÍPIO - IBGE - UF"]
    .fillna(tabela_chik_municipios["codigo_municipio"])
)

tabela_chik_municipios[contagens] = (
    tabela_chik_municipios[contagens].fillna(0)
)

tabela_chik_municipios["Variação (%)"] = (
    (
        tabela_chik_municipios["Casos 2026"]
        - tabela_chik_municipios["Casos 2025"]
    )
    / tabela_chik_municipios["Casos 2025"].where(
        tabela_chik_municipios["Casos 2025"].gt(0)
    )
)

tabela_chik_municipios["Positividade"] = (
    tabela_chik_municipios["Exames positivos"]
    / tabela_chik_municipios["Exames realizados"].where(
        tabela_chik_municipios["Exames realizados"].gt(0)
    )
)

tabela_chik_municipios = (
    tabela_chik_municipios
    .sort_values(
        ["Casos 2026", "Município"],
        ascending=[False, True]
    )
    [[
        "Município",
        "Casos 2025",
        "Casos 2026",
        "Variação (%)",
        "Exames realizados",
        "Exames positivos",
        "Positividade"
    ]]
    .reset_index(drop=True)
)

# Abre o arquivo já criado e acrescenta somente a nova aba.
with pd.ExcelWriter(
    arquivo_xlsx,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    tabela_chik_municipios.to_excel(
        writer,
        sheet_name="chik_municipios",
        index=False
    )

    ws_municipios = writer.sheets["chik_municipios"]
    ws_municipios.freeze_panes = "B2"
    ws_municipios.auto_filter.ref = ws_municipios.dimensions
    ws_municipios.column_dimensions["A"].width = 42

    for coluna in ["B", "C", "D", "E", "F", "G"]:
        ws_municipios.column_dimensions[coluna].width = 22

    for celula in ws_municipios[1]:
        celula.fill = PatternFill("solid", fgColor="A0A0A0")
        celula.font = Font(color="000000", bold=True)
        celula.alignment = Alignment(
            horizontal="center",
            wrap_text=True
        )

    ws_municipios.row_dimensions[1].height = 30

    for linha in ws_municipios.iter_rows(min_row=2):
        for celula in linha[1:]:
            celula.number_format = (
                "0.00%" if celula.column in [4, 7] else "#,##0"
            )

        variacao_municipio = linha[3].value

        variacao_municipio = pd.to_numeric(
            linha[3].value,
            errors="coerce"
        )

        if pd.notna(variacao_municipio):
            if variacao_municipio > 0:
                linha[3].fill = PatternFill(
                    "solid", fgColor="FCE4D6"
                )
                linha[3].font = Font(
                    color="C00000", bold=True
                )

            elif variacao_municipio < 0:
                linha[3].fill = PatternFill(
                    "solid", fgColor="E2EFDA"
                )
                linha[3].font = Font(
                    color="375623", bold=True
                )

# 6. PAINEL 4 × 4 — POSITIVIDADE SEMANAL DO TOP 15

from matplotlib.ticker import MaxNLocator

municipios_painel_positividade = ranking.merge(
    ref_mun[
        ["codigo_municipio", "MUNICÍPIO - IBGE - UF"]
    ],
    on="codigo_municipio",
    how="left",
    validate="one_to_one"
)

municipios_painel_positividade["Município"] = (
    municipios_painel_positividade["MUNICÍPIO - IBGE - UF"]
    .fillna(municipios_painel_positividade["codigo_municipio"])
)

semanas_painel_positividade = pd.Index(
    range(1, SE_LIMITE_TABELA + 1),
    name="SE"
)

with plt.rc_context({
    "font.family": "Arial",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": False
}):
    fig_positividade, eixos_positividade = plt.subplots(
        4, 4,
        figsize=(22, 16),
        sharex=True,
        sharey=True
    )

    for ax_pos, (_, municipio_pos) in zip(
        eixos_positividade.flat,
        municipios_painel_positividade.iterrows()
    ):
        dados_pos = (
            gal_semanal_municipios.loc[
                gal_semanal_municipios["codigo_municipio"].eq(
                    municipio_pos["codigo_municipio"]
                )
            ]
            .set_index("SE")[
                ["exames_realizados", "casos_positivos"]
            ]
            .reindex(semanas_painel_positividade)
        )

        # Sem exames: mantém lacuna, sem atribuir positividade zero.
        positividade_pos = (
            dados_pos["casos_positivos"]
            / dados_pos["exames_realizados"].where(
                dados_pos["exames_realizados"].gt(0)
            )
            * 100
        )

        ax_pos.plot(
            semanas_painel_positividade,
            positividade_pos,
            color="#AA0202",
            linewidth=1.7,
            marker="o",
            markersize=3
        )

        ax_pos.set_title(
            textwrap.fill(
                str(municipio_pos["Município"]), width=30
            ),
            fontsize=13,
            fontweight="bold",
            pad=10
        )

        ax_pos.set_ylim(0, 105)
        ax_pos.set_yticks([0, 25, 50, 75, 100])
        ax_pos.set_xlim(0.5, SE_LIMITE_TABELA + 0.5)

        ax_pos.xaxis.set_major_locator(
            MaxNLocator(integer=True, nbins=6)
        )

        ax_pos.set_xlabel(
            "Semana Epidemiológica", fontsize=12
        )
        ax_pos.set_ylabel(
            "Positividade (%)", fontsize=12
        )

        ax_pos.tick_params(
            axis="both",
            labelsize=10,
            labelbottom=True,
            labelleft=True
        )

        ax_pos.set_axisbelow(True)

        ax_pos.grid(
            axis="y",
            color="#DCE1E7",
            linestyle="--",
            linewidth=0.6
        )

        ax_pos.spines[["top", "right"]].set_visible(False)

        if not positividade_pos.notna().any():
            ax_pos.text(
                0.5, 0.5,
                "Sem exames no período",
                transform=ax_pos.transAxes,
                ha="center",
                va="center",
                fontsize=10,
                color="#666666"
            )

    # Oculta o último espaço do painel.
    for ax_pos in list(eixos_positividade.flat)[
        len(municipios_painel_positividade):
    ]:
        ax_pos.set_visible(False)

    fig_positividade.suptitle(
        "POSITIVIDADE SEMANAL PARA CHIKUNGUNYA\n"
        "15 MUNICÍPIOS COM MAIS CASOS PROVÁVEIS EM 2026 "
        f"- BRASIL - SE1 A SE{SE_LIMITE_TABELA}",
        fontsize=20,
        fontweight="bold",
        y=0.95
    )

    fig_positividade.text(
        0.035, 0.015,
        f"Fonte: GAL | Dados até: {data_analise}.\n"
        "Positividade = exames positivos / exames realizados × 100. "
        "Semanas sem exames permanecem sem ponto na linha.",
        fontsize=10
    )

    fig_positividade.tight_layout(
        rect=[0.02, 0.065, 0.99, 0.94],
        h_pad=2.5,
        w_pad=1.8
    )

    arquivo_painel_positividade = pasta_tabela_top15 / (
        "chik_brasil_top15_municipios_positividade_semanal_"
        f"2026_SE{SE_LIMITE_TABELA}.png"
    )

    fig_positividade.savefig(
        arquivo_painel_positividade,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

## 4. Vigilância laboratorial


In [ ]:
from PIL import Image

# Remove o limite de pixels para evitar o erro de decompression bomb
Image.MAX_IMAGE_PIXELS = None

# Exames realizados (barras) e positividade (linha) por SE e UF
se_final_grafico = se_final
semanas = range(1, se_final_grafico + 1)

dados_grafico = df_gal_final[['UF', 'SE', 'exames_realizados', 'casos_positivos']].copy()
dados_grafico['UF'] = pd.to_numeric(dados_grafico['UF'], errors='coerce')
dados_grafico['SE'] = pd.to_numeric(dados_grafico['SE'], errors='coerce')
dados_grafico['exames_realizados'] = pd.to_numeric(
    dados_grafico['exames_realizados'], errors='coerce'
).fillna(0)
dados_grafico['casos_positivos'] = pd.to_numeric(
    dados_grafico['casos_positivos'], errors='coerce'
).fillna(0)
dados_grafico = dados_grafico[dados_grafico['SE'].between(1, se_final_grafico)]

totais_uf_se = dados_grafico.groupby(['UF', 'SE'])[
    ['exames_realizados', 'casos_positivos']
].sum()

ufs = sorted(mapa_uf.items(), key=lambda item: item[1])

# Grade de 4 linhas x 7 colunas com proporção horizontal otimizada
fig, eixos = plt.subplots(4, 7, figsize=(37, 15), sharex=True)
fig.suptitle(
    f'Exames realizados e positividade para chikungunya por semana epidemiológica — UF — 2026'.upper(),
    fontsize=24, fontweight='bold', y=0.98
)

for ax, (codigo_uf, sigla_uf) in zip(eixos.flat, ufs):
    if codigo_uf in totais_uf_se.index.get_level_values('UF'):
        dados_uf = totais_uf_se.loc[codigo_uf].reindex(semanas, fill_value=0)
    else:
        dados_uf = pd.DataFrame(
            0, index=semanas,
            columns=['exames_realizados', 'casos_positivos']
        )

    exames = dados_uf['exames_realizados']
    positivos = dados_uf['casos_positivos']

    # Cálculo da taxa de positividade global do estado
    total_exames = exames.sum()
    total_positivos = positivos.sum()

    taxa_positividade_estado = (
        (total_positivos / total_exames * 100) 
        if total_exames > 0 else 0.0
    )
    
    positividade = (
        dados_uf['casos_positivos']
        .div(exames.where(exames.ne(0)))
        .mul(100)
    )

    ax.bar(
        semanas, exames,
        color=['#ade8f4' if se <= 26 else '#e9d8a6' for se in semanas],
        width=0.88
    )
    ax.axvline(26.5, color='#74879a', linestyle='--', linewidth=1.2)
    
    # Título do subplot proporcional
    ax.set_title(
        f'{sigla_uf} (exames: {int(exames.sum()):,} | positividade: {taxa_positividade_estado:.1f}%)'.replace(',', '.'),
        fontsize=14, fontweight='bold', pad=15
    )
    
    ax.set_xlim(0.5, se_final_grafico + 0.5)
    ax.set_xticks([1, 5, 10, 15, 20, 25, 30, 35, se_final])
    
    # Fontes eixos principais
    ax.tick_params(axis='x', labelbottom=True, labelsize=10)
    ax.tick_params(axis='y', labelsize=10)
    ax.grid(axis='y', linewidth=0.0, linestyle='--', alpha=0.2)
    ax.grid(axis='x', linewidth=0.8, linestyle='--', alpha=0.2)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

    # Eixo secundário (positividade)
    ax_positividade = ax.twinx()
    ax_positividade.plot(
        semanas, positividade,
        color='#982f45', linewidth=1.5, marker='o', markersize=2.0
    )
    ax_positividade.set_ylim(0, 100)
    ax.set_yticks([0, 25, 50, 75, 100]) if False else None # apenas garantindo escopo correto
    ax_positividade.set_yticks([0, 25, 50, 75, 100])
    ax_positividade.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
    ax_positividade.tick_params(axis='y', labelsize=9)
    ax_positividade.grid(axis='y', linewidth=1, linestyle='--', alpha=0.1)
    ax_positividade.spines[['top']].set_visible(False)

# Oculta os subplots vazios excedentes (4x7 = 28 espaços para 27 UFs)
for ax in eixos.flat[len(ufs):]:
    ax.set_visible(False)

fig.supxlabel('Semana Epidemiológica', fontsize=16, y=0.04)
fig.supylabel('Exames Realizados | Positividade (%)', fontsize=16, x=0.04)

fig.tight_layout(rect=(0.04, 0.04, 0.96, 0.96), h_pad=2.5, w_pad=1.5)

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_casos_positividade_painel_uf.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# COMPARATIVO ENTRE CONFIRMADOS SINAN E CONFIRMADOS GAL
# Filtrar casos confirmados na base do SINAN
df_semanas = df_notificados[df_notificados['CLASSI_FIN'] == 13]['SE'].value_counts().reset_index()
df_semanas.columns = ['SE', 'Total']
df_semanas = df_semanas.sort_values('SE').reset_index(drop=True)

# Contar os casos positivos na base do GAL
df_conf = df_gal_final.groupby('SE')['casos_positivos'].sum().reset_index()
df_conf.columns = ['SE', 'Total']

df_semanas['Confirmados'] = 'Confirmados (SINAN)'
df_conf['Confirmados'] = 'Confirmados (GAL)'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_combinado = pd.concat([df_semanas, df_conf], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_combinado, x='SE', y='Total', hue='Confirmados',
            palette= ['#ee9b00', '#780116'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title('Casos confirmados de chikungunya por SE - GAL e SINAN - Brasil - 2026'.upper(), 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
ax.set_xlim(-1, df_combinado['SE'].max() + 0)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: SINAN e GAL | Dados até: {data_analise} | '
                f'Total de confirmados SINAN: {df_semanas['Total'].sum():.0f} | '
                f'Total de confirmados GAL: {df_conf['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=10, color='black')

# Finalização e Salvamento
#salvar_pasta = os.path.join(local_pasta, 'chik_brasil_confirmados_sinan_gal.png')
#plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# COMPARATIVO ENTRE CONFIRMADOS GAL - DENGUE E CHIKUNGUNYA
# Contar os casos positivos na base do GAL
df_conf_dengue = df_gal_dengue.groupby('SE')['casos_positivos'].sum().reset_index()
df_conf_dengue.columns = ['SE', 'Total']

# Contar os casos positivos na base do GAL
df_conf_chik = df_gal_final.groupby('SE')['casos_positivos'].sum().reset_index()
df_conf_chik.columns = ['SE', 'Total']

df_conf_dengue['Confirmados'] = 'Confirmados Dengue'
df_conf_chik['Confirmados'] = 'Confirmados Chikungunya'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_combinado = pd.concat([df_conf_dengue, df_conf_chik], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_combinado, x='SE', y='Total', hue='Confirmados',
            palette= ['#ee9b00', '#780116'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title('Comparativo de casos confirmados de dengue e chikungunya por SE - GAL - Brasil - 2026'.upper(), 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
ax.set_xlim(-1, df_combinado['SE'].max() + 0)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: GAL | Dados até: {data_analise} | '
                f'Total de confirmados Dengue: {df_conf_dengue['Total'].sum():.0f} | '
                f'Total de confirmados Chikungunya: {df_conf_chik['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=10, color='black')

# Finalização e Salvamento
#salvar_pasta = os.path.join(local_pasta, 'chik_brasil_confirmados_dengue_chik.png')
#plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1. PREPARAÇÃO DOS DADOS
# Contar exames realizados na base do GAL
df_exames = df_gal_final.groupby('SE')['exames_realizados'].sum().reset_index()
df_exames.columns = ['SE', 'Exames_Realizados']

# Contar os casos positivos na base do GAL
df_conf = df_gal_final.groupby('SE')['casos_positivos'].sum().reset_index()
df_conf.columns = ['SE', 'Exames_Confirmados']

# Juntar as duas bases em uma só por Semana Epidemiológica
df_gal_combinado = pd.merge(df_exames, df_conf, on='SE', how='left').fillna(0)

# Calcular o Percentual de Positividade (Evitando divisão por zero)
df_gal_combinado['Positividade_Pct'] = np.where(
    df_gal_combinado['Exames_Realizados'] > 0,
    (df_gal_combinado['Exames_Confirmados'] / df_gal_combinado['Exames_Realizados']) * 100,
    0
)

# 2. CONFIGURAÇÃO VISUAL
grafico_sem = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

color_line = '#780116'

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax1 = grafico_sem.add_subplot(gs[0])

# 3. GRÁFICO DE COLUNAS (Exames Realizados - Eixo Y Principal)
sns.barplot(
    data=df_gal_combinado, 
    x='SE', 
    y='Exames_Realizados', 
    color='#dee2e6', 
    ax=ax1, 
    width=0.8
)

# Adicionar os rótulos nas barras (Volume absoluto)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%d', padding=3, fontsize=10, color='#adb5bd')

# 4. GRÁFICO DE LINHA (Positividade - Eixo Y Secundário)
ax2 = ax1.twinx()  # Cria um eixo Y secundário compartilhando o mesmo eixo X

# Plotar a linha usando os ticks exatos gerados pelo barplot para garantir o alinhamento
x_pos = ax1.get_xticks()
ax2.plot(
    x_pos, 
    df_gal_combinado['Positividade_Pct'], 
    color= color_line, 
    marker='o', 
    linewidth=2.5, 
    markersize=8
)

# Adicionar os rótulos na linha (Percentual)
offset_y = df_gal_combinado['Positividade_Pct'].max() * 0.05 # Deslocamento leve para o texto não ficar em cima da bolinha
for i, y in enumerate(df_gal_combinado['Positividade_Pct']):
    ax2.text(
        x_pos[i], 
        y + offset_y, 
        f'{y:.1f}', 
        color= color_line, # Destaque na cor vermelha para a porcentagem (opcional)
        ha='center', 
        va='bottom', 
        fontweight='bold', 
        fontsize=10 
    )

# 5. ESTILIZAÇÃO DOS EIXOS E LEGENDA
ax1.set_title('Exames realizados e positividade GAL para chikungunya por SE - Brasil - 2026'.upper(), 
              fontsize=18, fontweight='bold', pad=20)

ax1.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax1.set_ylabel('Exames Realizados', fontsize=16, labelpad=16, color='#000000')
ax2.set_ylabel('Positividade (%)', fontsize=16, labelpad=16, color='#000000')

# Ajustar o limite do eixo secundário para dar espaço aos rótulos da linha
ax2.set_ylim(0, df_gal_combinado['Positividade_Pct'].max() * 1.15)
ax2.set_xlim(-1, df_exames['SE'].max() + 0)

# Configurar as grades de fundo
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.grid(axis='x', linestyle='--', alpha=0.7)
ax2.grid(False) # Desliga a grade do eixo secundário para não poluir o gráfico

# Criar uma legenda combinada manual
patch_barras = mpatches.Patch(color='#dee2e6', label='Exames Realizados')
linha_percentual = lines.Line2D([], [], color= color_line, marker='o', label='Positividade (%)')
ax1.legend(handles=[patch_barras, linha_percentual], fontsize=11, loc='upper right', frameon=True)

# 6. RODAPÉ E SALVAMENTO
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')

# Separar as somas em variáveis evita erros de aspas na formatação do texto
total_realizados = df_gal_combinado['Exames_Realizados'].sum()
total_confirmados = df_gal_combinado['Exames_Confirmados'].sum()

rodape_texto = (f'Fonte: GAL | Dados até: {data_gal}')

ax_rodape.text(0.0, 0.5, rodape_texto, ha='left', va='center', fontsize=9, color='black')

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_realizados_confirmados_gal_semana.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1. PREPARAÇÃO DOS DADOS
# Agrupar casos prováveis por SE
df_prov = df_provaveis.groupby('SE')['SEM_PRI'].count().reset_index()
df_prov.columns = ['SE', 'Casos_Provaveis']

# Agrupar dados laboratoriais do GAL por SE
df_gal_agg = df_gal_final.groupby('SE')[['exames_realizados', 'casos_positivos']].sum().reset_index()

# Unir as bases
df_grafico = pd.merge(df_prov, df_gal_agg, on='SE', how='left').fillna(0)

# Calcular o percentual de positividade (%)
df_grafico['Positividade_Pct'] = np.where(
    df_grafico['exames_realizados'] > 0,
    (df_grafico['casos_positivos'] / df_grafico['exames_realizados']) * 100,
    0
)

# 2. CONFIGURAÇÃO VISUAL E FIGURA
sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(20, 10))

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax1 = fig.add_subplot(gs[0])

# 3. GRÁFICO DE COLUNAS (Casos Prováveis - Eixo Y Principal)
cor_barras = '#dee2e6' 

sns.barplot(
    data=df_grafico,
    x='SE',
    y='Casos_Provaveis',
    color=cor_barras,
    ax=ax1,
    width=0.8
)

# Rótulos simples em cima de cada barra
for container in ax1.containers:
    ax1.bar_label(container, fmt='%d', padding=3, fontsize=10, color=cor_barras)

# 4. GRÁFICO DE LINHA (Positividade GAL - Eixo Y Secundário)
ax2 = ax1.twinx()
x_pos = np.arange(len(df_grafico['SE']))
cor_linha = '#283618'  # Tom vinho/vermelho para contraste

ax2.plot(
    x_pos,
    df_grafico['Positividade_Pct'],
    color=cor_linha,
    marker='o',
    linewidth=2.5,
    markersize=7
)

# Rótulos da positividade (%) na linha
ymax_pos = df_grafico['Positividade_Pct'].max()
offset_y = (ymax_pos * 0.03) if ymax_pos > 0 else 1

for i, y in enumerate(df_grafico['Positividade_Pct']):
    if y > 0:  # Exibe rótulo apenas se houver positividade > 0%
        ax2.text(
            x_pos[i],
            y + offset_y,
            f'{y:.1f}',
            color=cor_linha,
            ha='center',
            va='bottom',
            fontweight='bold',
            fontsize=9
        )

# 5. ESTILIZAÇÃO E AJUSTES DE LIMITES
ax1.set_title(
    'CASOS PROVÁVEIS E POSITIVIDADE DO GAL PARA CHIKUNGUNYA POR SE - BRASIL - 2026',
    fontsize=16,
    fontweight='bold',
    pad=18
)

ax1.set_xlabel('Semana Epidemiológica', fontsize=12, labelpad=10)
ax1.set_ylabel('Casos Prováveis', fontsize=12, labelpad=10, color='#000000', fontweight='bold')
ax2.set_ylabel('Positividade GAL (%)', fontsize=12, labelpad=10, color='#000000', fontweight='bold')

# Limites dos eixos Y para evitar superposição dos rótulos
ymax_barras = df_grafico['Casos_Provaveis'].max()
ax1.set_ylim(0, ymax_barras * 1.15 if ymax_barras > 0 else 10)
ax2.set_ylim(0, ymax_pos * 1.25 if ymax_pos > 0 else 100)
ax1.set_xlim(-1, df_grafico['SE'].max() + 0)

# Suavização da grade
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.grid(axis='x', linestyle='--', alpha=0.5)
ax2.grid(False)

# Legenda unificada e discreta
barra_patch = mpatches.Patch(color=cor_barras, alpha=0.85, label='Casos Prováveis')
linha_legend = lines.Line2D([], [], color=cor_linha, marker='o', label='Positividade GAL (%)')

ax1.legend(
    handles=[barra_patch, linha_legend],
    loc='upper right',
    fontsize=10,
    frameon=True,
    facecolor='white',
    framealpha=0.9
)

# 6. RODAPÉ E SALVAMENTO
ax_rodape = fig.add_subplot(gs[1])
ax_rodape.axis('off')

rodape_texto = f'Fonte: Sinan e GAL | Dados até: {data_analise}'
ax_rodape.text(0.0, 0.5, rodape_texto, ha='left', va='center', fontsize=9, color='#000000')

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_casos_positividade_se.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# CASOS CONFIRMADOS POR CRITÉRIO DE CONFIRMAÇÃO APLICADO
# Filtrar casos conformados por critério laboratorial na base do SINAN
df_conf_lab = df_notificados[(df_notificados['CLASSI_FIN'] == 13) & (df_notificados['CRITERIO'] == 1)]['SE'].value_counts().reset_index()
df_conf_lab.columns = ['SE', 'Total']
df_conf_lab = df_conf_lab.sort_values('SE').reset_index(drop=True)

# Filtrar casos confirmados por criterio clini-epidemiologico na base do SINAN
df_conf_clep = df_notificados[(df_notificados['CLASSI_FIN'] == 13) & (df_notificados['CRITERIO'] == 2)]['SE'].value_counts().reset_index()
df_conf_clep.columns = ['SE', 'Total']
df_conf_clep = df_conf_clep.sort_values('SE').reset_index(drop=True)

df_conf_lab['Confirmados'] = 'Laboratorial (SINAN)'
df_conf_clep['Confirmados'] = 'Clinico-Epidemiológico'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_combinado = pd.concat([df_conf_lab, df_conf_clep], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_combinado, x='SE', y='Total', hue='Confirmados',
            palette= ['#ee9b00', '#780116'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title('Casos confirmados de chikungunya por SE e critério de confirmação - Brasil - 2026'.upper(), 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
ax.set_xlim(-1, df_combinado['SE'].max() + 0)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise} | '
                f'Confirmados - Laboratorial: {df_conf_lab['Total'].sum():.0f} | '
                f'Confirmados - Clinico-Epidemiológico: {df_conf_clep['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_confirmados_criterio.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Ajuste somente estes nomes se forem diferentes no pipeline
SINAN = {'Dengue': den_provaveis, 'Chikungunya': df_provaveis}
GAL = {'Dengue': df_gal_dengue, 'Chikungunya': df_gal_final}

PASTA_SAIDA = (
    Path(local_pasta) if 'local_pasta' in globals() else Path('.')
)
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

CORES = {
    'Casos prováveis': '#e9c46a',
    'Exames realizados': '#e07a5f',
    'Exames positivos': '#f95738'
}

def resumir_sinan(df):
    d = df.assign(
        SE=pd.to_numeric(df['SE'], errors='coerce')
    ).dropna(subset=['SE'])

    d = d[d['SE'].between(1, 53)].assign(
        SE=lambda x: x['SE'].astype(int)
    )

    return (
        d.groupby('SE')['casos_provaveis'].sum()
        if 'casos_provaveis' in d
        else d.groupby('SE').size()
    ).rename('Casos prováveis')


def resumir_gal(df):
    colunas = ['exames_realizados', 'casos_positivos']

    d = df.assign(
        SE=pd.to_numeric(df['SE'], errors='coerce')
    ).dropna(subset=['SE'])

    d = d[d['SE'].between(1, 53)].assign(
        SE=lambda x: x['SE'].astype(int)
    )

    d[colunas] = (
        d[colunas].apply(pd.to_numeric, errors='coerce').fillna(0)
    )

    return (
        d.groupby('SE')[colunas]
        .sum()
        .rename(columns={
            'exames_realizados': 'Exames realizados',
            'casos_positivos': 'Exames positivos'
        })
    )


sinan_se = {
    doenca: resumir_sinan(df)
    for doenca, df in SINAN.items()
}
gal_se = {
    doenca: resumir_gal(df)
    for doenca, df in GAL.items()
}

se_final = int(min(
    *(s.index.max() for s in sinan_se.values()),
    *(g.index.max() for g in gal_se.values())
))


def consolidar(doenca, janela):
    semanas = pd.Index(
        range(se_final - janela + 1, se_final + 1),
        name='SE'
    )

    return (
        pd.concat([sinan_se[doenca], gal_se[doenca]], axis=1)
        .reindex(semanas, fill_value=0)
        .reset_index()
    )


# Apenas as últimas oito semanas
dados = {
    (doenca, 8): consolidar(doenca, 8)
    for doenca in ('Dengue', 'Chikungunya')
}

print(
    f'Fechamento comum: SE {se_final} | '
    f'8 SE: {se_final - 7}–{se_final}'
)


def grafico(ax, df, titulo):
    x, largura = np.arange(len(df)), 0.25

    for metrica, deslocamento in zip(
        CORES, (-largura, 0, largura)
    ):
        barras = ax.bar(
            x + deslocamento,
            df[metrica],
            largura,
            label=metrica,
            color=CORES[metrica],
            edgecolor='white',
            linewidth=.5,
            zorder=3
        )

        ax.bar_label(
            barras,
            labels=[
                f'{v:,.0f}'.replace(',', '.') if v else ''
                for v in df[metrica]
            ],
            padding=4,
            fontsize=10,
            rotation=90
        )

    ax.set(
        title=titulo,
        xlabel='Semana epidemiológica',
        ylabel='Número de registros',
        xticks=x,
        xticklabels=[f'SE {s}' for s in df['SE']]
    )

    ax.title.set_fontweight('bold')
    ax.xaxis.label.set_fontweight('bold')
    ax.yaxis.label.set_fontweight('bold')

    ax.set_ylim(
        0, df[list(CORES)].to_numpy().max() * 1.23
    )

    ax.grid(
        axis='y', linestyle='--', alpha=0.6, zorder=0
    )
    ax.grid(
        axis='x', linestyle='--', alpha=0.6, zorder=0
    )
    ax.spines[['top', 'right']].set_visible(True)


def rodape(fig):
    fig.text(
        .01, .01,
        f'Fonte: SINAN e GAL | Dados até {data_analise}.',
        fontsize=9
    )


# Painel 1 × 2 — somente oito semanas
fig, axes = plt.subplots(1, 2, figsize=(19, 7))

ordem = [('Dengue', 8), ('Chikungunya', 8)]

for ax, chave in zip(axes.flat, ordem):
    grafico(
        ax,
        dados[chave],
        f'{chave[0]} — últimas {chave[1]} SE'.upper()
    )

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    ncol=3,
    frameon=False,
    loc='upper center',
    bbox_to_anchor=(.5, .85)
)

fig.suptitle(
    (
        'Casos prováveis, exames realizados e exames positivos '
        f'para dengue e chikungunya - Brasil - 2026'
    ).upper(),
    fontsize=17,
    fontweight='bold',
    y=.895
)

rodape(fig)

fig.tight_layout(
    rect=[0, .06, 1, .85],
    w_pad=2.5
)

fig.savefig(PASTA_SAIDA / 'painel_dengue_chik_casos_exames_4_8se.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Perfil dos casos e populações de atenção


In [ ]:
# 1. DEFINIR E FILTRAR AS VARIÁVEIS DE COMORBIDADES
comorbidades = {
    "DIABETES": "Diabetes", "HEMATOLOG": "Doença hematológica",
    "HEPATOPAT": "Doença hepática", "RENAL": "Doença renal",
    "HIPERTENSA": "Hipertensão arterial", "ACIDO_PEPT": "Doença ácido-péptica",
    "AUTO_IMUNE": "Doença autoimune"
}

# Pegar apenas as colunas que existem na base
cols_validas = [c for c in comorbidades if c in df_provaveis_filtrado.columns]

# 2. CÁLCULOS VETORIZADOS (Sem necessidade de loop for)
# Converter todas as colunas de uma vez para numérico
df_c = df_provaveis_filtrado[cols_validas].apply(pd.to_numeric, errors="coerce")

# Somar os casos (1), respostas válidas (1 ou 2) e calcular percentual
casos = (df_c == 1).sum()
validas = df_c.isin([1, 2]).sum()
percentuais = (casos / validas) * 100

# Criar o DataFrame final e ordenar
df_comorbidades = pd.DataFrame({
    "Comorbidade": [comorbidades[c] for c in cols_validas],
    "Casos": casos.values,
    "Percentual": percentuais.values
}).sort_values(by="Casos").reset_index(drop=True)

# 3. CRIAR O GRÁFICO
fig, ax = plt.subplots(figsize=(13, 7))

barras = ax.barh(df_comorbidades["Comorbidade"], df_comorbidades["Casos"], color="#450920", height=0.68)

# 4. ADICIONAR RÓTULOS (Lógica de texto simplificada)
limite_x = df_comorbidades["Casos"].max()
ax.set_xlim(0, limite_x * 1.25 if limite_x > 0 else 1)
deslocamento = max(limite_x * 0.015, 0.2)

for barra, p in zip(barras, df_comorbidades["Percentual"]):
    c = int(barra.get_width())
    
    # Formatação padrão brasileiro (ex: 1.234 e 12,3%)
    txt_casos = f"{c:,}".replace(",", ".")
    txt_perc = f"({p:.1f}%)".replace(".", ",") if pd.notna(p) else ""
    
    ax.text(c + deslocamento, barra.get_y() + barra.get_height() / 2,
            f"{txt_casos} {txt_perc}".strip(),
            va="center", ha="left", fontsize=10, color="black", fontweight= 'bold')

# 5. ESTILIZAÇÃO
ax.set_title(f"PERFIL DE COMORBIDADES ENTRE OS CASOS PROVÁVEIS DE CHIKUNGUNYA - BRASIL - 2026".upper(), 
             fontsize=16, fontweight="bold", loc="center", pad=15)

ax.set_xlabel("Casos Prováveis", fontsize=12, labelpad=15)

# Remover bordas usando um loop rápido
for espinha in ["top", "right", "left"]: 
    ax.spines[espinha].set_visible(True)

ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="x", labelsize=12)
ax.grid(axis="x", linestyle= '--', alpha=0.5)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

# Fonte e nota metodológica
fig.text(0.149, 0.020, 
         f"Fonte: SINAN | Dados até: {data_analise} | Percentuais calculados entre os registros válidos de cada comorbidade.",
         fontsize=9, ha="left", va="bottom")

plt.tight_layout(rect=[0, 0.07, 1, 1])

# 6. SALVAR
salvar_pasta = os.path.join(local_pasta, "chik_brasil_comorbidades.png")
plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# PIRAMIDE ETARIA E SEXO

with warnings.catch_warnings():

    warnings.simplefilter(
        "ignore",
        urllib3.exceptions.InsecureRequestWarning
    )

# 1. Função para extrair a idade em anos a partir do formato padrão do SINAN (NU_IDADE_N)
def extrair_idade_sinan(codigo):
    try:
        codigo = str(int(codigo))
        if len(codigo) == 4:
            tp = codigo[0]
            val = int(codigo[1:])
            if tp == '4':    # Anos
                return val
            elif tp in ['1', '2', '3']: # Horas, Dias, Meses (< 1 ano)
                return 0
    except:
        return np.nan
    return np.nan

# Aplicar o cálculo da idade (caso seu DF tenha a coluna NU_IDADE_N do SINAN)
# Se você já tiver uma coluna de idade pronta, pode pular esta etapa
df_provaveis['IDADE_ANOS'] = df_provaveis['NU_IDADE_N'].apply(extrair_idade_sinan)

# 2. Criar as faixas etárias
bins = [-1, 4, 9, 14, 19, 29, 39, 49, 59, 69, 79, 120]
labels = [
    '0-4', '5-9', '10-14', '15-19', '20-29', 
    '30-39', '40-49', '50-59', '60-69', '70-79', '80+'
]

df_provaveis['FAIXA_ETARIA'] = pd.cut(df_provaveis['IDADE_ANOS'], bins=bins, labels=labels)

# 3. Filtrar e agrupar por Sexo e Faixa Etária
# Limpa sexos indefinidos ou nulos
df_piramide = df_provaveis[df_provaveis['CS_SEXO'].isin(['M', 'F'])].copy()

agrupado = df_piramide.groupby(['FAIXA_ETARIA', 'CS_SEXO'], observed=False).size().unstack(fill_value=0)

# Garantir que Masculino fique com valores negativos para espelhar o gráfico
agrupado['M_neg'] = -agrupado['M']

# 4. Plotar a Pirâmide Etária
fig, ax = plt.subplots(figsize=(10, 6))

y = np.arange(len(labels))
height = 0.8

# Barras para Homens e Mulheres
rects_m = ax.barh(y, agrupado['M_neg'], height, label='Masculino', color='#415a77') ,
rects_f = ax.barh(y, agrupado['F'], height, label='Feminino', color='#a53860')

# Ajustar eixos e rótulos
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel('Casos Prováveis', labelpad=10)
ax.set_ylabel('Faixa Etária (anos)', labelpad=10)
ax.set_title('PIRÂMIDE ETÁRIA DOS CASOS PROVÁVEIS DE CHIKUNGUNYA - BRASIL - 2026', fontsize=14, fontweight='bold', pad=15)

# Formatar os números do eixo X para exibirem valores positivos em ambos os lados
ticks = ax.get_xticks()
ax.set_xticks(ticks)
ax.set_xticklabels([f'{abs(int(x)):,}' for x in ticks])

# Adicionar linha central
ax.axvline(0, color='black', linewidth=0.8)

# Adicionar rótulos com o valor total em cima das barras
for i in range(len(labels)):
    val_m = agrupado['M'].iloc[i]
    val_f = agrupado['F'].iloc[i]
    if val_m > 0:
        ax.text(agrupado['M_neg'].iloc[i] - (max(agrupado['F']) * 0.02), i, f'{val_m}', va='center', ha='right', fontsize=10)
    if val_f > 0:
        ax.text(val_f + (max(agrupado['F']) * 0.02), i, f'{val_f}', va='center', ha='left', fontsize=10)

ax.legend(loc='upper right')
ax.grid(axis='x', linestyle='--', alpha=0.5)
ax.grid(axis='y', linestyle='--', alpha=0.5)

salvar_pasta = os.path.join(local_pasta, "chik_brasil_provaveis_idade_sexo.png")
plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# PERFIL DE SINAIS E SINTOMAS DOS CASOS PROVÁVEIS DE CHIKUNGUNYA - BRASIL

# Variáveis de sinais e sintomas do SINAN
sintomas = {
    "FEBRE": "Febre", "MIALGIA": "Mialgia", "CEFALEIA": "Cefaleia", 
    "EXANTEMA": "Exantema", "VOMITO": "Vômito", "NAUSEA": "Náusea",
    "DOR_COSTAS": "Dor nas costas", "CONJUNTVIT": "Conjuntivite",
    "ARTRITE": "Artrite", "ARTRALGIA": "Artralgia", "PETEQUIA_N": "Petéquias",
    "LEUCOPENIA": "Leucopenia", "DOR_RETRO": "Dor retro-orbitária"
}

# Mantém apenas variáveis existentes na base
sintomas = {
    coluna: rotulo
    for coluna, rotulo in sintomas.items()
    if coluna in df_provaveis.columns
}

# Total de casos prováveis
total_provaveis = len(df_provaveis)

# Calcula frequência e percentual
dados_sintomas = []

for coluna, rotulo in sintomas.items():

    valores = pd.to_numeric(df_provaveis[coluna], errors="coerce")

    # SINAN: 1 = Sim
    n = (valores == 1).sum()
    percentual = (n / total_provaveis) * 100

    dados_sintomas.append({"Sintoma": rotulo, "n": n,
        "Percentual": percentual
    })

df_sintomas = pd.DataFrame(dados_sintomas)

# Ordena do mais frequente para o menos frequente
df_sintomas = df_sintomas.sort_values("Percentual", ascending=True)

# GRÁFICO

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(df_sintomas["Sintoma"], df_sintomas["Percentual"], color="#450920")

# Rótulos n (%)
for bar, n, percentual in zip(
    bars,
    df_sintomas["n"],
    df_sintomas["Percentual"]
):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{n:,.0f} ({percentual:.1f}%)".replace(",", "."),
        va="center",
        fontsize=10,
        fontweight= 'bold'
    )

ax.set_title("Perfil de sinais e sintomas dos casos prováveis de chikungunya - Brasil - 2026".upper(),
    fontsize=16, fontweight="bold", pad=15)

ax.set_xlabel("Casos prováveis (x1.000)", fontsize=12, labelpad= 15)
ax.set_ylabel("")
ax.set_xlim(0, max(df_sintomas["Percentual"].max() * 1.25, 5))
ax.grid(axis="x", alpha=0.5, linestyle= '--')
ax.grid(axis="y", alpha=0.5, linestyle= '--')

ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
ax.spines["left"].set_visible(True)

plt.tight_layout(rect=[0, 0.04, 1, 1])

plt.savefig(os.path.join(local_pasta, "chik_brasil_provaveis_sintomas.png"),
    dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# CONFIGURAÇÕES E UTILITÁRIOS
col_uf, col_se, col_hospital = 'SG_UF', 'SE', 'HOSPITALIZ'
se_inicial, se_final = 27, se_temporada_atual
ordem_regioes = ['Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste']

def tratar_serie(serie):
    return serie.astype(str).str.replace(r'\.0$', '', regex=True).str.strip().str.upper()

def preparar_hospitalizacoes_uf(df, ano):
    df_ano = df.copy()
    df_ano[col_se] = pd.to_numeric(df_ano[col_se], errors='coerce')
    df_ano = df_ano[df_ano[col_se].between(se_inicial, se_final, inclusive='both')].copy()

    vals = tratar_serie(df_ano[col_hospital])
    df_ano['Estado'] = tratar_serie(df_ano[col_uf]).replace(mapa_ibge_sigla)
    df_ano['Hospitalizado'] = vals.isin(['1', 'SIM', 'S'])
    df_ano['Hospitalizacao_informada'] = vals.isin(['1', '2', 'SIM', 'NÃO', 'NAO', 'S', 'N'])

    resumo = df_ano.groupby('Estado', as_index=False).agg(
        Provaveis=('Estado', 'size'),
        Hospitalizacao_informada=('Hospitalizacao_informada', 'sum'),
        Hospitalizacoes=('Hospitalizado', 'sum')
    )
    resumo['Completude'] = (resumo['Hospitalizacao_informada'] / resumo['Provaveis']) * 100
    resumo['Prop_hospitalizados'] = (resumo['Hospitalizacoes'] / resumo['Provaveis']) * 100

    return resumo.rename(columns={c: f"{c}_{ano}" for c in resumo.columns if c != 'Estado'})

# PROCESSAMENTO E CONSOLIDAÇÃO DE DADOS

df_hosp_2025 = preparar_hospitalizacoes_uf(df_provaveis_2025, 2025)
df_hosp_2026 = preparar_hospitalizacoes_uf(df_provaveis, 2026)

df_hosp = df_pop.merge(df_hosp_2025, on='Estado', how='left').merge(df_hosp_2026, on='Estado', how='left')
cols_num = [f"{c}_{a}" for a in [2025, 2026] for c in ['Provaveis', 'Hospitalizacao_informada', 'Hospitalizacoes', 'Completude', 'Prop_hospitalizados']]
df_hosp[cols_num] = df_hosp[cols_num].fillna(0)

# Variação Percentual
h25, h26 = df_hosp['Hospitalizacoes_2025'], df_hosp['Hospitalizacoes_2026']
df_hosp['Var_hospitalizacoes'] = np.where(h25 > 0, ((h26 - h25) / h25) * 100, np.nan)

# MONTAGEM DA HIERARQUIA (BRASIL > REGIÃO > UF)
def calc_metricas(df, nome, is_reg):
    p25, p26 = df['Provaveis_2025'].sum(), df['Provaveis_2026'].sum()
    h25_s, h26_s = df['Hospitalizacoes_2025'].sum(), df['Hospitalizacoes_2026'].sum()
    return {
        'Local': nome, 'is_regiao': is_reg,
        'Hospitalizacoes_2026': h26_s,
        'Prop_hospitalizados_2026': (h26_s / p26 * 100) if p26 > 0 else 0,
        'Hospitalizacoes_2025': h25_s,
        'Prop_hospitalizados_2025': (h25_s / p25 * 100) if p25 > 0 else 0,
        'Var_hospitalizacoes': ((h26_s - h25_s) / h25_s * 100) if h25_s > 0 else np.nan
    }

linhas = [calc_metricas(df_hosp, 'BRASIL', True)]

for reg in ordem_regioes:
    df_reg = df_hosp[df_hosp['Regiao'] == reg]
    if df_reg.empty: continue
    linhas.append(calc_metricas(df_reg, reg, True))
    for _, row in df_reg.iterrows():
        l_dict = row.to_dict()
        l_dict.update({'Local': f"  {row['Nome_UF']}", 'is_regiao': False})
        linhas.append(l_dict)

df_report = pd.DataFrame(linhas)

# FORMATAÇÃO E TABELA VISUAL

fmt_int = lambda x: f"{int(x):,}".replace(',', '.')
fmt_dec = lambda x: f"{x:,.1f}".replace(',', 'X').replace('.', ',').replace('X', '.')

def fmt_var(r):
    v, h25, h26 = r['Var_hospitalizacoes'], r['Hospitalizacoes_2025'], r['Hospitalizacoes_2026']
    if h25 == 0: return 'NOVO' if h26 > 0 else '—'
    if pd.isna(v): return '—'
    return f"▲ {fmt_dec(abs(v))}" if v > 0 else (f"▼ {fmt_dec(abs(v))}" if v < 0 else '0,0')

celulas = [['REGIÃO/UF', 'Hospitaliz_2026', 'Proporção (%)', 'Hospitaliz_2025', 'Proporção (%)', 'Variação (%)']]
for _, r in df_report.iterrows():
    celulas.append([
        r['Local'], fmt_int(r['Hospitalizacoes_2026']), fmt_dec(r['Prop_hospitalizados_2026']),
        fmt_int(r['Hospitalizacoes_2025']), fmt_dec(r['Prop_hospitalizados_2025']), fmt_var(r)
    ])

# Renderização com Matplotlib
fig, ax = plt.subplots(figsize=(11, len(celulas) * 0.38 + 2))
ax.axis('tight'); ax.axis('off')

tab = ax.table(cellText=celulas, cellLoc='center', loc='center', colWidths=[0.30] + [0.15]*5)
tab.auto_set_font_size(False); tab.set_fontsize(10.5); tab.scale(1.10, 1.50)

for (r, c), cell in tab.get_celld().items():
    cell.set_linewidth(0.6)
    cell.set_edgecolor('#a0a0a0')
    if r == 0:
        cell.set_facecolor('#cccccc')
        cell.set_text_props(weight='bold', color='black')
    else:
        info = df_report.iloc[r - 1]
        if c == 0: cell.set_text_props(ha='left')
        cell.set_facecolor('#d9d9d9' if info['is_regiao'] else '#ffffff')
        if info['is_regiao']: cell.set_text_props(weight='bold')
        
        # Destaque para variação de hospitalizações
        if c == 5:
            h25_v, h26_v, var_v = info['Hospitalizacoes_2025'], info['Hospitalizacoes_2026'], info['Var_hospitalizacoes']
            if h26_v > h25_v and (pd.isna(var_v) or var_v > 0):
                cell.set_facecolor('#fce4d6')
                cell.set_text_props(color='#c00000', weight='bold')
            elif not pd.isna(var_v) and var_v < 0:
                cell.set_text_props(color='#548235', weight='bold')

# Rodapé e Título
rodape = (
    "Fonte: SINAN (dados da coluna Hospitaliz*).\n"
    "Nota 1: A proporção corresponde ao percentual de casos prováveis com registro de hospitalização.\n"
    f"Nota 2: Comparação entre as semanas epidemiológicas {se_inicial} e {se_temporada_atual} de 2025 e 2026.\n"
    "Nota 3: “NOVO” indica ocorrência de hospitalização em 2026 quando não havia hospitalizações em 2025. “—” indica variação não aplicável."
)

plt.figtext(0.065, 0.24, rodape, ha='left', va='top', fontsize=8.5, style='italic')
plt.title('Hospitalizações por chikungunya segundo Região e UF - Brasil - 2025 e 2026'.upper(), fontsize=15, weight='bold', y=0.88)
plt.subplots_adjust(top=0.9, bottom=0.15)

# Salvar e Exibir
plt.savefig(os.path.join(local_pasta, 'chik_brasil_hospitalizacoes_comparativo_uf.png'), bbox_inches='tight', dpi=300, facecolor='white')
plt.show()

In [ ]:
# VARIÁVEIS E FUNÇÕES AUXILIARES
col_idade, col_sexo = 'NU_IDADE_N', 'CS_SEXO'

def converter_idade_sinan(serie):
    """Converte o código NU_IDADE_N do SINAN para idade em anos."""
    cod = pd.to_numeric(serie, errors='coerce')
    unidade, valor = cod // 1000, cod % 1000
    
    condicoes = [unidade == 1, unidade == 2, unidade == 3, unidade == 4]
    escolhas = [valor / 8766.0, valor / 365.25, valor / 12.0, valor]
    
    return pd.Series(np.select(condicoes, escolhas, default=np.nan), index=serie.index)

def hospitalizacao_sim(serie):
    """Identifica hospitalização registrada como Sim (1, SIM, S)."""
    vals = serie.astype(str).str.replace(r'\.0$', '', regex=True).str.strip().str.upper()
    return vals.isin(['1', 'SIM', 'S'])

def preparar_boxplot_hospitalizacao(df, ano):
    """Filtra e prepara a base de dados de hospitalizações por idade e sexo."""
    dados = df.copy()
    dados[col_se] = pd.to_numeric(dados[col_se], errors='coerce')
    
    # Filtro de semanas epidemiológicas e status de hospitalização
    m_se = dados[col_se].between(se_inicial, se_final, inclusive='both')
    m_hosp = hospitalizacao_sim(dados[col_hospital])
    dados = dados.loc[m_se & m_hosp].copy()
    
    # Tratamento de Sexo e Idade
    sexo_map = {'F': 'Feminino', '2': 'Feminino', 'M': 'Masculino', '1': 'Masculino'}
    dados['Sexo'] = dados[col_sexo].astype(str).str.strip().str.upper().map(sexo_map)
    dados['Idade'] = converter_idade_sinan(dados[col_idade])
    dados['Ano'] = str(ano)
    
    # Filtro final de consistência de dados
    val_sexo = dados['Sexo'].isin(['Feminino', 'Masculino'])
    val_idade = dados['Idade'].between(0, 120, inclusive='both')
    
    return dados.loc[val_sexo & val_idade, ['Sexo', 'Idade', 'Ano']]

# PREPARAÇÃO DOS DADOS
df_box_hospitalizacao = pd.concat([
    preparar_boxplot_hospitalizacao(df_provaveis_2025, 2025),
    preparar_boxplot_hospitalizacao(df_provaveis, 2026)
], ignore_index=True)

# Cálculo do N amostral
n_box = df_box_hospitalizacao.groupby(['Sexo', 'Ano']).size().reset_index(name='n')
print(n_box)

# VISUALIZAÇÃO (SEABORN / MATPLOTLIB)
cores_anos = {'2025': '#f6bd60', '2026': '#e76f51'}
ordem_sexo, ordem_ano = ['Feminino', 'Masculino'], ['2025', '2026']

fig, ax = plt.subplots(figsize=(10, 7))

flier_style = {'marker': 'o', 'markersize': 4, 'markerfacecolor': '#000000', 'markeredgecolor': '#000000', 'alpha': 0.75}
median_style = {'color': '#000000', 'linewidth': 1.5}

sns.boxplot(
    data=df_box_hospitalizacao, x='Sexo', y='Idade', hue='Ano',
    order=ordem_sexo, hue_order=ordem_ano, palette=cores_anos,
    width=0.65, linewidth=1.1, showfliers=True,
    flierprops=flier_style, medianprops=median_style, ax=ax
)

# Estilização do Gráfico
ax.set_title(
    f'Distribuição da idade dos casos hospitalizados por chikungunya segundo sexo e ano\nBrasil — SE 1 a {se_analise} de 2025 e 2026',
    fontsize=14, weight='bold', pad=15
)
ax.set_xlabel('Sexo', fontsize=12, weight='bold')
ax.set_ylabel('Idade (anos)', fontsize=12, weight='bold')
ax.tick_params(axis='both', labelsize=11)
ax.grid(axis='y', color='#d9d9d9', linestyle= '--', linewidth=0.7, alpha=0.8)
ax.grid(axis='x', visible=False)

ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1))

# Nota de Rodapé
texto_n = ' | '.join(f"{row.Sexo}, {row.Ano}: n = {row.n}" for row in n_box.itertuples())
rodape = f"{texto_n}\nNota: Foram considerados somente registros hospitalizados com sexo e idade válidos. | Fonte: SINAN"

fig.text(0.07, 0.08, rodape, ha='left', va='bottom', fontsize=8.5, style='italic')

plt.tight_layout(rect=[0, 0.12, 0.88, 1])

# Salvar e Exibir
salvar_boxplot_hosp = os.path.join(local_pasta, 'chik_brasil_hospitalizacoes_idade_sexo_2025_2026.png')
plt.savefig(salvar_boxplot_hosp, bbox_inches='tight', dpi=300, facecolor='white')
plt.show()

### Análises Indígenas - Brasil

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS PROVÁVEIS

df_indigena = df_provaveis[df_provaveis['CS_RACA'] == 5]

# 1. Tratamento e LIMPEZA dos códigos IBGE para o cruzamento
# Transformar o código IBGE de 7 dígitos do arquivo em 6 dígitos (como no SINAN) e remover possíveis decimais/espaços
df_municipios['COD_IBGE_6'] = (df_municipios['COD_IBGE']
                               .astype(str)
                               .str.replace('.0', '', regex=False)
                               .str.strip()
                               .str[:6])

# Garantir que a coluna do SINAN também seja tratada como texto limpo
df_indigena['ID_MN_RESI'] = (df_provaveis['ID_MN_RESI']
                              .astype(str)
                              .str.replace('.0', '', regex=False)
                              .str.strip()
                              .str[:6]) # Pega os 6 primeiros dígitos para parear exatamente

# 2. Cruzar (Merge) as duas bases
# Trazemos as informações do df_municipios para o df_provaveis baseando-se no código de 6 dígitos
df_completo = pd.merge(df_indigena, df_municipios, left_on='ID_MN_RESI', right_on='COD_IBGE_6', how='left')

# 3. Contar os casos por município
# Contamos a frequência de cada nome na coluna 'MUNICÍPIO - IBGE - UF'
casos_por_municipio = df_completo['MUNICÍPIO - IBGE - UF'].value_counts().reset_index()
casos_por_municipio.columns = ['Município', 'Total']

# 4. Selecionar os 10 municípios com mais casos para o gráfico não ficar ilegível
top_municipios = casos_por_municipio.head(8)

# 5. Configurar o estilo visual e Plotar
grafico_municipio = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=top_municipios, x='Município', y='Total', color="#3a5a40", ax=ax)

# 6. Trava de segurança para os rótulos de dados
if len(ax.containers) > 0:
    ax.bar_label(ax.containers[0], fmt='%d', padding=3, fontsize=10)
else:
    print("Aviso: Nenhum dado foi plotado. Verifique o cruzamento dos dados ('df_completo').")

# Formatações do Gráfico
ax.set_title('Municípios com maior frequência de casos prováveis de chikungunya - População Indígena - Brasil - 2026'.upper(), 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município-UF', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
#ax.set_ylim (0, 10000)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)

# Informações rodapé 
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise} | '
                f'Observação: a quantificação dos casos se deu a partir da variável município de residência (ID_MN_RESI) e filtro para população indígena (CS_RACA = 5)')
ax_rodape.text(0.0, -0.5, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_municipios_indigenas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS PROVÁVEIS

df_indigena = temporada_2026[temporada_2026['CS_RACA'] == 5]

# 1. Tratamento e LIMPEZA dos códigos IBGE para o cruzamento
# Transformar o código IBGE de 7 dígitos do arquivo em 6 dígitos (como no SINAN) e remover possíveis decimais/espaços
df_municipios['COD_IBGE_6'] = (df_municipios['COD_IBGE']
                               .astype(str)
                               .str.replace('.0', '', regex=False)
                               .str.strip()
                               .str[:6])

# Garantir que a coluna do SINAN também seja tratada como texto limpo
df_indigena['ID_MN_RESI'] = (temporada_2026['ID_MN_RESI']
                              .astype(str)
                              .str.replace('.0', '', regex=False)
                              .str.strip()
                              .str[:6]) # Pega os 6 primeiros dígitos para parear exatamente

# 2. Cruzar (Merge) as duas bases
# Trazemos as informações do df_municipios para o df_provaveis baseando-se no código de 6 dígitos
df_completo = pd.merge(df_indigena, df_municipios, left_on='ID_MN_RESI', right_on='COD_IBGE_6', how='left')

# 3. Contar os casos por município
# Contamos a frequência de cada nome na coluna 'MUNICÍPIO - IBGE - UF'
casos_por_municipio = df_completo['MUNICÍPIO - IBGE - UF'].value_counts().reset_index()
casos_por_municipio.columns = ['Município', 'Total']

# 4. Selecionar os 10 municípios com mais casos para o gráfico não ficar ilegível
top_municipios = casos_por_municipio.head(8)

# 5. Configurar o estilo visual e Plotar
grafico_municipio = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=top_municipios, x='Município', y='Total', color="#3a5a40", ax=ax)

# 6. Trava de segurança para os rótulos de dados
if len(ax.containers) > 0:
    ax.bar_label(ax.containers[0], fmt='%d', padding=3, fontsize=10)
else:
    print("Aviso: Nenhum dado foi plotado. Verifique o cruzamento dos dados ('df_completo').")

# Formatações do Gráfico
ax.set_title(f'Casos prováveis de chikungunya - População Indígena - Brasil - SE27 a SE{se_analise} - 2026'.upper(), 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município-UF', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
#ax.set_ylim (0, 10000)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)

# Informações rodapé 
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise} | '
                f'Observação: a quantificação dos casos se deu a partir da variável município de residência (ID_MN_RESI) e filtro para população indígena (CS_RACA = 5)')
ax_rodape.text(0.0, -0.5, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_municipios_indigenas_temporada.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# MAPA COROPLÉTICO DOS CASOS PROVÁVEIS POR MUNICÍPIO

from geobr import read_municipality
from geobr import read_state

with warnings.catch_warnings():

    warnings.simplefilter(
        "ignore",
        urllib3.exceptions.InsecureRequestWarning
    )

casos_indigenas = temporada_2026[temporada_2026['CS_RACA'] == 5]

# 1. Carregar a malha de municípios do Brasil direto do GitHub do geobr (sem precisar instalar a biblioteca)
mapa_municipios = read_municipality(code_muni='all', year=2025)
mapa_limites_estados = read_state(code_state="all", year=2020)

# 2. Ajustar os códigos da malha municipal para seis dígitos
mapa_municipios['code_muni_6'] = mapa_municipios['code_muni'].astype(str).str[:6]

# 3. Preparar a sua base do SINAN
df_mun_mapa = (casos_indigenas["ID_MN_RESI"].dropna().astype(str).str.replace(".0", "", regex=False)
               .value_counts().rename_axis("code_muni_6").reset_index(name="Total"))

# Mecanismo de garantia para que a base do SINAN tenha seis dígitos
df_mun_mapa["code_muni_6"] = (
    df_mun_mapa["code_muni_6"]
    .str.zfill(6))

# 4. Cruzar os dados com o mapa
mapa_casos = mapa_municipios.merge(df_mun_mapa, on='code_muni_6', how='left')
mapa_casos['Total'] = mapa_casos['Total'].fillna(0).astype(int)

# CRIAR AS FAIXAS DE CORES

cortes = [-1, 0, 50, 100, 300, 500, float('inf')]

rotulos = [
    'Sem casos',
    '1 - 50',
    '51 - 100',
    '101 - 300',
    '301 - 500',
    '>500'
]

mapa_casos['Categoria'] = pd.cut(mapa_casos['Total'], bins=cortes, labels=rotulos)

dicionario_cores = {
    'Sem casos': '#f4f4f4',
    '1 - 50': '#fdf0d5',
    '51 - 100': '#fdd49e',
    '101 - 300': '#fc8d59',
    '301 - 500': '#ef6548',
    '>500': '#7f0000'
}

mapa_casos['Cor_Estado'] = mapa_casos['Categoria'].map(dicionario_cores)

# PLOTAR O MAPA
fig, ax = plt.subplots(figsize=(12, 12))

# Reduzir o linewidth para 0.1 ou 0.2 (pois o Brasil tem +5.500 municípios e bordas grossas deixam o mapa preto)
mapa_casos.plot(
    color=mapa_casos['Cor_Estado'], 
    linewidth=0.1, 
    edgecolor='0.6', 
    ax=ax
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.5,
    edgecolor="black",
    ax=ax,
)

# CRIAR A LEGENDA CUSTOMIZADA
lista_legendas = [
    mpatches.Patch(
        color=dicionario_cores[rotulo],
        label=rotulo,
    )
    for rotulo in rotulos
]

ax.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=11,
    handles=lista_legendas,
    fontsize=14,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.09, 0.2),
    frameon=False,
)

# Estilização Final
ax.set_title(f'Casos prováveis de Chikungunya - SE27 a SE{se_temporada_atual}\nmunicípios com população indígena - 2026'.upper(), 
             fontsize=16, fontweight='bold', pad=8)
ax.axis('off')

fig.text(
    0.41,
    0.15,
    f"Fonte: SINAN | Dados até: {data_analise}",
    fontsize=9,
    color="black",
    ha="left",
)

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_mapa_municipio_indigena.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from geobr import read_municipality, read_state

with warnings.catch_warnings():
    warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)


# --- 1. PREPARAÇÃO DA BASE DE POPULAÇÃO MUNICIPAL ---
df_populacao = df_municipios.copy()
df_populacao["code_muni_6"] = (
    df_populacao["COD_IBGE"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
    .str.zfill(6)
    .str[:6]
)

# Garantir que a população é numérica
df_populacao["POPULACAO"] = (
    df_populacao["POPULACAO"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_populacao["POPULACAO"] = pd.to_numeric(df_populacao["POPULACAO"], errors="coerce")


# --- 2. CARREGAR MALHA MUNICIPAL E ESTADUAL (geobr) ---
mapa_limites_estados = read_state(code_state="all", year=2020)
mapa_municipios = read_municipality(code_muni="all", year=2020)
mapa_municipios["code_muni_6"] = mapa_municipios["code_muni"].astype(str).str[:6]


# --- 3. FUNÇÃO AUXILIAR DE PROCESSAMENTO MUNICIPAL ---
def processar_incidencia_muni(df_sinan, df_pop, mapa_muni):
    df_casos = (
        df_sinan["ID_MN_RESI"]
        .dropna()
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(6)
        .str[:6]
        .value_counts()
        .rename_axis("code_muni_6")
        .reset_index(name="Total")
    )

    mapa = mapa_muni.merge(df_casos, on="code_muni_6", how="left")
    mapa = mapa.merge(df_pop[["code_muni_6", "POPULACAO"]], on="code_muni_6", how="left")

    mapa["Total"] = mapa["Total"].fillna(0).astype(int)
    mapa["POPULACAO"] = mapa["POPULACAO"].fillna(1)  # Evita divisão por zero

    mapa["Incidencia"] = (mapa["Total"] / mapa["POPULACAO"]) * 100000
    return mapa


# Processar apenas o dataset recente (SE 27 em diante)
mapa_8semanas = processar_incidencia_muni(casos_indigenas, df_populacao, mapa_municipios)


# --- 4. DEFINIÇÃO DE FAIXAS E CORES ---
cortes_8sem = [-1, 5, 10, 20, 50, 100, float("inf")]

rotulos_8sem = [
    "Sem Casos",
    "0,1 - 10",
    "10,1 - 20",
    "20,1 - 50",
    "50,1 - 100",
    "Acima de 100",
]

dicionario_cores = {
    "0": "#edede9",
    "1": "#fdd49e",
    "2": "#fc8d59",
    "3": "#ef6548",
    "4": "#b30000",
    "5": "#7f0000",
}

# Categorização
mapa_8semanas["Categoria"] = pd.cut(
    mapa_8semanas["Incidencia"], bins=cortes_8sem, labels=rotulos_8sem
)
mapa_8semanas["Cor"] = mapa_8semanas["Categoria"].map(
    dict(zip(rotulos_8sem, dicionario_cores.values()))
)

# --- 5. CONSTRUÇÃO DA FIGURA (MAPA ÚNICO - ALTA RESOLUÇÃO) ---
fig, ax = plt.subplots(1, 1, figsize=(14, 12))

mapa_8semanas.plot(
    color=mapa_8semanas["Cor"],
    linewidth=0.1,
    edgecolor="0.7",
    ax=ax,
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.8,
    edgecolor="black",
    ax=ax,
)

ax.axis("off")

legendas_8sem = [
    mpatches.Patch(color=dicionario_cores[str(i)], label=rotulos_8sem[i])
    for i in range(len(rotulos_8sem))
]
ax.legend(
    #title="Incidência por 100 mil hab.",
    title_fontsize=11,
    handles=legendas_8sem,
    fontsize=14,
    ncols=1,
    loc="lower left",
    bbox_to_anchor=(0.09, 0.2),
    frameon=False,
)

# --- TÍTULO GERAL E RODAPÉ ---
plt.suptitle(
    f"INCIDÊNCIA DE CHIKUNGUNYA - BRASIL - SE 27 a SE {se_temporada_atual}\nmunicípios com população indígena - 2026".upper(),
    fontsize=18,
    fontweight="bold",
    y=0.90,
)

fig.text(
    0.41,
    0.07,
    f"Fonte: SINAN | Dados até: {data_analise}",
    fontsize=9,
    color="black",
    ha="left",
)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])

salvar_pasta = os.path.join(local_pasta, "chik_brasil_incidencia_municipio_indigenas_2026.png")
plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

## Síntese e relatório epidemiológico


In [ ]:
# SÍNTESE EPIDEMIOLÓGICA AUTOMÁTICA - BRASIL

# 1. FUNÇÕES AUXILIARES ENXUTAS
def fmt_int(valor):
    try: return f"{int(round(valor)):,}".replace(",", ".")
    except: return "0"
def fmt_dec(valor, casas=1):
    try: return f"{float(valor):,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except: return "0,0"

# 2. AJUSTAR SEMANAS EPIDEMIOLÓGICAS (SINAN E GAL)
for df in [df_notificados, df_provaveis, df_gal_final, df_gal_dengue]:
    if "SE" in df.columns:
        df["SE"] = pd.to_numeric(df["SE"], errors="coerce")

# SINAN: Captura a semana máxima (incompleta) e recua 1 para definir a semana atual da análise
#se_atual = int(df_provaveis["SE"].dropna().max())
#se_anterior, inicio_8se = se_atual - 1, max(1, se_atual - 7)
#inicio_8se_anterior, fim_8se_anterior = max(1, inicio_8se - 8), inicio_8se - 1

# GAL: Aplica a mesma lógica de recuar 1 semana para garantir o período completo
#se_gal_chik_max, se_gal_dengue_max = int(df_gal_final["SE"].dropna().max()), int(df_gal_dengue["SE"].dropna().max())
#se_gal_comum = min(se_gal_chik_max, se_gal_dengue_max) - 1
#inicio_gal_8se = max(1, se_gal_comum - 7)

# 3. CASOS PROVÁVEIS E VARIAÇÕES
total_provaveis = len(df_provaveis)
casos_temporada = len(temporada_2026)
casos_se_atual = (df_provaveis["SE"] == se_temporada_atual).sum()
perc_provaveis_temporada = (casos_temporada / total_provaveis * 100)

#variacao_semanal = ((casos_se_atual - casos_se_anterior) / casos_se_anterior * 100) if casos_se_anterior > 0 else np.nan
#casos_8se = df_provaveis["SE"].between(inicio_8se, se_atual).sum()
#casos_8se_anteriores = df_provaveis["SE"].between(inicio_8se_anterior, fim_8se_anterior).sum()
#variacao_8se = ((casos_8se - casos_8se_anteriores) / casos_8se_anteriores * 100) if casos_8se_anteriores > 0 else np.nan

# 4. CASOS EM INVESTIGAÇÃO E DESCARTADOS
classificacao = pd.to_numeric(df_notificados["CLASSI_FIN"], errors="coerce")
casos_investigacao, casos_descartados = classificacao.isna().sum(), (classificacao == 5).sum()
perc_investigacao = (casos_investigacao / len(df_notificados) * 100) if len(df_notificados) > 0 else 0
perc_descartados = (casos_descartados / len(df_notificados) * 100) if len(df_notificados) > 0 else 0

# 5. PERFIL ETÁRIO E SEXO
if "IDADE_ANOS" not in df_provaveis.columns:
    def extrair_idade_sinan(c):
        try:
            c = str(int(c))
            return int(c[1:]) if len(c) == 4 and c[0] == "4" else (0 if len(c) == 4 and c[0] in ["1", "2", "3"] else np.nan)
        except: return np.nan
    df_provaveis["IDADE_ANOS"] = df_provaveis["NU_IDADE_N"].apply(extrair_idade_sinan)
bins_idade, labels_idade = [-1, 4, 9, 14, 19, 29, 39, 49, 59, 69, 79, np.inf], ["0 a 4 anos", "5 a 9 anos", "10 a 14 anos", "15 a 19 anos", "20 a 29 anos", "30 a 39 anos", "40 a 49 anos", "50 a 59 anos", "60 a 69 anos", "70 a 79 anos", "80 anos ou mais"]
freq_idade = pd.cut(df_provaveis["IDADE_ANOS"], bins=bins_idade, labels=labels_idade).value_counts()
faixa_destaque, perc_faixa = (str(freq_idade.index[0]), (freq_idade.iloc[0] / freq_idade.sum() * 100)) if not freq_idade.empty else ("Não disponível", np.nan)
freq_sexo = df_provaveis[df_provaveis["CS_SEXO"].isin(["M", "F"])]["CS_SEXO"].value_counts()
sexo_destaque, perc_sexo = ({"F": "feminino", "M": "masculino"}.get(freq_sexo.index[0], "não disponível"), (freq_sexo.iloc[0] / freq_sexo.sum() * 100)) if not freq_sexo.empty else ("não disponível", np.nan)

# 6. LABORATORIAL (GAL)
def calcular_positividade_gal(df, se_inicio, se_fim):
    temp = df[df["SE"].between(se_inicio, se_fim)].copy()
    semanal = temp.groupby("SE")[["exames_realizados", "casos_positivos"]].sum().reset_index()
    semanal["positividade"] = np.where(semanal["exames_realizados"] > 0, (semanal["casos_positivos"] / semanal["exames_realizados"]) * 100, np.nan)
    return semanal
mediana_pos_chik = calcular_positividade_gal(df_gal_final, se_inicio, se_atual)["positividade"].median()
mediana_pos_dengue = calcular_positividade_gal(df_gal_dengue, se_inicio, se_atual)["positividade"].median()
#confirmados_gal_8se = df_gal_final[df_gal_final["SE"].between(inicio_gal_8se, se_gal_comum)]["casos_positivos"].sum()
confirmados_gal = df_gal_final["casos_positivos"].sum()

# 7. MUNICÍPIOS (PROVÁVEIS, GAL, INDÍGENAS)
df_municipios_resumo = df_municipios.copy()
df_municipios_resumo["COD_IBGE_6"] = df_municipios_resumo["COD_IBGE"].astype(str).str.replace(".0", "", regex=False).str.strip().str[:6]
def cruzar_municipios(df, col_codigo, col_casos=None, limit=10):
    df["ID_TEMP"] = df[col_codigo].astype(str).str.replace(".0", "", regex=False).str.strip().str[:6]
    merged = df.merge(df_municipios_resumo[["COD_IBGE_6", "MUNICÍPIO - IBGE - UF"]], left_on="ID_TEMP", right_on="COD_IBGE_6", how="left")
    if col_casos: return merged.groupby("MUNICÍPIO - IBGE - UF", dropna=True)[col_casos].sum().sort_values(ascending=False).head(limit).reset_index(name="Casos")
    return merged["MUNICÍPIO - IBGE - UF"].value_counts().head(limit).reset_index(name="Casos").rename(columns={"index": "Município"})
top10_provaveis = cruzar_municipios(df_provaveis[df_provaveis["SE"].between(se_temporada_inicial, se_temporada_atual)].copy(), "ID_MN_RESI").rename(columns={"MUNICÍPIO - IBGE - UF": "Município"})
top10_gal = cruzar_municipios(df_gal_final_filtrado[df_gal_final_filtrado["SE"].between(se_temporada_inicial, se_temporada_atual)].copy(), "IBGE", "casos_positivos").rename(columns={"MUNICÍPIO - IBGE - UF": "Município", "Casos": "Confirmados"})
indigenas = df_provaveis[pd.to_numeric(df_provaveis["CS_RACA"], errors="coerce") == 5].copy()
top5_indigenas, total_indigenas = cruzar_municipios(indigenas, "ID_MN_RESI", limit=5).rename(columns={"MUNICÍPIO - IBGE - UF": "Município"}), len(indigenas)

# 7.1. CASOS PROVÁVEIS POR PORTE POPULACIONAL DO MUNICÍPIO DE RESIDÊNCIA
# Usa o total de df_provaveis, conforme total_provaveis, sem novo filtro de SE.
# Base municipal: estimativas populacionais do IBGE de 2025.
# Não altera df_provaveis nem df_municipios.
def calcular_casos_por_porte(df_casos, df_pop_municipal):
    colunas_necessarias = {"COD_IBGE", "POPULACAO"}
    if not colunas_necessarias.issubset(df_pop_municipal.columns):
        raise ValueError("df_municipios deve conter COD_IBGE e POPULACAO (população municipal IBGE 2025).")

    def codigo_municipal_6(serie):
        codigos = serie.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True)
        return codigos.where(codigos.str.fullmatch(r"\d{6,7}", na=False)).str[:6]

    populacao = df_pop_municipal[["COD_IBGE", "POPULACAO"]].copy()
    populacao["COD_IBGE_6_PORTE"] = codigo_municipal_6(populacao["COD_IBGE"])
    populacao["POPULACAO_PORTE"] = pd.to_numeric(populacao["POPULACAO"], errors="coerce")
    populacao = populacao.dropna(subset=["COD_IBGE_6_PORTE"])
    if populacao.groupby("COD_IBGE_6_PORTE")["POPULACAO_PORTE"].nunique(dropna=False).gt(1).any():
        raise ValueError("Há populações divergentes para o mesmo código municipal na base populacional.")
    populacao = populacao.drop_duplicates("COD_IBGE_6_PORTE")
    populacao.loc[populacao["POPULACAO_PORTE"] <= 0, "POPULACAO_PORTE"] = np.nan
    mapa_populacao = populacao.set_index("COD_IBGE_6_PORTE")["POPULACAO_PORTE"]
    pop_residencia = codigo_municipal_6(df_casos["ID_MN_RESI"]).map(mapa_populacao)
    faixas = ["Até 5 mil habitantes", "Até 20 mil habitantes", "Até 50 mil habitantes", "Até 100 mil habitantes", "Até 1 milhão de habitantes", "Mais de 1 milhão de habitantes"]
    porte = pd.cut(pop_residencia, bins=[0, 5_000, 20_000, 50_000, 100_000, 1_000_000, np.inf],
                   labels=faixas, right=True)
    tabela = porte.value_counts(sort=False).reindex(faixas, fill_value=0).rename_axis("Porte populacional").reset_index(name="Casos prováveis")
    sem_classificacao = int(porte.isna().sum())
    tabela.loc[len(tabela)] = ["Sem classificação populacional", sem_classificacao]
    total = len(df_casos)
    tabela["Percentual do total"] = tabela["Casos prováveis"] / total * 100 if total else 0.0
    if int(tabela["Casos prováveis"].sum()) != total:
        raise ValueError("A soma dos casos por porte difere do total de casos prováveis.")
    return tabela

casos_por_porte_populacional = calcular_casos_por_porte(df_provaveis, df_municipios)
texto_porte_populacional = (
    "Distribuição do total de casos prováveis por porte populacional do município de residência "
    "(população IBGE 2025): " + "; ".join(
        f"{r['Porte populacional']}: {fmt_int(r['Casos prováveis'])} casos ({fmt_dec(r['Percentual do total'])}%)"
        for _, r in casos_por_porte_populacional.iterrows()
        if r['Porte populacional'] != "Sem classificação populacional"
    ) + "."
)
casos_sem_porte = int(casos_por_porte_populacional.iloc[-1]["Casos prováveis"])
if casos_sem_porte:
    texto_porte_populacional += (
        f" Outros {fmt_int(casos_sem_porte)} casos ({fmt_dec(casos_sem_porte / total_provaveis * 100)}%) "
        "não puderam ser classificados por ausência de código municipal válido ou população disponível."
    )

# 8. INCIDÊNCIA NACIONAL E POR UF (2026 x 2025)
pop_brasil = df_pop["Populacao"].sum()
#casos_2026_ate_se = (df_provaveis["SE"] <= se_max_sinan).sum()
inc_brasil_2026 = (total_provaveis / pop_brasil) * 100000
df_base_2025 = df_provaveis_2025.copy()
df_base_2025["SE"] = pd.to_numeric(df_base_2025["SE"], errors="coerce")
casos_2025_ate_se = (df_base_2025["SE"] <= se_atual).sum()
inc_brasil_2025 = (casos_2025_ate_se / pop_brasil) * 100000
variacao_inc_nacional = ((inc_brasil_2026 - inc_brasil_2025) / inc_brasil_2025 * 100) if inc_brasil_2025 > 0 else np.nan
mapa_ibge_sigla = {"11":"RO", "12":"AC", "13":"AM", "14":"RR", "15":"PA", "16":"AP", "17":"TO", "21":"MA", "22":"PI", "23":"CE", "24":"RN", "25":"PB", "26":"PE", "27":"AL", "28":"SE", "29":"BA", "31":"MG", "32":"ES", "33":"RJ", "35":"SP", "41":"PR", "42":"SC", "43":"RS", "50":"MS", "51":"MT", "52":"GO", "53":"DF"}
tratar_uf = lambda s: s.astype(str).str.replace(".0", "", regex=False).str.strip().replace(mapa_ibge_sigla)

def calc_inc_uf(df_filtrado, col_nome):
    df_filtrado["UF_SIGLA"] = tratar_uf(df_filtrado["SG_UF"])
    return df_filtrado["UF_SIGLA"].value_counts().rename_axis("Estado").reset_index(name=col_nome)
inc_uf = df_pop.merge(calc_inc_uf(df_provaveis[df_provaveis["SE"] <= se_atual].copy(), "Casos_2026"), on="Estado", how="left").merge(calc_inc_uf(df_base_2025[df_base_2025["SE"] <= se_atual].copy(), "Casos_2025"), on="Estado", how="left").fillna(0)
inc_uf["Inc_2026"], inc_uf["Inc_2025"] = (inc_uf["Casos_2026"] / inc_uf["Populacao"]) * 100000, (inc_uf["Casos_2025"] / inc_uf["Populacao"]) * 100000
top5_incidencia_uf = inc_uf.sort_values("Inc_2026", ascending=False).head(5).copy()
inc_uf_temporada = df_pop[["Estado", "Nome_UF", "Populacao"]].merge(calc_inc_uf(temporada_2026[temporada_2026["SE"].between(se_temporada_inicial, se_temporada_atual)].copy(), "Casos_Temporada"), on="Estado", how="left").fillna(0)
inc_uf_temporada["Inc_temporada"] = (inc_uf_temporada["Casos_Temporada"] / inc_uf_temporada["Populacao"]) * 100000
top5_incidencia_uf_temporada = inc_uf_temporada.sort_values("Inc_temporada", ascending=False).head(5).reset_index(drop=True)

# 8.1. TOP 5 POSITIVIDADE POR ESTADO (GAL)
def calcular_top5_positividade_uf(df_gal, df_pop_uf, se_ini, se_fim):
    # Padroniza a UF no GAL se necessário (supondo que exista coluna SG_UF)
    temp = df_gal[df_gal["SE"].between(se_ini, se_fim)].copy()
    if "UF_SIGLA" not in temp.columns:
        temp["UF_SIGLA"] = tratar_uf(temp["UF"])
    
    # Agrega por UF
    agregado = temp.groupby("UF_SIGLA")[["exames_realizados", "casos_positivos"]].sum().reset_index()
    
    # Calcula a positividade global e filtra quem tem ao menos 10 exames
    agregado["positividade"] = np.where(
        agregado["exames_realizados"] > 0,
        (agregado["casos_positivos"] / agregado["exames_realizados"]) * 100,
        np.nan
    )
    
    # Faz merge com os nomes dos estados para ficar bonito no texto
    top5 = (
        agregado[agregado["exames_realizados"] >= 10]
        .sort_values("positividade", ascending=False)
        .head(5)
        .merge(df_pop_uf[["Estado", "Nome_UF"]], left_on="UF_SIGLA", right_on="Estado", how="left")
    )
    return top5

top5_positividade_uf= calcular_top5_positividade_uf(df_gal_final, df_pop, se_temporada_inicial, se_temporada_atual)

# 9. CONSTRUÇÃO DOS TEXTOS (PANORAMA E TERRITORIAL)
#texto_var_sem = f"aumento de {fmt_dec(abs(variacao_semanal))}%" if variacao_semanal > 0 else (f"redução de {fmt_dec(abs(variacao_semanal))}%" if variacao_semanal < 0 else "estabilidade")
#texto_var_8se = "acima" if variacao_8se > 0 else "abaixo"

destaques_panorama = [
    f"Entre as SE 01 e SE {se_temporada_atual} foram registrados {fmt_int(total_provaveis)} casos prováveis de chikungunya no Brasil.",
    f"Entre as SE 27 e SE {se_temporada_atual}, foram registrados {fmt_int(casos_temporada)} casos prováveis de chikungunya, correspondendo a {fmt_dec(perc_provaveis_temporada)}% do total de casos de 2026.",
    f"Na SE {se_temporada_atual}, foram registrados {fmt_int(casos_se_atual)} casos prováveis de chikungunya no Brasil.",
    f"A incidência para chikungunya no Brasil (SE 1 a SE {se_temporada_atual}) é de {fmt_dec(inc_brasil_2026)} casos por 100 mil habitantes.", 
    f"Em 2025, no mesmo período, a incidência foi de {fmt_dec(inc_brasil_2025)} casos por 100 mil habitantes.",
    *( [texto_diagrama_controle] if "texto_diagrama_controle" in globals() else [] ),
    f"O GAL registrou {fmt_int(confirmados_gal)} resultados laboratoriais positivos para chikungunya.",
    f"A mediana semanal de positividade para chikungunya foi de {fmt_dec(mediana_pos_chik)}%, comparada a {fmt_dec(mediana_pos_dengue)}% para dengue.",
    f"Há {fmt_int(casos_investigacao)} casos em investigação, correspondendo a {fmt_dec(perc_investigacao)}% das notificações.",
    f"Entre a SE 1 e SE {se_temporada_atual}, {fmt_int(casos_descartados)} casos foram descartados, correspondendo a {fmt_dec(perc_descartados)}% das notificações para chikungunya.",
    f"A maior concentração de casos ocorreu na faixa etária de {faixa_destaque}, representando {fmt_dec(perc_faixa)}% dos registros com idade válida.", 
    f"O sexo {sexo_destaque} correspondeu a {fmt_dec(perc_sexo)}% dos casos prováveis com sexo informado.",
]
destaques_panorama = [txt for txt in destaques_panorama if txt] # Remove strings vazias
destaques_territoriais = [
    texto_porte_populacional,
    f"Estados com maior incidência (SE {se_inicio} a SE {se_temporada_atual}): " + "; ".join([f"{r['Nome_UF']} ({fmt_dec(r['Inc_2026'])} em 2026 | {fmt_dec(r['Inc_2025'])} em 2025)" for _, r in top5_incidencia_uf.iterrows()]) + " casos por 100 mil habitantes.",
    f"Estados com maior incidência (SE {se_temporada_inicial} a SE {se_temporada_atual}): " + "; ".join([f"{r['Nome_UF']} ({fmt_dec(r['Inc_temporada'])})" for _, r in top5_incidencia_uf_temporada.iterrows()]) + " casos por 100 mil habitantes.",
    f"Estados com maior positividade laboratorial (SE {se_temporada_inicial} a SE {se_temporada_atual}): " + "; ".join([f"{r['Nome_UF']} ({fmt_dec(r['positividade'])}%)" for _, r in top5_positividade_uf.iterrows()]) + ".",
    f"Municípios com mais casos prováveis entre as SE {se_temporada_inicial} a SE {se_temporada_atual}: " + "; ".join([f"{r['Município']} ({fmt_int(r['Casos'])})" for _, r in top10_provaveis.iterrows()]) + ".",
    f"Municípios com mais casos confirmados no GAL entre as SE {se_temporada_inicial} a SE {se_temporada_atual}: " + "; ".join([f"{r['Município']} ({fmt_int(r['Confirmados'])})" for _, r in top10_gal.iterrows()]) + ".",
    f"Foram registrados {fmt_int(total_indigenas)} casos prováveis em pessoas indígenas no acumulado de 2026. Os maiores números segundo município de residência foram observados em: " + "; ".join([f"{r['Município']} ({fmt_int(r['Casos'])})" for _, r in top5_indigenas.iterrows()]) + "."
]
# 10. MOSTRAR RESULTADOS NO NOTEBOOK
print(f"PANORAMA EPIDEMIOLÓGICO - BRASIL - SE {se_temporada_atual}\n{'-' * 50}")
for texto in destaques_panorama: print(f"• {texto}")
print(f"\nDESTAQUES TERRITORIAIS\n{'-' * 50}")
for texto in destaques_territoriais: print(f"• {texto}")

In [ ]:
# RELATÓRIO EPIDEMIOLÓGICO DE CHIKUNGUNYA - BRASIL

# 1. FUNÇÕES AUXILIARES DE FORMATAÇÃO E ESTRUTURAÇÃO

def fmt_int(valor):
    """Formata valores inteiros no padrão brasileiro (ex: 12540 -> 12.540)."""
    try:
        return f"{int(round(valor)):,}".replace(",", ".")
    except:
        return "0"

def fmt_dec(valor, casas=1):
    """Formata decimais no padrão brasileiro (ex: 23.45 -> 23,5)."""
    try:
        return f"{float(valor):,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except:
        return "0,0"

# Agrupamento lógico das figuras por eixo de análise
grupos_figuras = {
    "- EVOLUÇÃO TEMPORAL E PANORAMA EPIDEMIOLÓGICO -": [
        "chik_brasil_diagrama_controle.png",
        #"chik_brasil_provaveis_confirmados_semana.png",
        #"chik_brasil_provaveis_classificacao.png",
        'chik_brasil_casos_provaveis_2025_2026.png',
        #"chik_brasil_incidencia_semanal_2025_2026.png",
    ],
    "- DISTRIBUIÇÃO ESPACIAL -": [
        "chik_brasil_casos_provaveis_painel_uf.png",
        #"chik_brasil_provaveis_mapa_municipio.png",
        #"chik_brasil_provaveis_mapa_municipio_oito_semanas.png",
        #"chik_brasil_incidencia_uf_comparativo_2026.png",
        "chik_brasil_incidencia_uf_2026.png",
        #"chik_brasil_incidencia_municipio_comparativo_2026.png",
        "chik_brasil_incidencia_municipio_2026.png",
        "chik_brasil_incidencia_regiao_saude.png",
        "chik_regioes_saude_tabela.png",
        "chik_regioes_saude_painel.png"
    ],
    #"- CONCENTRAÇÃO TERRITORIAL DOS CASOS -": [
        #"chik_brasil_casos_provaveis_painel_uf.png",
        #"chik_brasil_tabela_comparativa_uf.png"
        #"chik_brasil_municipios_sinan_gal_oito_semanas.png",
        #"chik_brasil_provaveis_municipios_comparativo_oito_semanas.png",
    #],
    "- ANÁLISE LABORATORIAL -": [
        #"chik_brasil_confirmados_sinan_gal.png",
        #"chik_brasil_confirmados_dengue_chik.png",
        #"chik_brasil_realizados_confirmados_gal_semana.png",
        #"chik_brasil_casos_positividade_se.png",
        "chik_brasil_casos_positividade_painel_uf.png",
        "chik_brasil_painel_sinan_gal_uf.png",
        #"chik_brasil_confirmados_criterio.png",
        #"painel_dengue_chik_casos_exames_4_8se.png",
    ],
    #"- PERFIL EPIDEMIOLÓGICO -": [
    #    "chik_brasil_comorbidades.png",
    #    "chik_brasil_provaveis_idade_sexo.png",
    #    "chik_brasil_provaveis_sintomas.png",
    #],
    #"- HOSPITALIZAÇÕES -": [
    #    "chik_brasil_hospitalizacoes_comparativo_uf.png",
    #    "chik_brasil_hospitalizacoes_idade_sexo_2025_2026.png",
    #],
    #"- MONITORAMENTO POPULAÇÃO INDÍGENA -": [
    #    "chik_brasil_provaveis_municipios_indigenas.png",
    #],
}

# Converter nomes dos arquivos em caminhos completos
grupos_graficos = {
    nome_grupo: [os.path.join(local_pasta, img) for img in lista_nomes]
    for nome_grupo, lista_nomes in grupos_figuras.items()
}

salvar_pdf = os.path.join(local_pasta, f"Analise_Chikungunya_Brasil_SE{se_analise}_2026.pdf")

# 2. FUNÇÃO PRINCIPAL DE GERAÇÃO DO PDF

def gerar_pdf_brasil(grupos_graficos, destaques_panorama, top5_incidencia_uf, top5_incidencia_uf_temporada, top10_provaveis, top10_gal, 
                     casos_por_porte_populacional, top5_indigenas, total_indigenas, top5_positividade_uf, nome_pdf):
    
    # Configurações iniciais da página
    c = canvas.Canvas(nome_pdf, pagesize=A4)
    largura, altura = A4
    margem_esq, margem_dir = 35, 35
    largura_util = largura - margem_esq - margem_dir
    pagina = 1

    # --- Funções internas de layout ---
    def desenhar_cabecalho(subtitulo=None):
        c.setFillColor(colors.black)
        c.setFont("Helvetica-Bold", 14)
        c.drawCentredString(largura / 2, altura - 26, "Análise Epidemiológica - Chikungunya - Brasil - 2026")
        
        if subtitulo:
            linhas_subtitulo = simpleSplit(subtitulo, "Helvetica", 8, largura_util)
            y_sub = altura - 39
            c.setFont("Helvetica", 8)
            for linha in linhas_subtitulo:
                c.drawCentredString(largura / 2, y_sub, linha)
                y_sub -= 9
                
        c.setStrokeColor(colors.black)
        c.setLineWidth(0.5)
        c.line(margem_esq, altura - 51, largura - margem_dir, altura - 51)
        c.setStrokeColor(colors.black)

    def desenhar_rodape():
        c.setStrokeColor(colors.lightgrey)
        c.setLineWidth(0.4)
        c.line(margem_esq, 28, largura - margem_dir, 28)
        c.setFillColor(colors.black)
        c.setFont("Helvetica-Oblique", 5.8)
        c.drawString(margem_esq, 17, "Coordenação-Geral de Vigilância das Arboviroses (CGARB/DEDT/SVSA/MS)")
        c.drawRightString(largura - margem_dir, 17, f"Página {pagina}")
        c.setFillColor(colors.black)

    def titulo_secao(texto, y):
        c.setFont("Helvetica-Bold", 12)
        c.drawString(margem_esq, y, texto)
        c.setStrokeColor(colors.black)
        c.setLineWidth(0.5)
        c.line(margem_esq, y - 5, largura - margem_dir, y - 5)
        c.setStrokeColor(colors.black)
        return y - 25

    def desenhar_bullet(texto, y, tamanho=9.3):
        linhas = simpleSplit(texto, "Helvetica", tamanho, largura_util - 25)
        c.setFont("Helvetica-Bold", tamanho)
        c.drawString(margem_esq + 3, y, "•")
        c.setFont("Helvetica", tamanho)
        for linha in linhas:
            c.drawString(margem_esq + 17, y, linha)
            y -= 13
        return y - 7

    def desenhar_caixa(x, y, w, h, titulo, linhas, fonte=8):
        c.setStrokeColor(colors.grey)
        c.setLineWidth(0.5)
        c.roundRect(x, y - h, w, h, 4, stroke=1, fill=0)
        
        titulo_linhas = simpleSplit(titulo, "Helvetica-Bold", 8.5, w - 16)
        yy = y - 15
        c.setFont("Helvetica-Bold", 8.5)
        for linha in titulo_linhas:
            c.drawString(x + 8, yy, linha)
            yy -= 10
            
        yy -= 2
        c.setStrokeColor(colors.lightgrey)
        c.line(x + 8, yy, x + w - 8, yy)
        yy -= 13
        
        for linha in linhas:
            for parte in simpleSplit(linha, "Helvetica", fonte, w - 18):
                c.setFont("Helvetica", fonte)
                c.drawString(x + 9, yy, parte)
                yy -= 10
            yy -= 2

    def desenhar_imagem(caminho, x, y, largura_max, altura_max):
        if not os.path.exists(caminho):
            print(f"⚠️ Figura não encontrada: {os.path.basename(caminho)}")
            return False
        try:
            with Image.open(caminho) as img:
                largura_img, altura_img = img.size
            proporcao = min(largura_max / largura_img, altura_max / altura_img)
            largura_final, altura_final = largura_img * proporcao, altura_img * proporcao
            x_final = x + (largura_max - largura_final) / 2
            y_final = y + (altura_max - altura_final) / 2
            
            c.drawImage(caminho, x_final, y_final, width=largura_final, height=altura_final, preserveAspectRatio=True)
            return True
        except Exception as erro:
            print(f"⚠️ Erro ao inserir {os.path.basename(caminho)}: {erro}")
            return False

    # PÁGINA 1: PANORAMA EPIDEMIOLÓGICO NACIONAL
    desenhar_cabecalho(subtitulo="- BULLET POINTS -")
    y = titulo_secao(f"Panorama Epidemiológico Nacional - SE {se_analise}", altura - 82)

    for texto in destaques_panorama:
        y = desenhar_bullet(texto, y)

    y -= 3
    c.setFont("Helvetica-Bold", 7)
    c.drawString(margem_esq, y, "Observação")
    
    nota = ("As comparações entre semanas epidemiológicas representam variações observadas nos registros disponíveis "
            "e não devem ser interpretadas isoladamente como mudança na transmissão, considerando a possibilidade de "
            "atraso de notificação e atualização da base. Para os indicadores laboratoriais, são consideradas as "
            "oito últimas semanas com dados simultaneamente disponíveis para dengue e chikungunya.")
    
    c.setFillColor(colors.black)
    c.setFont("Helvetica-Oblique", 7)
    for linha in simpleSplit(nota, "Helvetica-Oblique", 7, largura_util):
        c.drawString(margem_esq, y - 13, linha)
        y -= 9
        
    c.setFillColor(colors.black)
    desenhar_rodape()
    c.showPage()

    # PÁGINA 2: DESTAQUES TERRITORIAIS
    pagina += 1
    desenhar_cabecalho(subtitulo="- DISTRIBUIÇÃO TERRITORIAL E PONTOS DE ATENÇÃO -")
    y = titulo_secao("Destaques Territoriais", altura - 82)

    # 1. Caixas de Incidência por UF (Lado a Lado: Acumulada vs 8 Semanas)
    largura_caixa = (largura_util - 12) / 2
    altura_uf = 100
    
    linhas_uf = [f"{pos}. {row.get('Nome_UF', row.get('Estado', ''))}: {fmt_dec(row['Inc_2026'])} casos" 
                 for pos, (_, row) in enumerate(top5_incidencia_uf.iterrows(), start=1)]
                 
    linhas_uf_8se = [f"{pos}. {row.get('Nome_UF', row.get('Estado', ''))}: {fmt_dec(row['Inc_temporada'])} casos" 
                     for pos, (_, row) in enumerate(top5_incidencia_uf_temporada.iterrows(), start=1)]
                     
    desenhar_caixa(margem_esq, y, largura_caixa, altura_uf, "Maior incidência* SE1 a SE41 - 2026", linhas_uf, 8)
    desenhar_caixa(margem_esq + largura_caixa + 12, y, largura_caixa, altura_uf, f"Maior incidência* recente - SE {se_temporada_inicial} a {se_temporada_atual}", linhas_uf_8se, 8)
    y -= 115

    # 2. Caixas de Top 10 Municípios (Prováveis e Confirmados GAL)
    altura_rankings = 160
    
    linhas_provaveis = [f"{pos}. {row['Município']} - {fmt_int(row['Casos'])} casos" for pos, (_, row) in enumerate(top10_provaveis.iterrows(), 1)]
    linhas_gal = [f"{pos}. {row['Município']} - {fmt_int(row['Confirmados'])} casos" for pos, (_, row) in enumerate(top10_gal.iterrows(), 1)]
    
    desenhar_caixa(margem_esq, y, largura_caixa, altura_rankings, f"Municípios com mais casos prováveis - SE {se_temporada_inicial} a SE {se_temporada_atual}", linhas_provaveis, 8)
    desenhar_caixa(margem_esq + largura_caixa + 12, y, largura_caixa, altura_rankings, f"Municípios com mais confirmações GAL - SE {se_temporada_inicial} a SE {se_temporada_atual}", linhas_gal, 8)
    y -= (altura_rankings + 15)

    # 3. Caixa de casos por porte populacional
    linhas_populacao = [f"{row['Porte populacional']}: " f"{fmt_int(row['Casos prováveis'])} casos " f"({fmt_dec(row['Percentual do total'])}%)"
    for _, row in casos_por_porte_populacional.iterrows()
    if (row["Porte populacional"] != "Sem classificação populacional" or row["Casos prováveis"] > 0)]

    altura_populacao = 125
    espaco_entre_caixas = 15

    desenhar_caixa(margem_esq, y, largura_util, altura_populacao, f"Casos prováveis por porte populacional - " f"{fmt_int(total_provaveis)} casos em 2026", linhas_populacao, 8)
    y -= altura_populacao + espaco_entre_caixas

    # 4. Caixas de Pessoas Indígenas e Positividade por UF
    linhas_indigenas = [
        f"{pos}. {row['Município']} - {fmt_int(row['Casos'])} casos"
        for pos, (_, row) in enumerate(top5_indigenas.iterrows(), 1)
    ]

    linhas_positividade = [
        f"{pos}. {row.get('Nome_UF', row.get('Estado', ''))}: "
        f"{fmt_dec(row['positividade'])}%"
        for pos, (_, row) in enumerate(top5_positividade_uf.iterrows(), 1)
    ]

    altura_indigenas = 125

    desenhar_caixa(
        margem_esq, y, largura_caixa, altura_indigenas,
        f"População Indígena - "
        f"{fmt_int(total_indigenas)} casos prováveis em 2026",
        linhas_indigenas, 8
    )

    desenhar_caixa(
        margem_esq + largura_caixa + 12, y,
        largura_caixa, altura_indigenas,
        f"Maior positividade GAL - SE "
        f"{se_temporada_inicial} a SE {se_temporada_atual}",
        linhas_positividade, 8
    )

    y -= altura_indigenas + espaco_entre_caixas

    desenhar_rodape()
    c.showPage()

    # PÁGINAS 3+: FIGURAS POR EIXO DE ANÁLISE
    
    for nome_grupo, lista_figuras in grupos_graficos.items():
        figuras_existentes = [img for img in lista_figuras if os.path.exists(img)]
        
        if not figuras_existentes:
            print(f"⚠️ Nenhuma figura encontrada no grupo: {nome_grupo}")
            continue

        # Insere duas figuras por página
        for i in range(0, len(figuras_existentes), 2):
            pagina += 1
            desenhar_cabecalho(subtitulo=nome_grupo)
            grupo_pagina = figuras_existentes[i:i + 2]
            
            if len(grupo_pagina) >= 1:
                desenhar_imagem(grupo_pagina[0], margem_esq, 430, largura_util, 335)
            if len(grupo_pagina) >= 2:
                desenhar_imagem(grupo_pagina[1], margem_esq, 65, largura_util, 335)
                
            desenhar_rodape()
            c.showPage()

    c.save()

    print("Arquivo gerado com sucesso!")

# 3. GERAR O PDF
gerar_pdf_brasil(
    grupos_graficos=grupos_graficos,
    destaques_panorama=destaques_panorama,
    top5_incidencia_uf=top5_incidencia_uf,
    top5_incidencia_uf_temporada=top5_incidencia_uf_temporada,
    top10_provaveis=top10_provaveis,
    top10_gal=top10_gal,
    casos_por_porte_populacional= casos_por_porte_populacional,
    top5_indigenas=top5_indigenas,
    total_indigenas=total_indigenas,
    top5_positividade_uf=top5_positividade_uf,
    nome_pdf=salvar_pdf)

#### CHIKUNGUNYA - ESTADOS BRASILEIROS

In [ ]:
# FILTROS PARA DELIMITAR ANÁLISE
nome_estado = 'São Paulo'
filtro_estado = 35
data_analise = '25/11/2026'

# Seu dicionário de siglas dos estados (códigos IBGE)
estados = {
    11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO',
    21: 'MA', 22: 'PI', 23: 'CE', 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL', 
    28: 'SE', 29: 'BA', 31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP', 41: 'PR', 
    42: 'SC', 43: 'RS', 50: 'MS', 51: 'MT', 52: 'GO', 53: 'DF'
}

# Aplicando o IF para verificar e puxar o texto
if filtro_estado in estados:
    estado_encontrado = estados[filtro_estado]
    print(f"O estado correspondente ao código {filtro_estado} é: {nome_estado}")
else:
    print(f"Erro: O código {filtro_estado} não foi encontrado no dicionário.")

provaveis_estado = df_provaveis[df_provaveis['SG_UF'] == filtro_estado]
estado_prim_semestre = provaveis_estado[provaveis_estado['SE'].between(1, 26)]
provaveis_estado_8s = temporada_2026[temporada_2026['SG_UF'] == filtro_estado]
gal_chik_estado = df_gal_final[df_gal_final['UF'] == filtro_estado]
gal_dengue_estado = df_gal_dengue[df_gal_dengue['UF'] == filtro_estado]
print('Bases filtradas com sucesso!')

# Criar pasta para salvar os arquivos
nome_pasta = f'analises_chikungunya_{nome_estado}_SE{se_analise}'
local_salvar = os.getcwd()
local_pasta = os.path.join(local_salvar, nome_pasta)

if not os.path.exists(local_pasta):
    os.makedirs(local_pasta)
    print(f"Pasta {nome_pasta} criada!")
else:
    print(f"A pasta {nome_pasta} já existe. Os arquivos serão salvos nela.")

In [ ]:
from geobr import read_municipality, read_state

with warnings.catch_warnings():
    warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)


# --- 1. PREPARAÇÃO DA BASE DE POPULAÇÃO MUNICIPAL ---
df_populacao = df_municipios.copy()
df_populacao["code_muni_6"] = (
    df_populacao["COD_IBGE"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
    .str.zfill(6)
    .str[:6]
)

# Garantir que a população é numérica
df_populacao["POPULACAO"] = (
    df_populacao["POPULACAO"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_populacao["POPULACAO"] = pd.to_numeric(df_populacao["POPULACAO"], errors="coerce")


# --- 2. CARREGAR MALHA MUNICIPAL E ESTADUAL (geobr) ---
mapa_limites_estados = read_state(code_state=filtro_estado, year=2020)
mapa_municipios = read_municipality(code_muni=filtro_estado, year=2020)
mapa_municipios["code_muni_6"] = mapa_municipios["code_muni"].astype(str).str[:6]


# --- 3. FUNÇÃO AUXILIAR DE PROCESSAMENTO MUNICIPAL ---
def processar_incidencia_muni(df_sinan, df_pop, mapa_muni):
    df_casos = (
        df_sinan["ID_MN_RESI"]
        .dropna()
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(6)
        .str[:6]
        .value_counts()
        .rename_axis("code_muni_6")
        .reset_index(name="Total")
    )

    mapa = mapa_muni.merge(df_casos, on="code_muni_6", how="left")
    mapa = mapa.merge(df_pop[["code_muni_6", "POPULACAO"]], on="code_muni_6", how="left")

    mapa["Total"] = mapa["Total"].fillna(0).astype(int)
    mapa["POPULACAO"] = mapa["POPULACAO"].fillna(1)  # Evita divisão por zero

    mapa["Incidencia"] = (mapa["Total"] / mapa["POPULACAO"]) * 100000
    return mapa


# Processar apenas o dataset do período recente (SE 27 em diante)
mapa_recente = processar_incidencia_muni(provaveis_estado_8s, df_populacao, mapa_municipios)


# --- 4. DEFINIÇÃO DE FAIXAS E CORES ---

def inserir_nomes_municipios(ax, mapa, tamanho_fonte=3):
    municipios_rotulos = mapa.loc[
        mapa.geometry.notna()
        & ~mapa.geometry.is_empty
        & mapa["Categoria"].isin([
            "10,1 - 30",
            "30,1 - 50",
            "50,1 - 100",
            "100,1 - 500",
            "Acima de 500"
        ])
    ].copy()

    pontos_rotulos = municipios_rotulos.geometry.representative_point()

    for (_, municipio), ponto in zip(
        municipios_rotulos.iterrows(), pontos_rotulos
    ):
        nome = municipio["name_muni"]

        if pd.isna(nome):
            continue

        ax.text(
            ponto.x,
            ponto.y,
            str(nome),
            fontsize=tamanho_fonte,
            color="black",
            ha="center",
            va="center",
            zorder=10,
            path_effects=[
                pe.withStroke(
                    linewidth=1.5,
                    foreground="white"
                )
            ]
        )

# Faixas para o período recente
cortes_recente = [-0.001, 0, 10, 30, 50, 100, 500, float("inf")]

rotulos_recente = [
    "Sem Casos",  # Vírgula adicionada aqui
    "0,1 - 10",
    "10,1 - 30",
    "30,1 - 50",
    "50,1 - 100",
    "100,1 - 500",
    "Acima de 500",
]

# Dicionário mapeando diretamente o texto do rótulo para a cor correspondente
dicionario_cores = {
    "Sem Casos": "#e9ecef",
    "0,1 - 10": "#fff7ec",
    "10,1 - 30": "#fdd49e",
    "30,1 - 50": "#fc8d59",
    "50,1 - 100": "#ef6548",
    "100,1 - 500": "#b30000",
    "Acima de 500": "#7f0000",
}

# Categorização
mapa_recente["Categoria"] = pd.cut(mapa_recente["Incidencia"], bins=cortes_recente, labels=rotulos_recente)
mapa_recente["Cor"] = mapa_recente["Categoria"].map(dicionario_cores)

# --- 5. CONSTRUÇÃO DA FIGURA (MAPA ÚNICO) ---
fig, ax = plt.subplots(1, 1, figsize=(15, 15))

mapa_recente.plot(
    color=mapa_recente["Cor"],
    linewidth=0.09,
    edgecolor="black",
    ax=ax,
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.8,
    edgecolor="black",
    ax=ax,
)

inserir_nomes_municipios(ax, mapa_recente, tamanho_fonte=9)

ax.axis("off")

legendas_recente = [
    mpatches.Patch(color=dicionario_cores[rotulo], label=rotulo)
    for rotulo in rotulos_recente
]

ax.legend(
    title=f"Incidência por 100 mil hab. | Fonte: SINAN | Dados até: {data_analise}",
    title_fontsize=12,
    handles=legendas_recente,
    fontsize=12,
    ncols=7,
    loc="lower center",          # Ancora o centro inferior da legenda no ponto definido
    bbox_to_anchor=(0.5, -0.03), # X=0.5 (meio horizontal), Y abaixo do eixo
    frameon=False,
)

# --- TÍTULO GERAL E RODAPÉ ---
plt.suptitle(
    f"TAXA DE INCIDÊNCIA DE CHIKUNGUNYA POR MUNICÍPIO\n{nome_estado} - SE 27 A SE {se_temporada_atual} - 2026".upper(),
    fontsize=20,
    fontweight="bold",
    y=0.9,
)

plt.tight_layout(rect=[0, 0.03, 1, 0.93])

salvar_pasta = os.path.join(local_pasta, f"chik_{nome_estado}_incidencia_municipio_recente_2026.png")
plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# MAPA COROPLÉTICO DE INCIDÊNCIA POR REGIÃO DE SAÚDE

# 1. CONFIGURAÇÃO DE CAMINHOS
pasta_atual = Path('.').resolve()
root = pasta_atual.parent

base_dados = root / 'BASES_DADOS'
base_ibge = root / 'BASE_IBGE'

arquivo_malha = base_ibge / "Regioes_Saude_Brasil.gpkg"
arquivo_vinculo = base_ibge / "vinculo_municipio_regiao.csv"

if not base_dados.exists():
    raise FileNotFoundError('A pasta não foi encontrada. Verificar!')


# 2. FUNÇÕES AUXILIARES (VETORIZADAS)
def padronizar_codigo_municipal(serie):
    """Preserva códigos de 6 dígitos ou remove o dígito verificador de 7 dígitos."""
    codigo = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )
    return codigo.where(codigo.str.fullmatch(r"\d{6,7}", na=False)).str[:6]


def converter_populacao(serie):
    """Converte valores de população de forma vetorizada e robusta."""
    s = serie.astype("string").str.strip()
    
    # Substituições vetorizadas para formatos com vírgula/ponto
    has_comma = s.str.contains(",", na=False)
    
    clean_s = s.copy()
    clean_s.loc[has_comma] = (
        clean_s.loc[has_comma]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    
    # Remove pontos de milhar caso não haja vírgula
    has_dot_thousands = ~has_comma & clean_s.str.contains(r"\d{1,3}(?:\.\d{3})+", regex=True)
    clean_s.loc[has_dot_thousands] = clean_s.loc[has_dot_thousands].str.replace(".", "", regex=False)
    
    return pd.to_numeric(clean_s, errors="coerce")


# 3. CARREGAR MALHAS GEOGRÁFICAS
mapa_regioes = (
    gpd.read_file(arquivo_malha, layer="regioes_saude")[
        ["COD_REGIAO_SAUDE", "REGIAO_SAUDE", "geometry"]
    ]
    .rename(columns={
        "COD_REGIAO_SAUDE": "code_health_region",
        "REGIAO_SAUDE": "name_health_region"
    })
    .assign(code_health_region=lambda x: x["code_health_region"].astype("string").str.strip())
)

mapa_limites_estados = gpd.read_file(arquivo_malha, layer="limites_uf").to_crs(mapa_regioes.crs)


# 4. CARREGAR E VALIDAR VÍNCULO MUNICÍPIO–REGIÃO
vinculo = (
    pd.read_csv(arquivo_vinculo, dtype="string", encoding="utf-8-sig")
    .assign(code_muni_6=lambda df: padronizar_codigo_municipal(df["COD_IBGE"]))
    [["code_muni_6", "Municipio", "UF", "COD_REGIAO_SAUDE", "REGIAO_SAUDE"]]
    .rename(columns={
        "Municipio": "name_muni",
        "UF": "abbrev_state",
        "COD_REGIAO_SAUDE": "code_health_region",
        "REGIAO_SAUDE": "name_health_region"
    })
)

if vinculo.isna().any().any():
    raise ValueError("Há campos ausentes ou códigos inválidos no vínculo.")
if vinculo["code_muni_6"].duplicated().any():
    raise ValueError("Há municípios duplicados na tabela de vínculo.")
if mapa_regioes["code_health_region"].duplicated().any():
    raise ValueError("Há regiões duplicadas na malha.")
if set(vinculo["code_health_region"]) != set(mapa_regioes["code_health_region"]):
    raise ValueError("Os códigos regionais do vínculo e da malha são diferentes.")


# 5. CONTABILIZAR CASOS POR MUNICÍPIO
df_casos_municipio = (
    padronizar_codigo_municipal(provaveis_estado_8s["ID_MN_RESI"])
    .value_counts(dropna=False)
    .rename_axis("code_muni_6")
    .reset_index(name="Total")
)

casos_sem_vinculo = df_casos_municipio.loc[
    ~df_casos_municipio["code_muni_6"].isin(vinculo["code_muni_6"])
].copy()

if not casos_sem_vinculo.empty:
    print("\nCasos sem vínculo com a regionalização:")
    print(casos_sem_vinculo.to_string(index=False))
    print(f"\nTotal de casos não incluídos no mapa: {int(casos_sem_vinculo['Total'].sum())}")


# 6. PREPARAR POPULAÇÃO MUNICIPAL
df_populacao = (
    df_municipios[["COD_IBGE", "POPULACAO"]]
    .assign(
        code_muni_6=lambda df: padronizar_codigo_municipal(df["COD_IBGE"]),
        POPULACAO=lambda df: converter_populacao(df["POPULACAO"])
    )
    .assign(POPULACAO=lambda df: np.where(df["POPULACAO"].gt(0), df["POPULACAO"], np.nan))
    [["code_muni_6", "POPULACAO"]]
    .dropna(subset=["code_muni_6"])
    .drop_duplicates()
)

if df_populacao["code_muni_6"].duplicated().any():
    raise ValueError("Há mais de uma população para o mesmo município.")


# 7. INTEGRAR DADOS MUNICIPAIS
dados_municipais = (
    vinculo
    .merge(df_casos_municipio, on="code_muni_6", how="left", validate="one_to_one")
    .merge(df_populacao, on="code_muni_6", how="left", validate="one_to_one")
    .assign(Total=lambda df: df["Total"].fillna(0).astype(int))
)

casos_mapeados = int(dados_municipais["Total"].sum())
casos_nao_mapeados = int(casos_sem_vinculo["Total"].sum())

if casos_mapeados + casos_nao_mapeados != len(provaveis_estado_8s):
    raise ValueError("Houve divergência na contabilização dos casos.")


# 8. AGREGAR POR REGIÃO DE SAÚDE E CALCULAR INCIDÊNCIA
df_regioes = (
    dados_municipais
    .groupby(["code_health_region", "name_health_region"], as_index=False)
    .agg(
        Total=("Total", "sum"),
        POPULACAO=("POPULACAO", "sum"),
        Municipios=("code_muni_6", "nunique"),
        Municipios_sem_populacao=("POPULACAO", lambda x: x.isna().sum())
    )
    .assign(
        POPULACAO=lambda df: np.where(
            df["Municipios_sem_populacao"].eq(0) & df["POPULACAO"].gt(0),
            df["POPULACAO"],
            np.nan
        ),
        Incidencia=lambda df: df["Total"].div(df["POPULACAO"]).mul(100_000)
    )
)

# 9. ASSOCIAR À MALHA E CLASSIFICAR
import numpy as np
import pandas as pd

# 9. ASSOCIAR À MALHA E CLASSIFICAR
# Ajustado para incluir o zero de forma isolada
cortes = [-0.001, 0, 10, 50, 100, 200, 300, np.inf]

rotulos = [
    "Sem Casos",
    "0,1 a 10",
    "10,1 a 50",
    "50,1 a 100",
    "100,1 a 200",
    "200,1 a 300",
    "Acima de 300"
]

sem_populacao = "População incompleta ou indisponível"

dicionario_cores = {
    "Sem Casos": "#e9ecef",
    "0,1 a 10": "#fff7ec",
    "10,1 a 50": "#fdd49e",
    "50,1 a 100": "#fc8d59",
    "100,1 a 200": "#ef6548",
    "200,1 a 300": "#b30000",
    "Acima de 300": "#7f0000",
    sem_populacao: "#d9d9d9"
}

mapa_casos = (
    mapa_regioes
    .merge(
        df_regioes.drop(columns="name_health_region"),
        on="code_health_region",
        how="left",
        validate="one_to_one"
    )
    .assign(
        Total=lambda df: df["Total"].astype(int),
        # pd.cut agora vai encaixar o valor 0 na categoria "Sem Casos"
        Categoria=lambda df: pd.cut(df["Incidencia"], bins=cortes, labels=rotulos, right=True),
        Categoria_mapa=lambda df: df["Categoria"].astype("string").fillna(sem_populacao),
        Cor_Regiao=lambda df: df["Categoria_mapa"].map(dicionario_cores)
    )
)

if mapa_casos["Total"].isna().any():
    raise ValueError("Há regiões da malha sem correspondência nos indicadores.")


# 10. RELATÓRIO DE PROBLEMAS POPULACIONAIS
problemas_populacao = dados_municipais.loc[
    dados_municipais["POPULACAO"].isna(),
    ["code_muni_6", "name_muni", "abbrev_state", "code_health_region", "name_health_region", "Total"]
]

if not problemas_populacao.empty:
    print("\nMunicípios sem população válida:")
    print(problemas_populacao.to_string(index=False))

# 11. FILTRAR ESTADO DE INTERESSE
# Selecionar os códigos das regiões pertencentes à UF
codigos_regioes_uf = vinculo.loc[
    vinculo["abbrev_state"].eq(estado_encontrado),
    "code_health_region"
].unique()

# Filtrar as regiões e o limite estadual
mapa_casos = mapa_casos.loc[
    mapa_casos["code_health_region"].isin(codigos_regioes_uf)
].copy()

mapa_limites_estados = mapa_limites_estados.loc[
    mapa_limites_estados["UF"].eq(estado_encontrado)
].copy()

if mapa_casos.empty:
    raise ValueError(f"Nenhuma região encontrada para {nome_estado}.")

# 11. PLOTAGEM DO MAPA
fig, ax = plt.subplots(figsize=(15, 15))

mapa_casos.plot(
    color=mapa_casos["Cor_Regiao"],
    linewidth=0.35,
    edgecolor="0.55",
    ax=ax
)

mapa_limites_estados.boundary.plot(
    linewidth=0.5,
    color="black",
    ax=ax
)

# INSERIR OS NOMES DAS REGIÕES AQUI

partes = mapa_casos.loc[
    mapa_casos["Categoria"].isin([
        "10,1 a 50",
        "50,1 a 100",
        "100,1 a 200",
        "200,1 a 300",
        "Acima de 300"
    ]),
    
    ["code_health_region", "name_health_region", "geometry"]
].explode(index_parts=False).reset_index(drop=True)

partes["area_m2"] = partes.to_crs(5880).geometry.area

maiores_partes = partes.loc[
    partes.groupby("code_health_region")["area_m2"].idxmax()
].copy()

pontos_rotulos = maiores_partes.geometry.representative_point()

for (_, regiao), ponto in zip(
    maiores_partes.iterrows(),
    pontos_rotulos
):
    ax.text(
        ponto.x,
        ponto.y,
        regiao["name_health_region"],
        fontsize=8,
        color="black",
        ha="center",
        va="center",
        zorder=5,
        path_effects=[
            pe.withStroke(linewidth=2.5, foreground="white")
        ]
    )

# Legenda
ordem_legenda = rotulos.copy()
if mapa_casos["Incidencia"].isna().any():
    ordem_legenda.append(sem_populacao)

lista_legendas = [
    mpatches.Patch(facecolor=dicionario_cores[rotulo], edgecolor="none", label=rotulo)
    for rotulo in ordem_legenda
]

ax.legend(
    title="Incidência por 100 mil habitantes",
    title_fontsize=12,
    handles=lista_legendas,
    fontsize=12,
    ncols=7,
    loc="lower center",
    bbox_to_anchor=(0.50, -0.04),
    frameon=False,
    alignment="center"
)

# Títulos e Rodapé (Correção do erro de sintaxe do hífen no f-string anterior)
ax.set_title(
    "INCIDÊNCIA DE CASOS PROVÁVEIS DE CHIKUNGUNYA POR REGIÃO DE SAÚDE\n"
    f"{nome_estado} - SE {se_temporada_inicial} A SE {se_temporada_atual} - 2026".upper(),
    fontsize=20,
    fontweight="bold",
    pad=-10
)

ax.axis("off")

# SALVAR
salvar_pasta = Path(local_pasta) / (f"chik_{nome_estado}_incidencia_regiao_saude.png")
fig.savefig(salvar_pasta, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 1. PARÂMETROS

pasta_atual = Path('.').resolve()
root = pasta_atual.parent

base_dados = root / 'BASES_DADOS'
base_ibge = root / 'BASE_IBGE'

if not base_dados.exists():
    raise FileNotFoundError('A pasta BASES_DADOS não foi encontrada. Verificar!')

ARQUIVO_CASOS = base_dados / "chik_brasil_provaveis_2026.csv"
ARQUIVO_POPULACAO = base_ibge / "IBGE_Municipios_Regioes_Saude_Populacao_2026.csv"

TOP_N = 10

OUTPUT_XLSX = os.path.join(local_pasta, f"chik_{nome_estado}_tabela_incidencia_2026.xlsx")
OUTPUT_PNG = os.path.join(local_pasta, f"chik_{nome_estado}_municipio_tabela_comparativo_2026.png")

# 2. LEITURA E FILTRAGEM OTIMIZADA DAS BASES

# Lendo População (apenas a UF de interesse para reduzir consumo de memória)
dtype_pop = {
    "COD_IBGE": "Int64",
    "UF": "string",
    "Municipio": "string",
    "REGIAO_SAUDE": "string",
    "POPULACAO": "float64"
}

populacao = pd.read_csv(
    ARQUIVO_POPULACAO,
    sep=None,
    engine="python",
    encoding="utf-8-sig",
    usecols=list(dtype_pop.keys()),
    dtype=dtype_pop
)

populacao["UF"] = populacao["UF"].str.strip().str.upper()

pop_uf = populacao[
    populacao["UF"].eq(estado_encontrado) & populacao["COD_IBGE"].notna()
].copy()

if pop_uf.empty:
    raise ValueError(f"UF não encontrada na base populacional: {estado_encontrado}")

if pop_uf["COD_IBGE"].duplicated().any():
    raise ValueError("Há códigos municipais duplicados na base populacional.")

# Lendo Casos (Filtragem rápida no carregamento)
dtype_casos = {
    "ID_MN_RESI": "Int64",
    "SE": "float64"
}

casos = pd.read_csv(
    ARQUIVO_CASOS,
    sep=None,
    engine="python",
    encoding="utf-8-sig",
    usecols=["ID_MN_RESI", "SE", "DT_SIN_PRI"],
    dtype=dtype_casos,
    parse_dates=["DT_SIN_PRI"]
)

casos = casos.rename(columns={"ID_MN_RESI": "COD_IBGE"})

# Filtro vetorizado
casos = casos[
    (casos["DT_SIN_PRI"].dt.year == 2026) &
    (casos["SE"] >= 1) & 
    (casos["SE"] <= se_analise) &
    (casos["COD_IBGE"].notna()) &
    (casos["COD_IBGE"].isin(pop_uf["COD_IBGE"])) # Filtra apenas códigos do estado selecionado
].copy()

# 3. CONTAGEM E PIVOTEAMENTO DE CASOS
coluna_inicial = "SE1–26"
coluna_recente = f"SE27–{se_analise}"

# Definindo o período com np.select/np.where para agregação em passo único
condicoes = [
    (casos["SE"] >= 1) & (casos["SE"] <= 26),
    (casos["SE"] >= 27) & (casos["SE"] <= se_analise)
]
escolhas = [coluna_inicial, coluna_recente]
casos["PERIODO"] = np.select(condicoes, escolhas, default=None)

casos = casos[casos["PERIODO"].notna()]

# Tabela cruzada/agrupada vetorizada
agrupado = (
    casos.groupby(["COD_IBGE", "PERIODO"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Garantir existência de ambas as colunas mesmo se faltar algum período
for col in [coluna_inicial, coluna_recente]:
    if col not in agrupado.columns:
        agrupado[col] = 0

# 4. JUNÇÃO E CÁLCULO DE INCIDÊNCIA
tabela = pop_uf.merge(agrupado, on="COD_IBGE", how="left")

tabela[coluna_inicial] = tabela[coluna_inicial].fillna(0).astype(int)
tabela[coluna_recente] = tabela[coluna_recente].fillna(0).astype(int)

# Incidência por 100 mil habitantes
denominador = np.where(tabela["POPULACAO"] > 0, tabela["POPULACAO"], np.nan)
tabela["INC_INICIAL"] = (tabela[coluna_inicial] / denominador) * 100_000
tabela["INC_RECENTE"] = (tabela[coluna_recente] / denominador) * 100_000

# Filtro e Ordenação
tabela = (
    tabela[
        (tabela[coluna_recente] > 0) & 
        (tabela["INC_RECENTE"].notna())
    ]
    .sort_values(
        by=["INC_RECENTE", coluna_recente, "Municipio"],
        ascending=[False, False, True]
    )
)

if TOP_N is not None:
    tabela = tabela.head(TOP_N)

# Organização e Ajuste de Colunas
tabela = tabela.rename(columns={
    "Municipio": "Município",
    "REGIAO_SAUDE": "Região de Saúde"
})[[
    "Município",
    "Região de Saúde",
    coluna_inicial,
    "INC_INICIAL",
    coluna_recente,
    "INC_RECENTE"
]].reset_index(drop=True)

tabela["Município"] = tabela["Município"].str.upper()

# 5. GERAR ARQUIVO EXCEL (.XLSX) COM LARGURA MANUAL DE COLUNAS
tabela_excel = tabela.copy()
tabela_excel.columns = [
    "Município", "Região de Saúde", 
    coluna_inicial, "Incidência (SE1-26)", 
    coluna_recente, "Incidência (Recente)"
]

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    tabela_excel.to_excel(writer, index=False, sheet_name=estado_encontrado)
    
    # Acessa a planilha ativa
    worksheet = writer.sheets[estado_encontrado]
    
    # Dicionário com as larguras (aumente bem os números para testar, ex: 40 ou 50)
    larguras_personalizadas = {
        'A': 40,  # Município (deixe bem largo para ver a diferença)
        'B': 35,  # Região de Saúde
        'C': 20,  # coluna_inicial
        'D': 25,  # Incidência (SE1-26)
        'E': 20,  # coluna_recente
        'F': 25   # Incidência (Recente)
    }
    
    # Aplica as larguras e força o dimensionamento
    for col_letter, largura in larguras_personalizadas.items():
        worksheet.column_dimensions[col_letter].width = largura

# 6. GERAR FIGURA PNG (MATPLOTLIB)
import textwrap

# Cópia exclusiva para apresentação.
tabela_formatada = tabela.copy()


def formatar_numero_figura(valor, casas=0):
    if pd.isna(valor):
        return "—"

    return (
        f"{valor:,.{casas}f}"
        .replace(",", "@")
        .replace(".", ",")
        .replace("@", ".")
    )


for coluna in [coluna_inicial, coluna_recente]:
    tabela_formatada[coluna] = tabela_formatada[coluna].map(
        lambda valor: formatar_numero_figura(valor)
    )

for coluna in ["INC_INICIAL", "INC_RECENTE"]:
    tabela_formatada[coluna] = tabela_formatada[coluna].map(
        lambda valor: formatar_numero_figura(valor, casas=2)
    )

# Quebra de nomes longos para evitar sobreposição.
for coluna, largura in [
    ("Município", 30),
    ("Região de Saúde", 32)
]:
    tabela_formatada[coluna] = tabela_formatada[coluna].map(
        lambda texto: textwrap.fill(str(texto), width=largura)
    )

headers = [
    "Município",
    "Região de Saúde",
    f"Casos\n{coluna_inicial}",
    f"Incidência\n{coluna_inicial}",
    f"Casos\n{coluna_recente}",
    f"Incidência\n{coluna_recente}"
]

cell_text = tabela_formatada.values.tolist()

# Larguras proporcionais: soma = 1.
larguras_colunas = [0.27, 0.29, 0.10, 0.12, 0.10, 0.12]

# Altura adaptada ao número de linhas e às quebras dos nomes.
altura_cabecalho = 0.58

alturas_linhas = [
    0.34 + 0.19 * (
        max(str(linha[0]).count("\n"), str(linha[1]).count("\n"))
    )
    for linha in cell_text
]

altura_tabela = altura_cabecalho + sum(alturas_linhas)
altura_figura = max(3.2, altura_tabela + 1.55)

with plt.rc_context({
    "font.family": "Arial",
    "figure.facecolor": "white",
    "axes.facecolor": "white"
}):
    fig = plt.figure(figsize=(16, altura_figura))

    # Eixo próximo das bordas, sem margens laterais excessivas.
    ax = fig.add_axes([
        0.015,
        0.65 / altura_figura,
        0.97,
        altura_tabela / altura_figura
    ])
    ax.axis("off")

    if cell_text:
        table = ax.table(
            cellText=cell_text,
            colLabels=headers,
            cellLoc="right",
            colWidths=larguras_colunas,
            bbox=[0, 0, 1, 1]
        )

        table.auto_set_font_size(False)
        table.set_fontsize(11)

        for (row, col), cell in table.get_celld().items():
            # Bordas discretas e uniformes em toda a tabela.
            cell.set_edgecolor("#DCE3EB")
            cell.set_linewidth(0.55)
            cell.PAD = 0.035

            if row == 0:
                cell.set_height(
                    altura_cabecalho / altura_tabela
                )
                cell.set_facecolor("#DCE3EB")
                cell.set_text_props(
                    color="black",
                    weight="bold",
                    ha="left" if col in [0, 1] else "center",
                    va="center"
                )

            else:
                cell.set_height(
                    alturas_linhas[row - 1] / altura_tabela
                )

                cor_fundo = (
                    "#FFFFFF" if row % 2 else "#ffffff"
                )
                cell.set_facecolor(cor_fundo)

                cell.set_text_props(
                    color="#000000",
                    ha="left" if col in [0, 1] else "right",
                    va="center"
                )

                # Destaque suave da incidência no período recente.
                if col == 5:
                    cell.set_facecolor(
                        "#ffffff" if row % 2 else "#ffffff"
                    )
                    cell.set_text_props(
                        color="#000000",
                    )

    else:
        ax.text(
            0.5, 0.5,
            "Nenhum dado encontrado para os filtros aplicados.",
            ha="center",
            va="center",
            fontsize=13,
            color="#666666",
            transform=ax.transAxes
        )

    fig.text(
        0.020,
        1 - 0.50 / altura_figura,
        f"CASOS PROVÁVEIS E INCIDÊNCIA DE CHIKUNGUNYA - {nome_estado} - SE 27 a {se_analise}".upper(),
        ha="left",
        va="top",
        fontsize=17,
        fontweight="bold",
        color="#000000"
    )

    fig.text(
        0.015,
        0.30 / altura_figura,
        (
            "Incidência por 100 mil habitantes. "
            "Municípios ordenados pela incidência no período recente.\n"
            f"Fonte: SINAN | Dados até: {data_analise}."
        ),
        ha="left",
        va="center",
        fontsize=9,
        color="#59697A"
    )

plt.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ESTADO — ANÁLISE DAS REGIÕES DE SAÚDE

resultado_regioes_estado = gerar_analise_regioes_saude(
    base_casos=provaveis_estado,
    base_populacao=df_municipios,
    arquivo_vinculo=base_ibge / "vinculo_municipio_regiao.csv",
    semana_final=se_analise,
    territorio=nome_estado,
    pasta_saida=(
        Path.cwd()
        / f"analises_chikungunya_{nome_estado}_SE{se_analise}"
    ),
    data_referencia=data_analise,
    uf=estado_encontrado,
    base_gal=df_gal_final
)

In [ ]:
# 1. PREPARAÇÃO DOS DADOS

# Filtrar bases de dados
gal_filtrado = df_gal_final[df_gal_final['UF'] == filtro_estado]

# Contar exames realizados na base do GAL
df_exames = gal_filtrado.groupby('SE')['exames_realizados'].sum().reset_index()
df_exames.columns = ['SE', 'Exames_Realizados']

# Contar os casos positivos na base do GAL
df_conf = gal_filtrado.groupby('SE')['casos_positivos'].sum().reset_index()
df_conf.columns = ['SE', 'Exames_Confirmados']

# Juntar as duas bases em uma só por Semana Epidemiológica
df_gal_combinado = pd.merge(df_exames, df_conf, on='SE', how='left').fillna(0)

# Calcular o Percentual de Positividade (Evitando divisão por zero)
df_gal_combinado['Positividade_Pct'] = np.where(
    df_gal_combinado['Exames_Realizados'] > 0,
    (df_gal_combinado['Exames_Confirmados'] / df_gal_combinado['Exames_Realizados']) * 100,
    0
)

# 2. CONFIGURAÇÃO VISUAL
grafico_sem = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax1 = grafico_sem.add_subplot(gs[0])

# 3. GRÁFICO DE COLUNAS (Exames Realizados - Eixo Y Principal)
sns.barplot(
    data=df_gal_combinado, 
    x='SE', 
    y='Exames_Realizados', 
    color='#adb5bd', 
    ax=ax1, 
    width=0.8
)

# Adicionar os rótulos nas barras (Volume absoluto)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%d', padding=3, fontsize=10, color='#adb5bd')

# 4. GRÁFICO DE LINHA (Positividade - Eixo Y Secundário)
ax2 = ax1.twinx()  # Cria um eixo Y secundário compartilhando o mesmo eixo X

# Plotar a linha usando os ticks exatos gerados pelo barplot para garantir o alinhamento
x_pos = ax1.get_xticks()
ax2.plot(
    x_pos, 
    df_gal_combinado['Positividade_Pct'], 
    color='#f2542d', 
    marker='o', 
    linewidth=2.5, 
    markersize=8
)

# Adicionar os rótulos na linha (Percentual)
offset_y = df_gal_combinado['Positividade_Pct'].max() * 0.05 # Deslocamento leve para o texto não ficar em cima da bolinha
for i, y in enumerate(df_gal_combinado['Positividade_Pct']):
    ax2.text(
        x_pos[i], 
        y + offset_y, 
        f'{y:.1f}', 
        color='#f2542d', # Destaque na cor vermelha para a porcentagem (opcional)
        ha='center', 
        va='bottom', 
        fontweight='bold', 
        fontsize=10 
    )

# 5. ESTILIZAÇÃO DOS EIXOS E LEGENDA
ax1.set_title(f'Exames realizados e positividade GAL para chikungunya - {nome_estado} - 2026'.upper(), 
              fontsize=18, fontweight='bold', pad=20)

ax1.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax1.set_ylabel('Exames Realizados', fontsize=16, labelpad=16, color='#000000')
ax2.set_ylabel('Positividade (%)', fontsize=16, labelpad=16, color='#000000')

# Ajustar o limite do eixo secundário para dar espaço aos rótulos da linha
ax2.set_ylim(0, df_gal_combinado['Positividade_Pct'].max() * 1.15)
ax2.set_xlim(-1, df_exames['SE'].max() + 0)

# Configurar as grades de fundo
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.grid(axis='x', linestyle='--', alpha=0.3)
ax2.grid(False) # Desliga a grade do eixo secundário para não poluir o gráfico

# Criar uma legenda combinada manual
patch_barras = mpatches.Patch(color='#dee2e6', label='Exames Realizados')
linha_percentual = lines.Line2D([], [], color='#f2542d', marker='o', label='Positividade (%)')
ax1.legend(handles=[patch_barras, linha_percentual], fontsize=11, loc='upper right', frameon=True)

# 6. RODAPÉ E SALVAMENTO
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')

# Separar as somas em variáveis evita erros de aspas na formatação do texto
total_realizados = df_gal_combinado['Exames_Realizados'].sum()
total_confirmados = df_gal_combinado['Exames_Confirmados'].sum()

rodape_texto = (f'Fonte: GAL | Dados até: {data_gal}')

ax_rodape.text(0.0, 0.5, rodape_texto, ha='left', va='center', fontsize=9, color='black')

salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_positividade_gal_semana.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1. PREPARAÇÃO DOS DADOS

# Agrupar casos prováveis por SE
df_prov_estado = provaveis_estado.groupby('SE')['SEM_PRI'].count().reset_index()
df_prov_estado.columns = ['SE', 'Casos_Provaveis']

# Agrupar dados laboratoriais do GAL por SE
df_gal_agg_estado = gal_chik_estado.groupby('SE')[['exames_realizados', 'casos_positivos']].sum().reset_index()

# Unir as bases
df_grafico = pd.merge(df_prov_estado, df_gal_agg_estado, on='SE', how='left').fillna(0)

# Calcular o percentual de positividade (%)
df_grafico['Positividade_Pct'] = np.where(
    df_grafico['exames_realizados'] > 0,
    (df_grafico['casos_positivos'] / df_grafico['exames_realizados']) * 100,
    0
)

# 2. CONFIGURAÇÃO VISUAL E FIGURA
sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(20, 10))

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax1 = fig.add_subplot(gs[0])

# 3. GRÁFICO DE COLUNAS (Casos Prováveis - Eixo Y Principal)
cor_barras = '#b8b8b8' 

sns.barplot(
    data=df_grafico,
    x='SE',
    y='Casos_Provaveis',
    color=cor_barras,
    ax=ax1,
    width=0.8
)

# Rótulos simples em cima de cada barra
for container in ax1.containers:
    ax1.bar_label(container, fmt='%d', padding=3, fontsize=10, color=cor_barras)

# 4. GRÁFICO DE LINHA (Positividade GAL - Eixo Y Secundário)
ax2 = ax1.twinx()
x_pos = np.arange(len(df_grafico['SE']))
cor_linha = '#f2542d' 

ax2.plot(
    x_pos,
    df_grafico['Positividade_Pct'],
    color=cor_linha,
    marker='o',
    linewidth=2.5,
    markersize=7
)

# Rótulos da positividade (%) na linha
ymax_pos = df_grafico['Positividade_Pct'].max()
offset_y = (ymax_pos * 0.03) if ymax_pos > 0 else 1

for i, y in enumerate(df_grafico['Positividade_Pct']):
    if y > 0:  # Exibe rótulo apenas se houver positividade > 0%
        ax2.text(
            x_pos[i],
            y + offset_y,
            f'{y:.1f}',
            color=cor_linha,
            ha='center',
            va='bottom',
            fontweight='bold',
            fontsize=9
        )

# 5. ESTILIZAÇÃO E AJUSTES DE LIMITES
ax1.set_title(
    f'CASOS PROVÁVEIS E POSITIVIDADE DO GAL PARA CHIKUNGUNYA - {nome_estado} - 2026'.upper(),
    fontsize=16,
    fontweight='bold',
    pad=18
)

ax1.set_xlabel('Semana Epidemiológica', fontsize=12, labelpad=10)
ax1.set_ylabel('Casos Prováveis', fontsize=12, labelpad=10, color='#000000', fontweight='bold')
ax2.set_ylabel('Positividade GAL (%)', fontsize=12, labelpad=10, color='#000000', fontweight='bold')

# Limites dos eixos Y para evitar superposição dos rótulos
ymax_barras = df_grafico['Casos_Provaveis'].max()
ax1.set_ylim(0, ymax_barras * 1.15 if ymax_barras > 0 else 10)
ax2.set_ylim(0, ymax_pos * 1.25 if ymax_pos > 0 else 100)
ax1.set_xlim(-1, df_grafico['SE'].max() - 0)

# Suavização da grade
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.grid(axis='x', linestyle='--', alpha=0.5)
ax2.grid(False)

# Legenda unificada e discreta
barra_patch = mpatches.Patch(color=cor_barras, alpha=0.85, label='Casos Prováveis')
linha_legend = lines.Line2D([], [], color=cor_linha, marker='o', label='Positividade GAL (%)')

ax1.legend(
    handles=[barra_patch, linha_legend],
    loc='upper right',
    fontsize=10,
    frameon=True,
    facecolor='white',
    framealpha=0.9
)

# 6. RODAPÉ E SALVAMENTO
ax_rodape = fig.add_subplot(gs[1])
ax_rodape.axis('off')

rodape_texto = f'Fonte: Sinan e GAL | Dados até: {data_analise}'
ax_rodape.text(0.0, 0.5, rodape_texto, ha='left', va='center', fontsize=9, color='#000000')

salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_casos_positividade_se.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# CLASSIFICAÇÃO FINAL POR SEMANA EPIDEMIOLÓGICA
# Tratamento da Classificação
# Transformar em número. O que for vazio ('') vira NaN (Nulo)
df_notificados['CLASSI_FIN'] = pd.to_numeric(df_notificados['CLASSI_FIN'], errors='coerce')
df_notif_filtrado = df_notificados[df_notificados['SG_UF'] == filtro_estado]

# Preencher os valores nulos (NaN) com um código numérico fictício (ex: 99) 
# Isso impede que o groupby ignore os casos "Em Investigação"
df_notif_filtrado['CLASSI_FIN'] = df_notif_filtrado['CLASSI_FIN'].fillna(99)

# Novo dicionário com o código 99 representando o que estava vazio
mapa_class = {5: 'Descartado', 13: 'Confirmado', 99: 'Em Investigação'}

# Agrupar os dados
# Dataframe para o gráfico de barras empilhadas e tabela
df_class = df_notif_filtrado.groupby(['SE', 'CLASSI_FIN']).size().unstack(fill_value=0) 
df_class = df_class.rename(columns=mapa_class)

# Configurar o estilo visual
grafico_class = plt.figure(figsize=(20, 10))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_class.add_subplot(gs[0])

# Criar o Gráfico de Barras Empilhadas
# Adicionei cores distintas para cada classificação
cores = {'Confirmado': '#e9c46a', 'Descartado': '#e07a5f', 'Em Investigação': '#f95738'}
df_class.plot(kind='bar', stacked=True, ax=ax, width=0.8, color=[cores.get(col, '#333333') for col in df_class.columns])

# Adicionar os rótulos de dados (bar_label)
# Como são barras empilhadas, precisamos passar por CADA container (cada pedaço da barra)
for c in ax.containers:
    # Condição para só escrever o número se ele for maior que 0 (evita poluir o gráfico com zeros)
    labels = [f'{v.get_height():.0f}' if v.get_height() > 0 else '' for v in c]
    # label_type='center' coloca o número bem no meio da cor respectiva
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='white', fontweight='bold')

ax.set_title(f'Classificação final dos casos notificados de chikungunya - {nome_estado} - 2026'.upper(), 
          fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Notificados', fontsize=16, labelpad=16)
#ax.set_ylim(0, len(df_class) - 20)

# O eixo X no pandas fica rotacionado em 90 graus por padrão. Isso conserta e deixa reto:
plt.xticks(rotation=0) 

# Ajustar o limite do eixo Y (adapte se necessário)
# ax.set_ylim(0, 1400) # Removi temporariamente para o pandas ajustar automaticamente às barras somadas

# Melhorar a legenda
ax.legend(title='Classificação Final', fontsize=11, title_fontsize=12, loc='upper right')

ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)

# Informações rodapé 
ax_rodape = grafico_class.add_subplot(gs[1])
ax_rodape.axis('off') # Minúsculo
rodape_texto = f'Fonte: SINAN | Dados até: {data_analise}'
# Ajustado para ficar centralizado verticalmente no rodapé
ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_provaveis_classificacao.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Filtro de dados para estado de interesse
provaveis_chik_estado = df_provaveis[
    df_provaveis['SG_UF'] == filtro_estado
]
provaveis_den_estado = den_provaveis[
    den_provaveis['SG_UF'] == filtro_estado
]
gal_chik_estado = df_gal_final[
    df_gal_final['UF'] == filtro_estado
]
gal_den_estado = df_gal_dengue[
    df_gal_dengue['UF'] == filtro_estado
]

SINAN = {
    'Dengue': provaveis_den_estado,
    'Chikungunya': provaveis_chik_estado
}
GAL = {
    'Dengue': gal_den_estado,
    'Chikungunya': gal_chik_estado
}

PASTA_SAIDA = (
    Path(local_pasta) if 'local_pasta' in globals() else Path('.')
)
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

CORES = {
    'Casos prováveis': '#e9c46a',
    'Exames realizados': '#e07a5f',
    'Exames positivos': '#f95738'
}


def resumir_sinan(df):
    d = df.assign(
        SE=pd.to_numeric(df['SE'], errors='coerce')
    ).dropna(subset=['SE'])

    d = d[d['SE'].between(1, 53)].assign(
        SE=lambda x: x['SE'].astype(int)
    )

    return (
        d.groupby('SE')['casos_provaveis'].sum()
        if 'casos_provaveis' in d
        else d.groupby('SE').size()
    ).rename('Casos prováveis')


def resumir_gal(df):
    colunas = ['exames_realizados', 'casos_positivos']

    d = df.assign(
        SE=pd.to_numeric(df['SE'], errors='coerce')
    ).dropna(subset=['SE'])

    d = d[d['SE'].between(1, 53)].assign(
        SE=lambda x: x['SE'].astype(int)
    )

    d[colunas] = (
        d[colunas].apply(pd.to_numeric, errors='coerce').fillna(0)
    )

    return (
        d.groupby('SE')[colunas]
        .sum()
        .rename(columns={
            'exames_realizados': 'Exames realizados',
            'casos_positivos': 'Exames positivos'
        })
    )


sinan_se = {
    doenca: resumir_sinan(df)
    for doenca, df in SINAN.items()
}
gal_se = {
    doenca: resumir_gal(df)
    for doenca, df in GAL.items()
}

se_final = int(min(
    *(s.index.max() for s in sinan_se.values()),
    *(g.index.max() for g in gal_se.values())
))


def consolidar(doenca, janela):
    semanas = pd.Index(
        range(se_final - janela + 1, se_final + 1),
        name='SE'
    )

    return (
        pd.concat([sinan_se[doenca], gal_se[doenca]], axis=1)
        .reindex(semanas, fill_value=0)
        .reset_index()
    )


# Apenas as últimas oito semanas
dados = {
    (doenca, 8): consolidar(doenca, 8)
    for doenca in ('Dengue', 'Chikungunya')
}

print(
    f'Fechamento comum: SE {se_final} | '
    f'8 SE: {se_final - 7}–{se_final}'
)


def grafico(ax, df, titulo):
    x, largura = np.arange(len(df)), 0.25

    for metrica, deslocamento in zip(
        CORES, (-largura, 0, largura)
    ):
        barras = ax.bar(
            x + deslocamento,
            df[metrica],
            largura,
            label=metrica,
            color=CORES[metrica],
            edgecolor='white',
            linewidth=.5,
            zorder=3
        )

        ax.bar_label(
            barras,
            labels=[
                f'{v:,.0f}'.replace(',', '.') if v else ''
                for v in df[metrica]
            ],
            padding=4,
            fontsize=10,
            rotation=90
        )

    ax.set(
        title=titulo,
        xlabel='Semana epidemiológica',
        ylabel='Número de registros',
        xticks=x,
        xticklabels=[f'SE {s}' for s in df['SE']]
    )

    ax.title.set_fontweight('bold')
    ax.xaxis.label.set_fontweight('bold')
    ax.yaxis.label.set_fontweight('bold')

    ax.set_ylim(
        0, df[list(CORES)].to_numpy().max() * 1.23
    )

    ax.grid(
        axis='y', linestyle='--', alpha=0.6, zorder=0
    )
    ax.grid(
        axis='x', linestyle='--', alpha=0.6, zorder=0
    )
    ax.spines[['top', 'right']].set_visible(True)


def rodape(fig):
    fig.text(
        .01, .01,
        f'Fonte: SINAN e GAL | Dados até {data_analise}.',
        fontsize=9
    )


# Painel 1 × 2 — somente oito semanas
fig, axes = plt.subplots(1, 2, figsize=(19, 7))

ordem = [('Dengue', 8), ('Chikungunya', 8)]

for ax, chave in zip(axes.flat, ordem):
    grafico(
        ax,
        dados[chave],
        f'{chave[0]} — últimas {chave[1]} SE'.upper()
    )

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    ncol=3,
    frameon=False,
    loc='upper center',
    bbox_to_anchor=(.5, .85)
)

fig.suptitle(
    (
        'Casos prováveis, exames realizados e exames positivos '
        f'para dengue e chikungunya - {nome_estado} - 2026'
    ).upper(),
    fontsize=17,
    fontweight='bold',
    y=.895
)

rodape(fig)

fig.tight_layout(
    rect=[0, .06, 1, .85],
    w_pad=2.5
)

fig.savefig(
    PASTA_SAIDA
    / f'painel_dengue_chik_{nome_estado}_casos_exames.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

In [ ]:
# VIGILÂNCIA EPIDEMIOLÓGICA E LABORATORIAL POR MUNICÍPIO

import textwrap

TOP_MUNICIPIOS = 20
SE_INICIAL_MUNICIPIOS = 27
SE_FINAL_MUNICIPIOS = int(se_analise)


def codigo_municipio_painel(serie):
    codigo = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )
    return codigo.where(
        codigo.str.fullmatch(r"\d{6,7}", na=False)
    ).str[:6]


def filtrar_sinan_municipios(base):
    # Trabalha sobre uma cópia, preservando a base original.
    dados = base[
        ["ID_MN_RESI", "DT_SIN_PRI", "SE"]
    ].copy()

    dados["codigo_municipio"] = codigo_municipio_painel(
        dados["ID_MN_RESI"]
    )
    dados["SE"] = pd.to_numeric(dados["SE"], errors="coerce")
    dados["DT_SIN_PRI"] = pd.to_datetime(
        dados["DT_SIN_PRI"], errors="coerce"
    )

    # UF definida pelo município de residência.
    return dados.loc[
        dados["codigo_municipio"].str.startswith(
            str(int(filtro_estado)), na=False
        )
        & dados["DT_SIN_PRI"].dt.year.eq(2026)
        & dados["SE"].between(
            SE_INICIAL_MUNICIPIOS,
            SE_FINAL_MUNICIPIOS
        )
    ].copy()

# 1. CONTAGEM SINAN

notificados_municipios = (
    filtrar_sinan_municipios(df_notificados)
    .groupby("codigo_municipio")
    .size()
    .rename("Notificados")
)

provaveis_municipios = (
    filtrar_sinan_municipios(df_provaveis)
    .groupby("codigo_municipio")
    .size()
    .rename("Prováveis")
)

dados_municipios_painel = (
    pd.concat(
        [notificados_municipios, provaveis_municipios],
        axis=1
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

if (
    dados_municipios_painel["Prováveis"]
    > dados_municipios_painel["Notificados"]
).any():
    raise ValueError(
        "Há municípios com mais casos prováveis do que notificações."
    )

# Seleciona os 20 maiores totais de notificações.
dados_municipios_painel = (
    dados_municipios_painel.loc[
        dados_municipios_painel["Notificados"].gt(0)
    ]
    .sort_values(
        ["Notificados", "codigo_municipio"],
        ascending=[False, True]
    )
    .head(TOP_MUNICIPIOS)
    .copy()
)

if dados_municipios_painel.empty:
    raise ValueError(
        "Não há notificações para o estado e período selecionados."
    )

# 2. AGREGAÇÃO GAL PARA OS MESMOS MUNICÍPIOS

gal_municipios_painel = df_gal_final[
    ["IBGE", "SE", "exames_realizados", "casos_positivos"]
].copy()

gal_municipios_painel["codigo_municipio"] = (
    codigo_municipio_painel(gal_municipios_painel["IBGE"])
)

gal_municipios_painel["SE"] = pd.to_numeric(
    gal_municipios_painel["SE"], errors="coerce"
)

gal_municipios_painel = gal_municipios_painel.loc[
    gal_municipios_painel["SE"].between(
        SE_INICIAL_MUNICIPIOS,
        SE_FINAL_MUNICIPIOS
    )
    & gal_municipios_painel["codigo_municipio"].isin(
        dados_municipios_painel["codigo_municipio"]
    )
].copy()

for coluna in ["exames_realizados", "casos_positivos"]:
    gal_municipios_painel[coluna] = pd.to_numeric(
        gal_municipios_painel[coluna], errors="coerce"
    ).fillna(0)

totais_gal_municipios = (
    gal_municipios_painel
    .groupby("codigo_municipio")[
        ["exames_realizados", "casos_positivos"]
    ]
    .sum()
    .rename(columns={
        "exames_realizados": "Exames",
        "casos_positivos": "Positivos"
    })
    .reset_index()
)

dados_municipios_painel = dados_municipios_painel.merge(
    totais_gal_municipios,
    on="codigo_municipio",
    how="left",
    validate="one_to_one"
)

dados_municipios_painel[["Exames", "Positivos"]] = (
    dados_municipios_painel[["Exames", "Positivos"]]
    .fillna(0)
)

if (
    dados_municipios_painel["Positivos"]
    > dados_municipios_painel["Exames"]
).any():
    raise ValueError(
        "Há municípios com mais exames positivos do que realizados."
    )

# 3. NOMES DOS MUNICÍPIOS

referencia_municipios_painel = df_municipios[
    ["COD_IBGE", "MUNICÍPIO - IBGE - UF"]
].copy()

referencia_municipios_painel["codigo_municipio"] = (
    codigo_municipio_painel(
        referencia_municipios_painel["COD_IBGE"]
    )
)

referencia_municipios_painel = (
    referencia_municipios_painel[
        ["codigo_municipio", "MUNICÍPIO - IBGE - UF"]
    ]
    .dropna(subset=["codigo_municipio"])
    .drop_duplicates()
)

if referencia_municipios_painel["codigo_municipio"].duplicated().any():
    raise ValueError(
        "Há nomes diferentes para o mesmo código municipal."
    )

dados_municipios_painel = dados_municipios_painel.merge(
    referencia_municipios_painel,
    on="codigo_municipio",
    how="left",
    validate="one_to_one"
)

dados_municipios_painel["Município"] = (
    dados_municipios_painel["MUNICÍPIO - IBGE - UF"]
    .fillna(dados_municipios_painel["codigo_municipio"])
)

# Ordem crescente de notificações.
dados_municipios_painel = (
    dados_municipios_painel
    .sort_values(
        ["Notificados", "Município"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

# 4. PAINEL — MESMO PADRÃO VISUAL DA FIGURA POR UF

dados_municipios_painel = (
    dados_municipios_painel
    .sort_values(
        ["Notificados", "Município"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# Remove apenas o sufixo da UF, quando presente.
nomes_municipios_painel = (
    dados_municipios_painel["Município"]
    .astype(str)
    .str.replace(
        rf"\s*[-/]\s*{estado_encontrado}\s*$",
        "",
        regex=True
    )
)

# Altura proporcional ao número de municípios.
altura_figura_mun = max(
    9, len(dados_municipios_painel) * 0.43 + 5
)

# Contexto local: não altera o estilo dos demais gráficos.
with plt.rc_context({
    "font.family": "Arial",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": False
}):
    fig_municipios, (ax_sinan_mun, ax_gal_mun) = plt.subplots(
        1, 2,
        figsize=(26, altura_figura_mun),
        sharey=True,
        gridspec_kw={"wspace": 0.02}
    )

    posicoes_mun = np.arange(len(dados_municipios_painel))
    altura_barra_mun = 0.72

    # SINAN

    ax_sinan_mun.barh(
        posicoes_mun,
        dados_municipios_painel["Prováveis"],
        height=altura_barra_mun,
        color="#1b263b",
        edgecolor="white",
        linewidth=0.4,
        label="Prováveis",
        zorder=3
    )

    ax_sinan_mun.barh(
        posicoes_mun,
        (
            dados_municipios_painel["Notificados"]
            - dados_municipios_painel["Prováveis"]
        ),
        left=dados_municipios_painel["Prováveis"],
        height=altura_barra_mun,
        color="#a9b7c8",
        edgecolor="white",
        linewidth=0.4,
        label="Notificados",
        zorder=3
    )

    # GAL

    ax_gal_mun.barh(
        posicoes_mun,
        dados_municipios_painel["Positivos"],
        height=altura_barra_mun,
        color="#c32f27",
        edgecolor="white",
        linewidth=0.4,
        label="Positivos",
        zorder=3
    )

    ax_gal_mun.barh(
        posicoes_mun,
        (
            dados_municipios_painel["Exames"]
            - dados_municipios_painel["Positivos"]
        ),
        left=dados_municipios_painel["Positivos"],
        height=altura_barra_mun,
        color="#f4a261",
        edgecolor="white",
        linewidth=0.4,
        label="Exames Realizados",
        zorder=3
    )

    # RÓTULOS E FORMATAÇÃO

    configuracao_municipios = [
        {
            "ax": ax_sinan_mun,
            "total": "Notificados",
            "destaque": "Prováveis",
            "lbl_total": "Notificados",
            "lbl_destaque": "Prováveis"
        },
        {
            "ax": ax_gal_mun,
            "total": "Exames",
            "destaque": "Positivos",
            "lbl_total": "Testes",
            "lbl_destaque": "Positivos"
        }
    ]

    textos_painel_mun = []

    for cfg_mun in configuracao_municipios:
        ax_mun = cfg_mun["ax"]
        maximo_mun = max(
            float(dados_municipios_painel[cfg_mun["total"]].max()),
            1
        )

        # Proporção inicial igual à figura por UF.
        ax_mun.set_xlim(0, maximo_mun * 1.55)
        ax_mun.set_xticks([])
        ax_mun.set_axisbelow(True)

        # Linhas horizontais para acompanhar cada município.
        ax_mun.grid(
            axis="y",
            color="#c7c7c7",
            linestyle="--",
            linewidth=1,
            alpha=0.7
        )

        textos_eixo_mun = []

        for indice_mun, linha_mun in dados_municipios_painel.iterrows():
            total_mun = int(linha_mun[cfg_mun["total"]])
            destaque_mun = int(linha_mun[cfg_mun["destaque"]])

            percentual_mun = (
                f"{100 * destaque_mun / total_mun:.0f}%"
                if total_mun > 0 else "—"
            )

            texto_total_mun = f"{total_mun:,}".replace(",", ".")
            texto_destaque_mun = f"{destaque_mun:,}".replace(",", ".")

            rotulo_mun = (
                f"{cfg_mun['lbl_total']}: {texto_total_mun} | "
                f"{cfg_mun['lbl_destaque']}: "
                f"{texto_destaque_mun} ({percentual_mun})"
            )

            # Distância fixa em pontos: texto sempre próximo da barra.
            texto_mun = ax_mun.annotate(
                rotulo_mun,
                xy=(total_mun, indice_mun),
                xytext=(5, 0),
                textcoords="offset points",
                ha="left",
                va="center",
                fontsize=14,
                color="black",
                annotation_clip=False,
                bbox={
                    "boxstyle": "square,pad=0.08",
                    "facecolor": "white",
                    "edgecolor": "none",
                },
                zorder=4
            )

            textos_eixo_mun.append(texto_mun)

        textos_painel_mun.append((ax_mun, textos_eixo_mun))

        for borda_mun in ax_mun.spines.values():
            borda_mun.set_visible(False)

        ax_mun.legend(
            loc="upper left",
            bbox_to_anchor=(-0.02, -0.025),
            fontsize=16,
            ncols=2,
            frameon=False,
            handlelength=1.8,
            columnspacing=2
        )

    # MUNICÍPIOS E ORDEM DAS LINHAS

    ax_sinan_mun.set_yticks(
        posicoes_mun,
        nomes_municipios_painel
    )

    ax_sinan_mun.tick_params(
        axis="y",
        length=0,
        labelsize=16,
        pad=4
    )

    ax_gal_mun.tick_params(
        axis="y",
        length=0,
        left=False,
        labelleft=False
    )

    # Menor número de notificações no topo.
    ax_sinan_mun.set_ylim(
        len(dados_municipios_painel) - 0.4,
        -0.6
    )

    # TÍTULOS

    ax_sinan_mun.set_title(
        "NOTIFICAÇÕES E CASOS PROVÁVEIS",
        fontsize=16,
        fontweight="bold",
        pad=22
    )

    ax_gal_mun.set_title(
        "EXAMES REALIZADOS E POSITIVOS",
        fontsize=16,
        fontweight="bold",
        pad=22
    )

    fig_municipios.suptitle(
        (
            f"CHIKUNGUNYA POR MUNICÍPIO - "
            f"SE {SE_INICIAL_MUNICIPIOS} A SE {SE_FINAL_MUNICIPIOS} "
            f"- {nome_estado} - 2026"
        ).upper(),
        fontsize=19,
        fontweight="bold",
        y=0.98
    )

    fig_municipios.subplots_adjust(
        left=0.18,
        right=0.99,
        top=0.88,
        bottom=0.10
    )

    # Ajusta a margem esquerda ao nome municipal mais longo.
    fig_municipios.canvas.draw()
    renderer_mun = fig_municipios.canvas.get_renderer()

    largura_nome_mun = max(
        rotulo.get_window_extent(renderer_mun).width
        for rotulo in ax_sinan_mun.get_yticklabels()
    )

    margem_esquerda_mun = max(
        0.08,
        (largura_nome_mun + 22) / fig_municipios.bbox.width
    )

    fig_municipios.subplots_adjust(left=margem_esquerda_mun)

    # Amplia o eixo somente quando algum texto ultrapassa seu painel.
    # Isso evita sobreposição entre os rótulos SINAN e as barras GAL.
    for _ in range(12):
        fig_municipios.canvas.draw()
        renderer_mun = fig_municipios.canvas.get_renderer()
        houve_ajuste_mun = False

        for ax_mun, textos_mun in textos_painel_mun:
            limite_texto_mun = max(
                texto.get_window_extent(renderer_mun).x1
                for texto in textos_mun
            )

            if limite_texto_mun > ax_mun.bbox.x1 - 8:
                ax_mun.set_xlim(0, ax_mun.get_xlim()[1] * 1.08)
                houve_ajuste_mun = True

        if not houve_ajuste_mun:
            break

    # SALVAR

    arquivo_painel_municipios = os.path.join(
        local_pasta,
        f"chik_{nome_estado}_painel_sinan_gal_municipios_"
        f"SE{SE_FINAL_MUNICIPIOS}.png"
    )

    fig_municipios.savefig(
        arquivo_painel_municipios,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

In [ ]:
# SALVAR GRÁFICOS EM PDF ÚNICO
def gerar_pdf(lista_arquivos, nome_pdf: f'Relatorio_Chikungunya_{nome_estado}_2026.pdf'):
    c = canvas.Canvas(nome_pdf, pagesize=A4)
    largura, altura = A4
    margem = 30
    
    # Função auxiliar para escrever o cabeçalho em todas as páginas
    def desenhar_cabecalho(canvas):
        canvas.setFont("Helvetica-Bold", 16)
        canvas.drawCentredString(largura / 2, altura - 25, f"Análise Chikungunya - {nome_estado} - 2026")       
        canvas.line(margem, altura - 35, largura - margem, altura - 35)                                         

    def processar_paginas(imagens_lista):
        espacamento = 5                                 
        area_h = (altura / 2) - (margem + 40)   
        largura_max = largura - (2 * margem)    
    
        def desenhar_figura(img_path, pos_y):
            # Verifica se a imagem existe antes de desenhar (evita o erro OSError)
            if os.path.exists(img_path):
                c.drawImage(img_path, margem, pos_y, width=largura_max, height=area_h, preserveAspectRatio=True)
            else:
                print(f"⚠️ Aviso: A imagem {img_path} não foi encontrada.")

        def desenhar_rodape(canvas, texto):
            canvas.setFont("Helvetica-Oblique", 5) 
            canvas.drawString(30, 20, texto)

        texto_rodape = 'Análise realizada por Coordenação-Geral de Vigilância das Arboviroses (CGARB/DEDT/SVSA/MS)'

        # 1. PÁGINA 1
        desenhar_cabecalho(c)
        desenhar_figura(imagens_lista[0], (altura / 2) + espacamento) 
        desenhar_figura(imagens_lista[1], margem)
        desenhar_rodape(c, texto_rodape) # Rodapé da Página 1
        c.showPage()
    
        # 2. PÁGINAS SEGUINTES
        restante = imagens_lista[2:]
        for i in range(0, len(restante), 2):
            desenhar_cabecalho(c)
            grupo = restante[i:i+2]
        
            pos_y_lista = [(altura / 2) + espacamento, margem]
        
            for idx, img_path in enumerate(grupo):
                desenhar_figura(img_path, pos_y_lista[idx])
            
            desenhar_rodape(c, texto_rodape) # CORREÇÃO: Rodapé nas outras páginas!
            c.showPage()

    processar_paginas(lista_arquivos)
    c.save()
    print(f"Arquivo {os.path.basename(nome_pdf)} gerado com sucesso e salvo na pasta!")

# Lista dos arquivos
arquivos = [
    f"chik_{nome_estado}_municipio_tabela_comparativo_2026.png",
    f"chik_{nome_estado}_incidencia_municipio_recente_2026.png",
    f"chik_{nome_estado}_incidencia_regiao_saude.png",
    f"chik_{nome_estado}_incidencia_indigena_municipio_recente_2026.png",
    f'chik_{nome_estado}_positividade_gal_semana.png',
    f'chik_{nome_estado}_casos_positividade_se.png',
    f'chik_{nome_estado}_provaveis_classificacao.png',
    #f'painel_dengue_chik_{nome_estado}_casos_exames_4_8se.png',
]

#graficos = [os.path.join(local_pasta, nome) for nome in arquivos]
#salvar_pdf = os.path.join(local_pasta, f'Relatorio_Chikungunya_{nome_estado}_2026.pdf')

#gerar_pdf(graficos, nome_pdf=salvar_pdf)

### Análise Indígenas - Estado

In [ ]:
from geobr import read_municipality, read_state

with warnings.catch_warnings():
    warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)


# --- 1. PREPARAÇÃO DA BASE DE POPULAÇÃO MUNICIPAL ---
df_populacao = df_municipios.copy()
df_populacao["code_muni_6"] = (
    df_populacao["COD_IBGE"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
    .str.zfill(6)
    .str[:6]
)

# Garantir que a população é numérica
df_populacao["POPULACAO"] = (
    df_populacao["POPULACAO"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_populacao["POPULACAO"] = pd.to_numeric(df_populacao["POPULACAO"], errors="coerce")


# --- 2. CARREGAR MALHA MUNICIPAL E ESTADUAL (geobr) ---
mapa_limites_estados = read_state(code_state=filtro_estado, year=2020)
mapa_municipios = read_municipality(code_muni=filtro_estado, year=2020)
mapa_municipios["code_muni_6"] = mapa_municipios["code_muni"].astype(str).str[:6]


# --- 3. FUNÇÃO AUXILIAR DE PROCESSAMENTO MUNICIPAL ---
def processar_incidencia_muni(df_sinan, df_pop, mapa_muni):
    df_casos = (
        df_sinan["ID_MN_RESI"]
        .dropna()
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(6)
        .str[:6]
        .value_counts()
        .rename_axis("code_muni_6")
        .reset_index(name="Total")
    )

    mapa = mapa_muni.merge(df_casos, on="code_muni_6", how="left")
    mapa = mapa.merge(df_pop[["code_muni_6", "POPULACAO"]], on="code_muni_6", how="left")

    mapa["Total"] = mapa["Total"].fillna(0).astype(int)
    mapa["POPULACAO"] = mapa["POPULACAO"].fillna(1)  # Evita divisão por zero

    mapa["Incidencia"] = (mapa["Total"] / mapa["POPULACAO"]) * 100000
    return mapa


# FILTRAR POPULAÇÃO INDÍGENA
chik_indigenas_1 = provaveis_estado[provaveis_estado["CS_RACA"] == 5]
chik_indigenas_2 = provaveis_estado_8s[provaveis_estado_8s["CS_RACA"] == 5]

# Processar os dois datasets
mapa_acumulado = processar_incidencia_muni(chik_indigenas_1, df_populacao, mapa_municipios)
mapa_8semanas = processar_incidencia_muni(chik_indigenas_2, df_populacao, mapa_municipios)


# --- 4. DEFINIÇÃO DE FAIXAS E CORES PARA CADA PERÍODO ---

def inserir_nomes_municipios(ax, mapa, tamanho_fonte=6):
    municipios_rotulos = mapa.loc[
        mapa.geometry.notna() & ~mapa.geometry.is_empty
    ].copy()

    pontos_rotulos = municipios_rotulos.geometry.representative_point()

    for (_, municipio), ponto in zip(
        municipios_rotulos.iterrows(), pontos_rotulos
    ):
        nome = municipio["name_muni"]

        if pd.isna(nome):
            continue

        ax.text(
            ponto.x,
            ponto.y,
            str(nome),
            fontsize=tamanho_fonte,
            color="black",
            ha="center",
            va="center",
            zorder=10,
            path_effects=[
                pe.withStroke(
                    linewidth=1.8,
                    foreground="white"
                )
            ]
        )

# Faixas para o Acumulado do Ano
cortes_ano = [-1, 10, 50, 100, 200, 300, float("inf")]

rotulos = [
    '0,1 - 10',
    '11 - 50',
    '51 - 100',
    '101 - 200',
    '201 - 300',
    'Acima de 300'
]

# Faixas para as Últimas 8 Semanas
cortes_8sem = [-1, 5, 10, 20, 50, 100, float("inf")]

rotulos_8sem = [
    "0,1 - 5",
    "5,1 - 10",
    "10,1 - 20",
    "20,1 - 50",
    "50,1 - 100",
    "Acima de 100",
]

dicionario_cores = {
    "0": "#fff7ec",
    "1": "#fdd49e",
    "2": "#fc8d59",
    "3": "#ef6548",
    "4": "#b30000",
    "5": "#7f0000",
}

# Categorização do Acumulado
mapa_acumulado["Categoria"] = pd.cut(mapa_acumulado["Incidencia"], bins=cortes_ano, labels=rotulos)
mapa_acumulado["Cor"] = mapa_acumulado["Categoria"].map(dict(zip(rotulos, dicionario_cores.values())))

# Categorização das 8 Semanas
mapa_8semanas["Categoria"] = pd.cut(mapa_8semanas["Incidencia"], bins=cortes_8sem, labels=rotulos_8sem)
mapa_8semanas["Cor"] = mapa_8semanas["Categoria"].map(dict(zip(rotulos_8sem, dicionario_cores.values())))


# --- 5. CONSTRUÇÃO DA FIGURA (2 MAPAS LADO A LADO - ALTA RESOLUÇÃO) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 13))

# ---------------- MAPA 1: ACUMULADO DO ANO ----------------
mapa_acumulado.plot(
    color=mapa_acumulado["Cor"],
    linewidth=0.06,
    edgecolor="black",
    ax=ax1,
)

mapa_limites_estados.plot(
    color="none",
    linewidth=1.0,
    edgecolor="black",
    ax=ax1,
)

inserir_nomes_municipios(ax1, mapa_acumulado, tamanho_fonte=9)

ax1.set_title(
    "INCIDÊNCIA SE 1 A SE 26 - 2026", 
    fontsize=16, 
    fontweight="bold", 
    pad=12
)
ax1.axis("off")

legendas_ano = [
    mpatches.Patch(color=dicionario_cores[str(i)], label=rotulos[i])
    for i in range(len(rotulos))
]
ax1.legend(
    title="Incidência por 100 mil hab.",
    title_fontsize=12,
    handles=legendas_ano,
    fontsize=13,
    ncols=6,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.06),
    frameon=False,
)

# ---------------- MAPA 2: ÚLTIMAS 8 SEMANAS ----------------
mapa_8semanas.plot(
    color=mapa_8semanas["Cor"],
    linewidth=0.06,
    edgecolor="black",
    ax=ax2,
)

mapa_limites_estados.plot(
    color="none",
    linewidth=1.0,
    edgecolor="black",
    ax=ax2,
)

inserir_nomes_municipios(ax2, mapa_8semanas, tamanho_fonte=9)

ax2.set_title(
    f"INCIDÊNCIA SE 26 A SE {se_temporada_atual} - 2026",
    fontsize=16,
    fontweight="bold",
    pad=12,
)
ax2.axis("off")

legendas_8sem = [
    mpatches.Patch(color=dicionario_cores[str(i)], label=rotulos_8sem[i])
    for i in range(len(rotulos_8sem))
]
ax2.legend(
    title="Incidência por 100 mil hab.",
    title_fontsize=12,
    handles=legendas_8sem,
    fontsize=13,
    ncols=6,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.06),
    frameon=False,
)

# --- TÍTULO GERAL E RODAPÉ ---
plt.suptitle(
    f"TAXA DE INCIDÊNCIA DE CHIKUNGUNYA EM MUNICÍPIOS COM ÁREA INDÍGENA - {nome_estado} - 2026".upper(),
    fontsize=18,
    fontweight="bold",
    y=0.96,
)

fig.text(
    0.05,
    -0.02,
    f"Fonte: SINAN | Dados até: {data_analise} | ATENÇÃO: Os dados foram filtrados pela coluna CS_RACA(5) e não permitem inferir ocorrência direta em territórios indígenas",
    fontsize=10,
    color="black",
    ha="left",
)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])

#salvar_pasta = os.path.join(local_pasta, f"chik_{nome_estado}_incidencia_indigena_municipio_comparativo_2026.png")
#plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from geobr import read_municipality, read_state

with warnings.catch_warnings():
    warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)


# --- 1. PREPARAÇÃO DA BASE DE POPULAÇÃO MUNICIPAL ---
df_populacao = df_municipios.copy()
df_populacao["code_muni_6"] = (
    df_populacao["COD_IBGE"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
    .str.zfill(6)
    .str[:6]
)

# Garantir que a população é numérica
df_populacao["POPULACAO"] = (
    df_populacao["POPULACAO"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_populacao["POPULACAO"] = pd.to_numeric(df_populacao["POPULACAO"], errors="coerce")


# --- 2. CARREGAR MALHA MUNICIPAL E ESTADUAL (geobr) ---
mapa_limites_estados = read_state(code_state=filtro_estado, year=2020)
mapa_municipios = read_municipality(code_muni=filtro_estado, year=2020)
mapa_municipios["code_muni_6"] = mapa_municipios["code_muni"].astype(str).str[:6]


# --- 3. FUNÇÃO AUXILIAR DE PROCESSAMENTO MUNICIPAL ---
def processar_incidencia_muni(df_sinan, df_pop, mapa_muni):
    df_casos = (
        df_sinan["ID_MN_RESI"]
        .dropna()
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(6)
        .str[:6]
        .value_counts()
        .rename_axis("code_muni_6")
        .reset_index(name="Total")
    )

    mapa = mapa_muni.merge(df_casos, on="code_muni_6", how="left")
    mapa = mapa.merge(df_pop[["code_muni_6", "POPULACAO"]], on="code_muni_6", how="left")

    mapa["Total"] = mapa["Total"].fillna(0).astype(int)
    mapa["POPULACAO"] = mapa["POPULACAO"].fillna(1)  # Evita divisão por zero

    mapa["Incidencia"] = (mapa["Total"] / mapa["POPULACAO"]) * 100000
    return mapa


# FILTRAR POPULAÇÃO INDÍGENA (Apenas o período recente - SE 27 em diante)
chik_indigenas_recente = provaveis_estado_8s[provaveis_estado_8s["CS_RACA"] == 5]

# Processar dataset
mapa_recente = processar_incidencia_muni(chik_indigenas_recente, df_populacao, mapa_municipios)


# --- 4. DEFINIÇÃO DE FAIXAS E CORES ---

def inserir_nomes_municipios(ax, mapa, tamanho_fonte=5):
    municipios_rotulos = mapa.loc[
        mapa.geometry.notna() & ~mapa.geometry.is_empty
    ].copy()

    pontos_rotulos = municipios_rotulos.geometry.representative_point()

    for (_, municipio), ponto in zip(
        municipios_rotulos.iterrows(), pontos_rotulos
    ):
        nome = municipio["name_muni"]

        if pd.isna(nome):
            continue

        ax.text(
            ponto.x,
            ponto.y,
            str(nome),
            fontsize=tamanho_fonte,
            color="black",
            ha="center",
            va="center",
            zorder=10,
            path_effects=[
                pe.withStroke(
                    linewidth=1.5,
                    foreground="white"
                )
            ]
        )

# Faixas para o período recente (indígena)
cortes_recente = [-1, 5, 10, 20, 50, 100, float("inf")]

rotulos_recente = [
    "0,1 - 5",
    "5,1 - 10",
    "10,1 - 20",
    "20,1 - 50",
    "50,1 - 100",
    "Acima de 100",
]

dicionario_cores = {
    "0": "#fff7ec",
    "1": "#fdd49e",
    "2": "#fc8d59",
    "3": "#ef6548",
    "4": "#b30000",
    "5": "#7f0000",
}

# Categorização
mapa_recente["Categoria"] = pd.cut(mapa_recente["Incidencia"], bins=cortes_recente, labels=rotulos_recente)
mapa_recente["Cor"] = mapa_recente["Categoria"].map(dict(zip(rotulos_recente, dicionario_cores.values())))


# --- 5. CONSTRUÇÃO DA FIGURA (MAPA ÚNICO) ---
fig, ax = plt.subplots(1, 1, figsize=(15, 15))

mapa_recente.plot(
    color=mapa_recente["Cor"],
    linewidth=0.05,
    edgecolor="black",
    ax=ax,
)

mapa_limites_estados.plot(
    color="none",
    linewidth=0.8,
    edgecolor="black",
    ax=ax,
)

inserir_nomes_municipios(ax, mapa_recente, tamanho_fonte=9)

ax.axis("off")

legendas_recente = [
    mpatches.Patch(color=dicionario_cores[str(i)], label=rotulos_recente[i])
    for i in range(len(rotulos_recente))
]
ax.legend(
    title="Incidência por 100 mil hab.",
    title_fontsize=12,
    handles=legendas_recente,
    fontsize=12,
    ncols=6,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    frameon=False,
)

# --- TÍTULO GERAL E RODAPÉ ---
plt.suptitle(
    f"TAXA DE INCIDÊNCIA DE CHIKUNGUNYA - {nome_estado} - 2026\nMUNICÍPIOS COM ÁREAS INDÍGENAS".upper(),
    fontsize=20,
    fontweight="bold",
    y=0.9,
)

fig.text(
    0.5,
    0.07,
    f"Fonte: SINAN | Dados até: {data_analise} | ATENÇÃO: Os dados foram filtrados pela coluna CS_RACA(5) e não permitem inferir ocorrência direta em territórios indígenas",
    fontsize=9,
    color="black",
    ha="center",
)

plt.tight_layout(rect=[0, 0.08, 1, 0.93])

#salvar_pasta = os.path.join(local_pasta, f"chik_{nome_estado}_incidencia_indigena_municipio_recente_2026.png")
#plt.savefig(salvar_pasta, dpi=300, bbox_inches="tight")
plt.show()

#### Algoritmo desenvolvido por Dengue Brazil